# 01 — Pruning YOLO26s dengan Target GFLOPs

**NO RECOVERY · NO TRAINING · NO KD**

Notebook standalone khusus **YOLO26s** untuk pruning terstruktur Poltrada V3 dengan LAMP +
DepGraph. Ukur variasi perubahan akurasi dan biaya komputasi pada target 1,20× di lima student
baseline yang dilatih dengan seed berbeda. Pembanding 1,10× dan 1,70× memakai baseline seed 42.
Setiap perlakuan dibandingkan dengan baseline seed yang sama memakai mAP50–95, AP per kelas,
parameter, GFLOPs, dan latency p50/p95 pada protokol yang sama.

Speed-up GFLOPs adalah GFLOPs baseline dibagi GFLOPs pruned. Speed-up latency memakai rasio
median raw forward. Adaptasi YOLO26 mempertahankan kedua cabang Detect dan dimensi antarmuka.
Contoh konfigurasi MP-YOLO memakai 1,70× sebagai target komputasi.
[MP-YOLO](https://github.com/Wang-jj-zs/MP-YOLO/blob/master/MP-YOLO/compress.py) ·
[LAMP](https://arxiv.org/abs/2010.07611) · [Kode penulis LAMP](https://github.com/jaeho-lee/layer-adaptive-sparsity) ·
[DepGraph](https://arxiv.org/abs/2301.12900)

Di Kaggle pilih Tesla T4, aktifkan internet, pasang input **Poltrada V3 YOLO26 Clean**, dan
hubungkan Secret `WANDB_API_KEY`. `Run All` menjalankan antrean serial pada `cuda:0`, termasuk
saat runtime menyediakan dua GPU. Di Filkom pilih kernel CUDA, isi path dataset dan manifest
sumber terverifikasi, serta set `WANDB_API_KEY` pada environment Jupyter.

## 02 — Konfigurasi YOLO26s

Default `MODE="full_val"` dan `RESUME_SUITE_DIR=None` membuat suite baru dari lima `best.pt` W&B
yang dikunci. Gunakan FP32 640 px dan batch validasi 16. Path dataset, perangkat, output, dan
salinan baseline lokal opsional tersedia di sel konfigurasi. Set `DATASET_MANIFEST_ONLY=True`
pada mount resmi Kaggle untuk mengekspor manifest yang bisa disalin ke Filkom.

| Mode | Data dan tujuan |
| --- | --- |
| `smoke` | Subset tetap hingga 128 citra validation untuk memeriksa alur dan menyimpan hasil lokal |
| `full_val` | Seluruh validation Poltrada V3 terverifikasi, benchmark CUDA, dan publikasi checkpoint W&B |

In [ ]:
import os
from pathlib import Path

PLATFORM = "kaggle" if Path("/kaggle").is_dir() else "filkom"
WORKSPACE_ROOT = Path(os.environ.get("SKRIPSI_ROOT", Path.cwd()))
DATASET_ROOT = (Path("/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean") if PLATFORM == "kaggle"
                else WORKSPACE_ROOT / "datasets/poltrada-v3")
DATA_YAML = DATASET_ROOT / "data.yaml"
DATASET_MANIFEST_PATH = None  # Filkom: Path ke dataset_manifest.json dari mount Kaggle resmi.
DATASET_MANIFEST_ONLY = False
OUTPUT_DIR = (Path("/kaggle/working") if PLATFORM == "kaggle" else WORKSPACE_ROOT / "outputs") / "pruning/lamp_yolo26s_per_seed"
RESUME_SUITE_DIR = None
LOCAL_BASELINE_PATHS = {}  # Opsional, misalnya {42: Path(".../best.pt")}. Hash harus cocok dengan pin.

MODE = "full_val"  # "smoke" memakai hingga 128 citra val dan menyimpan hasil secara lokal.
DEVICE = "auto"
IMGSZ, BATCH, WORKERS = 640, 16, 2
INSTALL_DEPENDENCIES = True
RUN_LATENCY_BENCHMARK = True
LATENCY_WARMUP, LATENCY_REPEATS = 50, 200
RELOAD_TIMEOUT_SECONDS = 300
EDA_DPI, EDA_MAX_LAYERS_PER_PAGE = 300, 32
EDA_DISPLAY_RATIO = 0.50

# Proteksi dan tier cap dikunci di helper audit GFLOPs.
GLOBAL_PRUNING = True
RECOVERY_ENABLED = FINETUNING_ENABLED = KD_ENABLED = False

WANDB_ENTITY = 'ilhamrafiqin-brawijaya-university'
WANDB_PROJECT = 'yolom26-kd-indotraffic'
DATASET_ID = 'ilhamrafiqin/poltrada-v3-yolo26-clean'

BASELINE_SOURCES = {
    42: {
        "ref": f"{WANDB_ENTITY}/{WANDB_PROJECT}/02_baseline_yolo26s-q5kp9k9d-checkpoints:v62",
        "sha256": "5bc3ae61f8da1c9f03916922de10146805c27c64a11e46edd766a00d0f6f18fd",
    },
    0: {
        "ref": f"{WANDB_ENTITY}/{WANDB_PROJECT}/02_baseline_yolo26s-7f5jmtwb-checkpoints:v71",
        "sha256": "d02d8d90d5000fc8379379acb0c0cb3cde0c01143b197d237b25be1bae99bf4f",
    },
    2024: {
        "ref": f"{WANDB_ENTITY}/{WANDB_PROJECT}/02_baseline_yolo26s-1o9ggnyw-checkpoints:v41",
        "sha256": "47cfab586889d2b464c882239c86224b645363a5a4fa3badc31c46991ad27d70",
    },
    7: {
        "ref": f"{WANDB_ENTITY}/{WANDB_PROJECT}/02_baseline_yolo26s-9cq437l3-checkpoints:v49",
        "sha256": "3e59d5f5302fb524da729e139ef7e9b8456c08d0952c19b08af19c7f39bce6ab",
    },
    3407: {
        "ref": f"{WANDB_ENTITY}/{WANDB_PROJECT}/02_baseline_yolo26s-bmshdzv9-checkpoints:v43",
        "sha256": "de9f06892953a343629a4b3a59df880d771973ae95160dee02cd96e37de59e46",
    },
}
IMPLEMENTATION_SHA256 = '81a80fccd427426c4dfc72368dc0307ca2d761bdd0194e8bb409f872bbb8a83a'

# Filter tampilan EDA berdasarkan ID perlakuan, misalnya "target-1.20x-seed0". () menampilkan semua.
EDA_CONFIGURATIONS = ()
EDA_DETAIL_CONFIGURATIONS = ()

## 03 — Environment dan Dependensi

Gunakan Python 3.11 atau lebih baru. Bootstrap mengunci Ultralytics 8.4.155, Torch-Pruning 1.6.1,
dan W&B 0.28.1 sambil mempertahankan Torch serta torchvision dari kernel. Suite mencatat versi
dependensi, perangkat, build CUDA, presisi, dan hash implementasi importance.

In [ ]:
import csv
import hashlib
import importlib.metadata as metadata
import inspect
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
import traceback
import uuid
import warnings
from contextlib import contextmanager
from copy import deepcopy
from datetime import datetime, timezone
from types import SimpleNamespace

PINNED = {"ultralytics": "8.4.155", "wandb": "0.28.1", "torch-pruning": "1.6.1"}
REQUIRED_CHECKS = (
    "baseline_loaded", "baseline_forward", "dependency_graph", "structural_channels_removed",
    "parameters_reduced", "pruned_forward", "protected_head_unchanged", "checkpoint_saved",
    "checkpoint_reloaded", "reload_consistent", "detection_prediction",
)


def write_json(path, value):
    """Tulis JSON standar; NaN/Inf ditolak, bukan disamarkan sebagai metrik."""
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")


def sha256_file(path):
    """Hash berkas tanpa membaca seluruh checkpoint ke RAM sekaligus."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
def safe_error(error):
    message = str(error)
    secret = os.environ.get("WANDB_API_KEY")
    return message.replace(secret, "[REDACTED]") if secret else message


def model_state_sha256(core):
    """Buktikan setiap konfigurasi dimulai dari bobot/buffer FP32 yang identik."""
    digest = hashlib.sha256()
    for name, value in sorted(core.state_dict().items()):
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str((str(tensor.dtype), tuple(tensor.shape))).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


def metric_comparison(before, after):
    result = {}
    for key, label in (("parameters", "parameter"), ("gflops", "gflops"),
                       ("checkpoint_bytes", "checkpoint")):
        base, pruned = before.get(key), after.get(key)
        result[label + "_reduction_pct"] = 100 * (base - pruned) / base if base else None
    base = (before.get("validation") or {}).get("map50_95")
    pruned = (after.get("validation") or {}).get("map50_95")
    delta = pruned - base if base is not None and pruned is not None else None
    result.update({"delta_map50_95": delta, "delta_map50_95_pp": 100 * delta if delta is not None else None,
                   "relative_map50_95_pct": 100 * delta / base if delta is not None and base else None})
    return result

In [ ]:
def ensure_dependencies(output_dir, install=True):
    """Pasang dependency tambahan sambil mengunci Torch/torchvision environment."""
    if sys.version_info < (3, 11):
        raise RuntimeError("Gunakan Python >=3.11.")
    preserved = {name: metadata.version(name) for name in ("torch", "torchvision")}
    needed = []
    for name, expected in PINNED.items():
        try:
            found = metadata.version(name)
        except metadata.PackageNotFoundError:
            found = None
        if found != expected:
            needed.append(f"{name}=={expected}")
    if needed and not install:
        raise RuntimeError(f"Dependency belum cocok: {needed}; aktifkan INSTALL_DEPENDENCIES.")
    if needed:
        constraints = Path(output_dir) / "preserve_torch.txt"
        constraints.write_text("\n".join(f"{k}=={v}" for k, v in preserved.items()) + "\n", encoding="utf-8")
        subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                        "--constraint", str(constraints), *needed], check=True)
    actual = {name: metadata.version(name) for name in preserved}
    if actual != preserved:
        raise RuntimeError(f"Torch/torchvision berubah: {preserved} -> {actual}; restart runtime asli.")
    for name, expected in PINNED.items():
        if metadata.version(name) != expected:
            raise RuntimeError(f"Versi {name} tidak sesuai pin {expected}; restart kernel bila perlu.")

In [ ]:
from datetime import UTC
import re

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def write_csv(path, rows):
    import csv
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with temporary.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=keys)
        writer.writeheader()
        writer.writerows({key: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value
                         for key, value in row.items()} for row in rows)
    os.replace(temporary, path)


def workflow_digest(value):
    import hashlib
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str, allow_nan=False).encode()).hexdigest()


def workflow_bind(folder, name, contract):
    """Reject semantic changes on resume, allowing absolute paths to relocate."""
    path = Path(folder) / name
    contract = json.loads(json.dumps(contract, sort_keys=True, default=str, allow_nan=False))
    record = {"contract": contract, "sha256": workflow_digest(contract)}
    if path.is_file() and json.loads(path.read_text(encoding="utf-8")) != record:
        raise ValueError(f"Kontrak resume berbeda: {name}. Gunakan suite baru.")
    write_json(path, record)
    return record


def workflow_save_checkpoint(core, path, provenance):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        existing = YOLO(str(path), task="detect").model.cpu().float().eval()
        if model_state_sha256(existing) != model_state_sha256(core):
            raise ValueError(f"Checkpoint immutable sudah berisi state berbeda: {path}")
        return {"path": str(path.resolve()), "sha256": sha256_file(path), "file_bytes": path.stat().st_size}
    temporary = path.with_name(path.stem + "_" + uuid.uuid4().hex + ".pt")
    saved = save_fp32_checkpoint(core, temporary, provenance)
    os.replace(temporary, path)
    return {**saved, "path": str(path.resolve())}


def workflow_suite_dir(output, resume=None):
    if resume:
        path = Path(resume).expanduser().resolve(strict=True)
        if not path.is_dir():
            raise NotADirectoryError(path)
        return path
    from datetime import datetime
    path = Path(output).expanduser().resolve() / (datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
    path.mkdir(parents=True, exist_ok=False)
    return path


def workflow_status(statuses, *, useful=False):
    statuses = list(statuses)
    if statuses and all(status == "PASS" for status in statuses):
        return "PASS"
    if useful or any(status in {"PASS", "PARTIAL"} for status in statuses):
        return "PARTIAL"
    return "FAIL" if any(status == "FAIL" for status in statuses) else "NOT_RUN"


def workflow_version_ref(reference):
    if not isinstance(reference, str) or not re.fullmatch(r"[^/:]+/[^/:]+/[^/:]+:v\d+", reference):
        raise ValueError("Input W&B memerlukan entity/project/artifact:vN (versi tetap).")
    return reference


def workflow_dataset_manifest(dataset, source_yaml, root, folder):
    """Portable full train/val content identity, never an evaluation train=val placeholder."""
    raw = yaml.safe_load(Path(source_yaml).read_text(encoding="utf-8-sig"))
    names = ordered_names(raw["names"])
    root = Path(root).resolve()
    train = _gflops_split_images(root, raw.get("train"))
    val = _gflops_split_images(root, raw.get("val"))
    if any(image.with_suffix(".npy").is_file() for image in (*train, *val)):
        raise ValueError("Gunakan sumber dataset tanpa cache gambar .npy.")
    train_fp, train_rows = _gflops_fingerprint(root, train, "train", names)
    val_fp, val_rows = _gflops_fingerprint(root, val, "val", names)
    if {row["image_sha256"] for row in train_rows}.intersection(row["image_sha256"] for row in val_rows):
        raise ValueError("Train dan val berbagi konten gambar; recovery tidak boleh memakai train=val.")
    record = {"format_version": 1, "dataset_id": DATASET_ID, "class_names": names,
              "train_fingerprint": train_fp, "val_fingerprint": val_fp,
              "provenance_status": dataset.get("provenance_status", "UNVERIFIED"),
              "train_count": len(train), "val_count": len(val), "source_yaml_sha256": sha256_file(source_yaml)}
    write_json(Path(folder) / "dataset_manifest.json", record)
    return record, train, val

In [ ]:
SUITE_DIR = workflow_suite_dir(OUTPUT_DIR, RESUME_SUITE_DIR)
os.environ["YOLO_AUTOINSTALL"] = "false"
os.environ.setdefault("YOLO_CONFIG_DIR", str(SUITE_DIR / "ultralytics_config"))
ensure_dependencies(SUITE_DIR, install=INSTALL_DEPENDENCIES)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch_pruning as tp
import ultralytics
import wandb
import yaml
from PIL import Image
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from ultralytics.nn.modules import Bottleneck, C2PSA, Detect
from ultralytics.nn.modules.block import AAttn, Attention
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import _attention_ops
from thop import profile as thop_profile

In [ ]:
def seed_environment(seed):
    """Samakan RNG dan backend FP32 di notebook dan proses restore."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.set_float32_matmul_precision("highest")


def environment_report():
    """Periksa API aktual, bukan API yang diasumsikan dari contoh YOLOv8."""
    for name, expected in PINNED.items():
        if metadata.version(name) != expected:
            raise RuntimeError(f"Pin {name} tidak terpenuhi.")
    if ultralytics.__version__ != PINNED["ultralytics"]:
        raise RuntimeError("Modul Ultralytics lama masih di kernel; restart runtime.")
    signature = inspect.signature(tp.pruner.BasePruner)
    for key in ("global_pruning", "max_pruning_ratio", "round_to", "output_transform", "forward_fn"):
        if key not in signature.parameters:
            raise RuntimeError(f"BasePruner tidak mendukung {key}.")
    return {
        "python": sys.version, "platform": platform.platform(),
        "packages": {name: metadata.version(name) for name in (*PINNED, "torch", "torchvision", "numpy", "ultralytics-thop")},
        "cuda_available": torch.cuda.is_available(), "cuda_build": torch.version.cuda,
        "device": str(TARGET_DEVICE),
        "gpu": torch.cuda.get_device_name(TARGET_DEVICE) if TARGET_DEVICE.type == "cuda" else None,
        "precision": "FP32", "tf32": False, "cudnn_benchmark": False,
        "base_pruner_signature": str(signature),
        "lamp_signature": str(inspect.signature(tp.importance.LAMPImportance)),
        "lamp_source_sha256": hashlib.sha256(inspect.getsource(tp.importance.LAMPImportance).encode()).hexdigest(),
        "importance_module_sha256": sha256_file(inspect.getfile(tp.importance.LAMPImportance)),
        "flops_method": "THOP full-resolution Conv/BN + separate Ultralytics attention matmul counter; GFLOPs=2*MACs/1e9",
        "thop_profile_signature": str(inspect.signature(thop_profile)),
        "thop_distribution_versions": thop_versions(),
    }


def thop_versions():
    """Catat kedua distribution yang dapat menyediakan namespace thop yang sama."""
    result = {}
    for name in ("thop", "ultralytics-thop"):
        try:
            result[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            result[name] = None
    return result

In [ ]:
TARGET_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else torch.device(DEVICE)
ENVIRONMENT = environment_report()
print("Runtime", PLATFORM, "|", TARGET_DEVICE, "|", ENVIRONMENT["gpu"], "|", SUITE_DIR)
display(pd.DataFrame(ENVIRONMENT["packages"].items(), columns=["package", "version"]))

## 04 — Pengambilan Baseline

Periksa seed, versi W&B, dan SHA-256 `best.pt` sumber. Suite memeriksa metadata seed dan SHA-256
unduhan sebelum memuat model. Salinan dalam `LOCAL_BASELINE_PATHS` harus cocok dengan SHA-256 seed
yang bersangkutan. Baseline dimuat kembali dari `best.pt` untuk setiap perlakuan.

In [ ]:
def select_checkpoint(root, selector="best.pt"):
    """Pilih checkpoint unik secara eksplisit; tidak menebak dari urutan filesystem."""
    root = Path(root).expanduser().resolve()
    if root.is_file():
        candidates = [root] if root.suffix.lower() == ".pt" else []
        base = root.parent
    else:
        candidates = sorted(p.resolve() for p in root.rglob("*.pt") if p.is_file())
        base = root
    print("Kandidat checkpoint:", [p.relative_to(base).as_posix() for p in candidates])
    if not candidates:
        raise FileNotFoundError(f"Tidak ada file .pt di {root}.")
    if root.is_file():
        return root  # LOCAL_CHECKPOINT_PATH berupa file adalah pilihan eksplisit.
    if selector:
        selector = str(selector).replace("\\", "/")
        matches = [p for p in candidates if p.relative_to(base).as_posix() == selector] if "/" in selector else [p for p in candidates if p.name == selector]
        if len(matches) == 1:
            return matches[0]
        raise ValueError(f"Selector {selector!r} memiliki {len(matches)} kecocokan; pilih path relatif tepat.")
    if len(candidates) == 1 and candidates[0].name == "best.pt":
        return candidates[0]
    raise ValueError("Checkpoint ambigu atau best.pt tidak tersedia; isi CHECKPOINT_SELECTOR secara eksplisit.")


def checkpoint_provenance(path, artifact_name, *, expected_hash=None, resolved_artifact=None):
    """Catat identitas sumber; hash lokal opsional wajib cocok bila diberikan."""
    digest = sha256_file(path)
    if expected_hash is not None and digest.lower() != expected_hash.lower():
        raise ValueError("SHA-256 checkpoint lokal tidak cocok dengan LOCAL_EXPECTED_SHA256.")
    return {"requested_artifact": artifact_name, "resolved_artifact": resolved_artifact,
            "checkpoint_name": Path(path).name, "checkpoint_path": str(Path(path).resolve()),
            "sha256": digest, "file_bytes": Path(path).stat().st_size,
            "source_verification": "wandb_download" if resolved_artifact else ("expected_sha256_matched" if expected_hash else "user_asserted_local_copy")}

In [ ]:
def ms_wandb_environment():
    if not os.environ.get("WANDB_API_KEY") and PLATFORM == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("WANDB_API_KEY")
            if key:
                os.environ["WANDB_API_KEY"] = key
        except Exception as error:
            raise RuntimeError("Lampirkan Kaggle Secret WANDB_API_KEY atau setel variabel environment.") from error
    if not os.environ.get("WANDB_API_KEY"):
        raise RuntimeError("Setel WANDB_API_KEY pada environment atau Kaggle Secret.")
    if os.environ.get("WANDB_MODE", "online") not in {"online", "shared"}:
        raise RuntimeError("Aktifkan mode online W&B untuk mengunduh atau mempublikasikan model.")


def ms_source(folder, seed):
    cache = folder / "source"
    cache.mkdir(parents=True, exist_ok=True)
    journal = cache / "provenance.json"
    prior = ms_read(journal)
    reference = workflow_version_ref(BASELINE_SOURCES[seed]["ref"])
    expected = BASELINE_SOURCES[seed]["sha256"]
    if prior:
        path = cache / prior["relative_path"]
        ms_check_source(path, prior, seed)
        return path, prior

    if seed in LOCAL_BASELINE_PATHS:
        local = select_checkpoint(LOCAL_BASELINE_PATHS[seed], "best.pt")
        record = checkpoint_provenance(local, reference, expected_hash=expected)
        path = cache / "best.pt"
        if not path.is_file():
            temporary = path.with_suffix(".pt.tmp")
            shutil.copyfile(local, temporary)
            os.replace(temporary, path)
    else:
        ms_wandb_environment()
        artifact = wandb.Api(timeout=30).artifact(reference, type="model")
        if artifact.qualified_name != reference or artifact.metadata.get("seed") != seed:
            raise ValueError(f"Versi atau metadata seed baseline W&B tidak cocok untuk seed {seed}.")
        path = Path(artifact.get_path("best.pt").download(root=str(cache))).resolve()
        record = checkpoint_provenance(path, reference, expected_hash=expected, resolved_artifact=artifact.qualified_name)
    record.pop("checkpoint_path")
    record.update(relative_path=path.resolve().relative_to(cache.resolve()).as_posix(), model_variant="s", baseline_seed=seed)
    ms_check_source(path, record, seed)
    write_json(journal, record)
    return path, record

In [ ]:
display(pd.DataFrame([{"baseline_seed": seed, "artifact": source["ref"], "source_sha256": source["sha256"],
                       "local_checkpoint": LOCAL_BASELINE_PATHS.get(seed)} for seed, source in BASELINE_SOURCES.items()]))

## 05 — Audit Arsitektur YOLO26s

Audit backbone, neck, dan Detect memeriksa identitas skala YOLO26s, dimensi keluaran, urutan kelas,
serta keberadaan cabang one-to-many dan one-to-one. Inventaris menyimpan tipe, asal fitur, dan
parameter tiap stage. Counter operasi memakai model FP32 unfused.

In [ ]:
def ordered_names(names):
    """Pertahankan identitas dan urutan indeks kelas."""
    if isinstance(names, dict):
        indexed = {int(k): str(v) for k, v in names.items()}
        if sorted(indexed) != list(range(len(indexed))):
            raise ValueError("Indeks kelas wajib berurutan mulai dari nol.")
        return [indexed[i] for i in range(len(indexed))]
    return [str(value) for value in names]


def assert_class_match(model_names, dataset_names):
    """Jangan remap kelas diam-diam untuk membuat evaluasi berjalan."""
    if ordered_names(model_names) != ordered_names(dataset_names):
        raise ValueError(f"Urutan/nama kelas berbeda: model={ordered_names(model_names)}, dataset={ordered_names(dataset_names)}")


def architecture_audit(core, model_variant="n"):
    """Bangun allowlist hidden channel berdasarkan tipe modul dan hubungan konsumen."""
    if not isinstance(core, DetectionModel):
        raise TypeError("Checkpoint bukan DetectionModel Ultralytics.")
    heads = [(name, module) for name, module in core.named_modules() if isinstance(module, Detect)]
    if len(heads) != 1:
        raise ValueError("Diperlukan tepat satu Detect head.")
    head_name, head = heads[0]
    cfg = core.yaml
    first_conv = next(module for module in core.modules() if isinstance(module, torch.nn.Conv2d))
    if "yolo26" not in str(cfg.get("yaml_file", "")).lower() or cfg.get("reg_max") != 1 or first_conv.out_channels != {"n": 16, "s": 32}[model_variant]:
        raise ValueError("Metadata/struktur tidak cocok dengan YOLO26 N/S standar; audit manual diperlukan.")
    if cfg.get("scale") not in (None, "", model_variant):
        raise ValueError("Checkpoint bukan skala model yang dipilih.")
    branches = ("cv2", "cv3", "one2one_cv2", "one2one_cv3")
    if any(getattr(head, attr, None) is None or len(getattr(head, attr)) != head.nl for attr in branches):
        raise ValueError("Kedua cabang head wajib tersedia; checkpoint fused/head berubah ditolak.")
    names = ordered_names(core.names)
    if len(names) != head.nc or head.no != head.nc + 4 * head.reg_max:
        raise ValueError("Metadata kelas/dimensi Detect tidak konsisten.")
    modules = dict(core.named_modules())
    attention_prefixes = [name for name, m in modules.items() if isinstance(m, (C2PSA, Attention, AAttn))]
    allowed = {}
    for name, module in modules.items():
        if not isinstance(module, Bottleneck) or name.startswith(head_name + "."):
            continue
        if any(name == prefix or name.startswith(prefix + ".") for prefix in attention_prefixes):
            continue
        producer, consumer = module.cv1.conv, module.cv2.conv
        bn = getattr(module.cv1, "bn", None)
        if not isinstance(bn, torch.nn.BatchNorm2d) or producer.groups != 1 or consumer.groups != 1:
            raise ValueError(f"Bottleneck {name} tidak mendukung kebijakan hidden-channel konservatif.")
        if producer.out_channels != consumer.in_channels or producer.out_channels != bn.num_features:
            raise ValueError(f"Dimensi hidden channel {name} tidak konsisten.")
        allowed[name + ".cv1.conv"] = {"root": producer, "bn": bn, "consumer": consumer,
                                        "initial_channels": producer.out_channels, "block": name}
    if not allowed:
        raise ValueError("Tidak ditemukan hidden-channel group yang eligible.")
    backbone_count = len(cfg["backbone"])
    stages = []
    for i, module in enumerate(core.model):
        stages.append({"index": i, "type": type(module).__name__, "from": module.f,
                       "region": "detection_head" if module is head else ("backbone" if i < backbone_count else "neck"),
                       "parameters": sum(p.numel() for p in module.parameters())})
    inventory = [{"name": name, "type": type(module).__name__,
                  "residual_add": bool(getattr(module, "add", False))}
                 for name, module in modules.items()
                 if isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d, Bottleneck, C2PSA, Attention, AAttn))
                 or type(module).__name__ in {"Concat", "C3k2", "SPPF"}]
    protection = []
    for name, module in modules.items():
        if isinstance(module, torch.nn.Conv2d) and name not in allowed:
            reason = "head one2many/one2one utuh" if name.startswith(head_name + ".") else "dimensi CSP/residual/attention/antartahap dilindungi"
            protection.append({"name": name, "reason": reason,
                               "protected_dimension": "all" if name.startswith(head_name + ".") else "out_channels",
                               "dependent_input_pruning_allowed": any(module is entry["consumer"] for entry in allowed.values())})
    return {"head": head, "head_name": head_name, "eligible": allowed, "modules": modules,
            "report": {"stages": stages, "module_inventory": inventory, "protected_layers": protection,
                       "eligible_layers": list(allowed), "head_feature_sources": list(head.f),
                       "class_names": names, "nc": head.nc, "reg_max": head.reg_max, "nl": head.nl,
                       "end2end": head.end2end, "unknown_policy": "fail explicitly"}}


def raw_branches(output):
    """Ambil raw kedua cabang tanpa mengganti forward atau algoritme deteksi."""
    if not isinstance(output, tuple) or len(output) != 2 or not isinstance(output[1], dict):
        raise ValueError("Output eval YOLO26 harus (decoded, raw_predictions).")
    branches = output[1]
    if set(branches) != {"one2many", "one2one"}:
        raise ValueError(f"Cabang YOLO26 tidak lengkap: {list(branches)}")
    return branches


def trace_outputs(output):
    """Root autograd mencakup boxes/scores kedua cabang, tanpa top-k/NMS."""
    branches = raw_branches(output)
    return [branches[branch][key] for branch in ("one2many", "one2one") for key in ("boxes", "scores")]


def finite_tensors(value):
    """Validasi semua tensor, termasuk feature maps dalam output nested."""
    if isinstance(value, torch.Tensor):
        if not torch.isfinite(value).all():
            raise ValueError("Output model mengandung NaN/Inf.")
    elif isinstance(value, dict):
        for item in value.values():
            finite_tensors(item)
    elif isinstance(value, (tuple, list)):
        for item in value:
            finite_tensors(item)


def forward_snapshot(core, image):
    """Forward FP32, cek kontrak raw/decoded, simpan tensor kecil untuk restore parity."""
    core.eval()
    with torch.no_grad():
        output = core(image)
    finite_tensors(output)
    branches = raw_branches(output)
    head = core.model[-1]
    snapshot = {"decoded": output[0].detach().cpu().clone()}
    anchors = None
    for branch in ("one2many", "one2one"):
        boxes, scores = branches[branch]["boxes"], branches[branch]["scores"]
        if boxes.ndim != 3 or scores.ndim != 3 or boxes.shape[:2] != (image.shape[0], 4 * head.reg_max):
            raise ValueError(f"Bentuk boxes {branch} invalid: {tuple(boxes.shape)}")
        if scores.shape[:2] != (image.shape[0], head.nc) or boxes.shape[2] != scores.shape[2]:
            raise ValueError(f"Bentuk scores {branch} tidak cocok dengan jumlah kelas.")
        if anchors is not None and anchors != boxes.shape[2]:
            raise ValueError("Jumlah anchor kedua cabang berbeda.")
        anchors = boxes.shape[2]
        snapshot[branch + "/boxes"] = boxes.detach().cpu().clone()
        snapshot[branch + "/scores"] = scores.detach().cpu().clone()
    decoded = snapshot["decoded"]
    if head.end2end:
        if decoded.ndim != 3 or decoded.shape[0] != image.shape[0] or decoded.shape[-1] != 6:
            raise ValueError("Output end-to-end bukan [batch, detections, 6].")
        ids = decoded[..., 5]
        if ((ids < 0) | (ids >= head.nc) | (ids != ids.floor())).any():
            raise ValueError("Indeks kelas pada output deteksi invalid.")
    elif decoded.ndim != 3 or decoded.shape[:2] != (image.shape[0], head.nc + 4):
        raise ValueError("Output one-to-many tidak memiliki dimensi nc+4.")
    return snapshot


def model_measurements(core, imgsz):
    """Hitung Conv/BN dan attention matmul tanpa menghilangkan child Conv attention."""
    parameters = sum(p.numel() for p in core.parameters())
    copy = deepcopy(core).eval()
    parameter = next(copy.parameters())
    sample = torch.zeros(1, 3, imgsz, imgsz, device=parameter.device, dtype=torch.float32)
    attention_macs = []
    def count_attention(module, inputs, output):
        counter = SimpleNamespace(total_ops=0, num_heads=module.num_heads, head_dim=module.head_dim,
                                  key_dim=getattr(module, "key_dim", module.head_dim), area=getattr(module, "area", 1))
        _attention_ops(counter, inputs, output)
        attention_macs.append(float(counter.total_ops))
    handles = [module.register_forward_hook(count_attention) for module in copy.modules()
               if isinstance(module, (Attention, AAttn))]
    try:
        with torch.no_grad():
            # A custom Attention handler in classic THOP would hide its child Conv/BN counts.
            macs, _ = thop_profile(copy, inputs=(sample,), verbose=False)
    finally:
        for handle in handles:
            handle.remove()
    gflops = (float(macs) + sum(attention_macs)) * 2 / 1e9
    if not math.isfinite(gflops) or gflops <= 0:
        raise RuntimeError("Profiling GFLOPs gagal/bernilai nol; cek thop dan source dependency.")
    return {"parameters": parameters, "gflops": gflops, "precision": "FP32", "fusion": False}


def head_snapshot(head):
    """Salin parameter/buffer serta kontrak dimensi seluruh head yang dilindungi."""
    return {"state": {key: value.detach().cpu().clone() for key, value in head.state_dict().items()},
            "contract": (head.nc, head.no, head.nl, head.reg_max, head.end2end)}


def assert_head_unchanged(head, before):
    """Pruning tersembunyi tidak boleh menyentuh bobot atau dimensi head."""
    after = head_snapshot(head)
    if after["contract"] != before["contract"] or after["state"].keys() != before["state"].keys():
        raise ValueError("Kontrak/struktur detection head berubah.")
    if any(not torch.equal(after["state"][key], value) for key, value in before["state"].items()):
        raise ValueError("Parameter/buffer detection head berubah.")

In [ ]:
def assert_layer_consistency(core):
    """Dimensi atribut modul harus sesuai dengan tensor parameter/buffer aktual."""
    for name, module in core.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            expected = (module.out_channels, module.in_channels // module.groups, *module.kernel_size)
            if tuple(module.weight.shape) != expected or module.in_channels % module.groups or module.out_channels % module.groups:
                raise ValueError(f"Bentuk Conv invalid pada {name}.")
        elif isinstance(module, torch.nn.BatchNorm2d):
            if any(t.numel() != module.num_features for t in (module.weight, module.bias, module.running_mean, module.running_var)):
                raise ValueError(f"Bentuk BatchNorm invalid pada {name}.")

## 06 — Audit Dependency Graph dan Eligible Layer

DepGraph menelusuri dimensi channel yang terikat antara Conv, BatchNorm, dan konsumennya.
Audit mencatat group eligible, alasan penolakan atau proteksi, serta indeks channel asli.
Pemetaan mengikuti residual, offset concat, dan seluruh cabang fan-out.

In [ ]:
"""Standalone LAMP V2 policy source, embedded verbatim in the Kaggle notebook.

This file is an authoring source, not a notebook runtime dependency.
"""
import math
import warnings

import torch
import torch_pruning as tp
from ultralytics.nn.modules import Bottleneck, C2f, C3k2, Detect
from ultralytics.nn.modules.block import C2PSA, Attention, PSABlock
from ultralytics.nn.tasks import DetectionModel


def _v2_trace_outputs(output):
    if not isinstance(output, tuple) or len(output) != 2 or not isinstance(output[1], dict):
        raise ValueError("Output YOLO26 eval harus (decoded, raw).")
    if set(output[1]) != {"one2many", "one2one"}:
        raise ValueError("Graph wajib mencakup raw output kedua cabang.")
    return [output[1][branch][key] for branch in ("one2many", "one2one")
            for key in ("boxes", "scores")]


def _v2_policy_structure(core, model_variant=None):
    if not isinstance(core, DetectionModel):
        raise TypeError("Policy memerlukan DetectionModel YOLO26 unfused.")
    modules = dict(core.named_modules())
    heads = [(name, module) for name, module in modules.items() if isinstance(module, Detect)]
    if len(heads) != 1:
        raise ValueError("Diperlukan satu Detect head.")
    head_name, head = heads[0]
    if any(getattr(head, name, None) is None for name in ("cv2", "cv3", "one2one_cv2", "one2one_cv3")):
        raise ValueError("Detection head kedua cabang harus unfused dan lengkap.")
    first = next(module for module in modules.values() if isinstance(module, torch.nn.Conv2d))
    actual_variant = core.yaml.get("scale")
    if actual_variant in (None, ""):
        actual_variant = {16: "n", 32: "s"}.get(first.out_channels)
    if model_variant is not None and model_variant not in ("n", "s"):
        raise ValueError("Model variant harus n atau s.")
    if ("yolo26" not in str(core.yaml.get("yaml_file", "")).lower()
            or core.yaml.get("reg_max") != 1 or actual_variant not in ("n", "s")
            or first.out_channels != {"n": 16, "s": 32}.get(actual_variant)):
        raise ValueError("V2 memerlukan arsitektur standar YOLO26n/S dengan stem 16/32 yang sesuai.")
    if model_variant is not None and model_variant != actual_variant:
        raise ValueError(f"Model variant tidak cocok: diminta {model_variant}, checkpoint {actual_variant}.")
    envelopes = [(name, module) for name, module in modules.items()
                 if isinstance(module, (Detect, C2PSA, PSABlock, Attention))
                 or type(module).__name__ == "AAttn"]
    protected = {}
    for name, module in modules.items():
        for prefix, envelope in envelopes:
            if name == prefix or name.startswith(prefix + "."):
                protected[module] = "detection_head_all_dimensions" if isinstance(envelope, Detect) else "attention_envelope_all_dimensions"
                break
    split_outputs = {}
    for name, module in modules.items():
        if isinstance(module, (C2f, C3k2)):
            for child in (module.cv1.conv, getattr(module.cv1, "bn", None)):
                if child is not None:
                    split_outputs[child] = "CSP_split_output_dimension"
    conservative = {}
    for name, module in modules.items():
        if not isinstance(module, Bottleneck) or module in protected:
            continue
        producer, consumer = module.cv1.conv, module.cv2.conv
        bn = getattr(module.cv1, "bn", None)
        if (not isinstance(bn, torch.nn.BatchNorm2d) or producer.groups != 1 or consumer.groups != 1
                or producer.out_channels != consumer.in_channels or producer.out_channels != bn.num_features):
            raise ValueError(f"Bottleneck hidden channel tidak didukung: {name}")
        conservative[producer] = {"name": name + ".cv1.conv", "bn": bn, "consumer": consumer}
    if not conservative:
        raise ValueError("Tidak ada kandidat konservatif.")
    return {"model_variant": actual_variant, "modules": modules, "names": {module: name for name, module in modules.items()},
            "head": head, "head_name": head_name, "protected": protected,
            "split_outputs": split_outputs, "conservative": conservative}


def protected_policy_snapshot(core, model_variant=None):
    """Snapshot seluruh bobot, buffer, dan kontrak head/attention yang dilindungi."""
    structure = _v2_policy_structure(core, model_variant)
    result = {}
    for name, module in structure["modules"].items():
        if isinstance(module, (Detect, C2PSA, PSABlock, Attention)) or type(module).__name__ == "AAttn":
            result[name] = {"state": {key: value.detach().cpu().clone() for key, value in module.state_dict().items()},
                            "dimensions": {key: getattr(module, key) for key in ("nc", "no", "nl", "reg_max", "end2end", "c")
                                           if hasattr(module, key)}}
    return result


def assert_policy_protected_unchanged(core, snapshot):
    modules = dict(core.named_modules())
    for name, entry in snapshot.items():
        if name not in modules:
            raise ValueError(f"Envelope terlindungi hilang: {name}")
        module = modules[name]
        after = module.state_dict()
        if after.keys() != entry["state"].keys() or any(
                not torch.equal(value.detach().cpu(), entry["state"][key]) for key, value in after.items()):
            raise ValueError(f"Bobot/buffer envelope terlindungi berubah: {name}")
        if any(getattr(module, key) != value for key, value in entry["dimensions"].items()):
            raise ValueError(f"Kontrak envelope terlindungi berubah: {name}")


def _v2_output_signature(group, graph, names):
    """Identitas satu group adalah seluruh dimensi Conv-output yang terikat."""
    return tuple(sorted(names[dep.target.module] for dep, _ in group
                        if isinstance(dep.target.module, torch.nn.Conv2d)
                        and graph.is_out_channel_pruning_fn(dep.handler)))


def _v2_validate_group(group, graph, structure, policy):
    root = group[0].dep.target.module
    root_name = structure["names"].get(root, "<unknown>")
    entry = structure["conservative"].get(root)
    if policy == "conservative" and entry is None:
        raise ValueError("root_not_conservative_hidden_channel")
    for dep, indices in group:
        module, handler = dep.target.module, dep.handler
        name = structure["names"].get(module, type(module).__name__)
        output = graph.is_out_channel_pruning_fn(handler)
        if module in structure["protected"]:
            raise ValueError(f"{name}: {structure['protected'][module]}")
        if output and module in structure["split_outputs"]:
            raise ValueError(f"{name}: {structure['split_outputs'][module]}")
        if isinstance(module, torch.nn.Conv2d):
            if not output and not graph.is_in_channel_pruning_fn(handler):
                raise ValueError(f"{name}: unknown_conv_pruning_handler")
            if module.groups != 1:
                raise ValueError(f"{name}: grouped_or_depthwise_conv_unsupported")
            if policy == "conservative" and not (
                    (module is root and output) or (module is entry["consumer"] and graph.is_in_channel_pruning_fn(handler))):
                raise ValueError(f"{name}: outside_conservative_hidden_channel")
        elif isinstance(module, torch.nn.BatchNorm2d):
            if policy == "conservative" and module is not entry["bn"]:
                raise ValueError(f"{name}: outside_conservative_batchnorm")
        else:
            kind = type(module).__name__
            if kind not in {"_ElementWiseOp", "_ConcatOp"}:
                raise ValueError(f"{name}: unsupported_operation_{kind}")
            if kind == "_ElementWiseOp":
                operation = str(module)
                supported = ("SiluBackward", "AddBackward", "MaxPool2DWithIndicesBackward", "UpsampleNearest2DBackward", "CloneBackward")
                if not any(token.lower() in operation.lower() for token in supported):
                    raise ValueError(f"{name}: unsupported_elementwise_{operation}")
        # Torch-Pruning maps concat offsets and coupled residual identities. Reject invalid maps.
        mapped = [int(index) for index in indices]
        if len(mapped) != len(set(mapped)) or any(index < 0 for index in mapped):
            raise ValueError(f"{name}: invalid_or_duplicate_index_mapping")
        channels = graph.get_out_channels(module) if output else graph.get_in_channels(module)
        if channels is not None and any(index >= channels for index in mapped):
            raise ValueError(f"{name}: dependency_index_out_of_bounds")
    return root_name


def _v2_channel_ledger(structure):
    """Track each physical model dimension independently in original index space."""
    ledger = {}
    for name, module in structure["modules"].items():
        if isinstance(module, torch.nn.Conv2d):
            dimensions = {"in": module.in_channels, "out": module.out_channels}
        elif isinstance(module, torch.nn.BatchNorm2d):
            dimensions = {"channel": module.num_features}
        else:
            continue
        for dimension, count in dimensions.items():
            ledger[(module, dimension)] = {
                "layer": name, "module_type": type(module).__name__, "dimension": dimension,
                "channels_before": count, "kept": list(range(count)), "group_ids": set()}
    return ledger


def _v2_dependency_dimension(dependency, graph):
    module = dependency.target.module
    if isinstance(module, torch.nn.BatchNorm2d):
        return module, "channel"
    if isinstance(module, torch.nn.Conv2d):
        return module, "out" if graph.is_out_channel_pruning_fn(dependency.handler) else "in"
    return None


def _v2_refresh_channel_report(audit):
    mappings, masks = [], []
    for entry in audit["channel_ledger"].values():
        kept = set(entry["kept"])
        removed = [index for index in range(entry["channels_before"]) if index not in kept]
        row = {key: entry[key] for key in ("layer", "module_type", "dimension", "channels_before")}
        row.update(channels_after=len(entry["kept"]), kept_original_indices=list(entry["kept"]),
                   removed_original_indices=removed, dependency_group_ids=sorted(entry["group_ids"]))
        mappings.append(row)
        masks.append({"layer": row["layer"], "module_type": row["module_type"],
                      "dimension": row["dimension"], "channels_before": row["channels_before"],
                      "channels_after": row["channels_after"],
                      "dependency_group_ids": row["dependency_group_ids"],
                      "kept_mask": [index in kept for index in range(entry["channels_before"])]})
    audit["report"]["channel_mapping"] = mappings
    audit["report"]["channel_masks"] = masks


def _v2_capacity(entry):
    current = entry["root"].out_channels
    capacity = current - 1
    for module, initial in entry["output_dimensions"].items():
        capacity = min(capacity, math.floor(initial * 0.5) - (initial - module.out_channels))
    return max(0, capacity)


def build_policy_pruner(core, image, policy, ratio, model_variant="n"):
    """Discover, audit and canonicalize dependencies before global LAMP ranking."""
    if policy not in ("conservative", "extended"):
        raise ValueError("Policy harus conservative atau extended.")
    if not 0 < float(ratio) <= 0.5:
        raise ValueError("Rasio harus di dalam (0, 0.50].")
    core.eval().requires_grad_(True)
    structure = _v2_policy_structure(core, model_variant)
    importance = tp.importance.LAMPImportance(p=2, group_reduction="mean", bias=False)
    initial_ignored = [module for module in structure["modules"].values()
                       if isinstance(module, torch.nn.Conv2d) and
                       (module in structure["protected"] or module in structure["split_outputs"]
                        or (policy == "conservative" and module not in structure["conservative"]))]
    with torch.enable_grad(), warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        pruner = tp.pruner.BasePruner(core, image, importance=importance,
            global_pruning=True, pruning_ratio=float(ratio), iterative_steps=1,
            max_pruning_ratio=0.50, round_to=None, root_module_types=[torch.nn.Conv2d],
            ignored_layers=initial_ignored,
            forward_fn=lambda model, example: model(example), output_transform=_v2_trace_outputs)
    graph = pruner.DG
    eligible, rejected, seen, seen_outputs = {}, [], set(), set()
    # Inspect every Conv root explicitly, including rejected roots. The later canonical
    # scan follows BasePruner's own graph order to preserve V1 ranking semantics.
    for name, module in structure["modules"].items():
        if not isinstance(module, torch.nn.Conv2d):
            continue
        if module in initial_ignored:
            reason = structure["protected"].get(module) or structure["split_outputs"].get(module) or "root_not_conservative_hidden_channel"
            rejected.append({"root": name, "reason": reason})
            continue
        group = graph.get_pruning_group(module, tp.prune_conv_out_channels, list(range(module.out_channels)))
        try:
            _v2_validate_group(group, graph, structure, policy)
        except ValueError as error:
            rejected.append({"root": name, "reason": str(error)})
    rejected_roots = {entry["root"] for entry in rejected}
    ignored = [module for name, module in structure["modules"].items()
               if isinstance(module, torch.nn.Conv2d) and name in rejected_roots]
    for group in graph.get_all_groups(ignored_layers=ignored, root_module_types=[torch.nn.Conv2d]):
        root_name = _v2_validate_group(group, graph, structure, policy)
        signature = _v2_output_signature(group, graph, structure["names"])
        if not signature or signature in seen:
            continue
        overlap = seen_outputs.intersection(signature)
        if overlap:
            raise ValueError(f"Canonical dependency groups overlap output dimensions: {sorted(overlap)}")
        seen.add(signature)
        seen_outputs.update(signature)
        root = group[0].dep.target.module
        score = importance(group)
        if score is None or score.numel() != root.out_channels or not torch.isfinite(score).all():
            raise ValueError(f"Importance LAMP invalid: {root_name}")
        dimensions = {dep.target.module: int(dep.target.module.out_channels)
                      for dep, _ in group if isinstance(dep.target.module, torch.nn.Conv2d)
                      and graph.is_out_channel_pruning_fn(dep.handler)}
        membership = {dep.target.module: ("out" if graph.is_out_channel_pruning_fn(dep.handler) else "in")
                      for dep, _ in group
                      if isinstance(dep.target.module, (torch.nn.Conv2d, torch.nn.BatchNorm2d))}
        capacity = min(root.out_channels - 1, *(math.floor(initial * 0.5) for initial in dimensions.values()))
        permitted = set(torch.argsort(score, stable=True)[:capacity].detach().cpu().tolist())
        eligible[root] = {"name": root_name, "root": root, "signature": signature,
                          "initial_channels": root.out_channels, "output_dimensions": dimensions,
                          "member_dimensions": membership,
                          "initial_lamp_scores": score.detach().cpu().tolist(),
                          "initial_capacity_mask": [index in permitted for index in range(root.out_channels)],
                          "group_id": f"{policy}-group-{len(eligible):03d}",
                          "dependencies": [str(dep) for dep, _ in group]}
    if not eligible:
        raise ValueError("Audit tidak menemukan group aman untuk policy ini.")
    pruner.ignored_layers = ignored
    audit = {"structure": structure, "eligible": eligible, "policy": policy, "ratio": float(ratio),
             "model_variant": structure["model_variant"], "channel_ledger": _v2_channel_ledger(structure),
             "independent_channels_before": sum(entry["initial_channels"] for entry in eligible.values()),
             "physical_conv_output_channels_before": sum(sum(entry["output_dimensions"].values()) for entry in eligible.values())}
    audit["report"] = {"model_variant": structure["model_variant"], "policy": policy,
         "requested_ratio": float(ratio), "max_pruning_ratio": 0.5,
         "importance": "LAMPImportance(p=2, group_reduction='mean', bias=False)",
         "ranking": "BasePruner global one-step; capacity exclusions are +inf sentinels, finite eligible LAMP scores unchanged",
         "group_count": len(eligible), "independent_channels_before": audit["independent_channels_before"],
         "physical_conv_output_channels_before": audit["physical_conv_output_channels_before"],
         "eligible_groups": [{key: value for key, value in entry.items() if key not in ("root", "output_dimensions", "member_dimensions")}
                             for entry in eligible.values()],
         "rejected_groups": rejected, "warnings": [str(item.message) for item in captured],
         "lamp_scores": [{"group_id": entry["group_id"], "root": entry["name"],
                          "channel_idx": index, "score": score,
                          "capacity_eligible": entry["initial_capacity_mask"][index], "removed": False}
                         for entry in eligible.values() for index, score in enumerate(entry["initial_lamp_scores"])],
         "channel_mapping_status": "RECORDED_FROM_LIVE_DEPENDENCY_GRAPH",
         "channel_mapping_index_space": "Original model dimension indices; current indices translated before every successful prune"}
    for entry in eligible.values():
        group = graph.get_pruning_group(entry["root"], tp.prune_conv_out_channels,
                                        list(range(entry["initial_channels"])))
        for dependency, _ in group:
            dimension = _v2_dependency_dimension(dependency, graph)
            if dimension is not None:
                audit["channel_ledger"][dimension]["group_ids"].add(entry["group_id"])
    _v2_refresh_channel_report(audit)

    def capacity_limited_lamp(group):
        _v2_validate_group(group, graph, structure, policy)
        entry = eligible.get(group[0].dep.target.module)
        if entry is None:
            raise ValueError("Proposal berakar di luar group yang diaudit.")
        score = importance(group)
        current = entry["root"].out_channels
        if score is None or score.numel() != current or not torch.isfinite(score).all():
            raise ValueError("Importance LAMP tidak finite sebelum cap.")
        # Bound every coupled output dimension, preserving the original LAMP score.
        capacity = _v2_capacity(entry)
        permitted = torch.argsort(score, stable=True)[:capacity]
        capped = torch.full_like(score, float("inf"))
        capped[permitted] = score[permitted]
        return capped
    pruner.importance = capacity_limited_lamp
    return pruner, audit


def apply_policy_pruning(pruner, audit):
    """Audit each concrete proposal before irreversible tensor shape mutation."""
    history = []
    graph, structure = pruner.DG, audit["structure"]
    # The runtime report retains this live list even if a later proposal raises.
    audit["report"]["pruning_history"] = history
    audit["report"]["pruning_status"] = "IN_PROGRESS"
    for group in pruner.step(interactive=True):
        audit["report"]["last_proposal"] = {
            "root": structure["names"].get(group[0].dep.target.module, "<unknown>"),
            "removed_indices": [int(index) for index in group[0].idxs]}
        name = _v2_validate_group(group, graph, structure, audit["policy"])
        entry = audit["eligible"].get(group[0].dep.target.module)
        if entry is None:
            raise ValueError("Group proposal tidak terdapat pada audit.")
        indices = [int(index) for index in group[0].idxs]
        if not indices:
            continue
        if _v2_output_signature(group, graph, structure["names"]) != entry["signature"]:
            raise ValueError("Dimensi output proposal berbeda dari group canonical yang diaudit.")
        if not graph.check_pruning_group(group):
            raise ValueError(f"Dependency group tidak valid: {name}")
        mapped_dimensions = {}
        for dependency, mapped in group:
            dimension = _v2_dependency_dimension(dependency, graph)
            if dimension is None:
                continue
            # A residual/concat may reach a physical dimension along multiple paths.
            # The ledger records its union once, in the current dimension's space.
            mapped_dimensions.setdefault(dimension, set()).update(int(index) for index in mapped)
        dimension_records = []
        for dimension, mapped in mapped_dimensions.items():
            ledger = audit["channel_ledger"][dimension]
            current_indices = sorted(mapped)
            if any(index < 0 or index >= len(ledger["kept"]) for index in current_indices):
                raise ValueError(f"Ledger dependency index out of bounds: {ledger['layer']}:{dimension[1]}")
            dimension_records.append({"layer": ledger["layer"], "module_type": ledger["module_type"],
                "dimension": dimension[1], "current_indices": current_indices,
                "original_indices": [ledger["kept"][index] for index in current_indices],
                "channels_before": len(ledger["kept"]), "channels_after": len(ledger["kept"]) - len(mapped)})
        physical_removed = 0
        touched = []
        for (module, dimension), mapped in mapped_dimensions.items():
            if isinstance(module, torch.nn.Conv2d) and dimension == "out":
                if module not in entry["output_dimensions"]:
                    raise ValueError("Proposal mengubah output di luar group canonical.")
                initial = entry["output_dimensions"][module]
                remaining = module.out_channels - len(mapped)
                if remaining < 1 or initial - remaining > math.floor(initial * 0.5):
                    raise ValueError(f"Cap 50%/minimum satu channel dilanggar: {structure['names'][module]}")
                physical_removed += len(mapped)
                touched.append({"layer": structure["names"][module], "before": module.out_channels, "after": remaining})
        before = entry["root"].out_channels
        root_ledger = audit["channel_ledger"][(entry["root"], "out")]
        original_root_indices = [root_ledger["kept"][index] for index in indices]
        record = {"root": name, "group_id": entry["group_id"], "removed_indices": indices,
                  "removed_original_indices": original_root_indices, "mapped_dimensions": dimension_records,
                  "channels_before": before, "channels_after": before - len(indices),
                  "physical_conv_output_channels_removed": physical_removed,
                  "coupled_conv_output_count": len(touched),
                  "output_changes": touched, "dependencies": [str(dep) for dep, _ in group]}
        group.prune()
        for (module, dimension), mapped in mapped_dimensions.items():
            ledger = audit["channel_ledger"][(module, dimension)]
            ledger["kept"] = [original for index, original in enumerate(ledger["kept"]) if index not in mapped]
            ledger["group_ids"].add(entry["group_id"])
            actual = (module.in_channels if dimension == "in" else module.out_channels) if isinstance(
                module, torch.nn.Conv2d) else module.num_features
            if len(ledger["kept"]) != actual:
                raise ValueError(f"Ledger/model channel mismatch: {ledger['layer']}:{dimension}")
        removed_set = set(original_root_indices)
        for row in audit["report"]["lamp_scores"]:
            if row["group_id"] == entry["group_id"] and row["channel_idx"] in removed_set:
                row["removed"] = True
        _v2_refresh_channel_report(audit)
        history.append(record)
        audit["report"]["independent_channels_after"] = sum(entry["root"].out_channels for entry in audit["eligible"].values())
        audit["report"]["independent_channels_removed"] = sum(item["channels_before"] - item["channels_after"] for item in history)
        audit["report"]["physical_conv_output_channels_removed"] = sum(item["physical_conv_output_channels_removed"] for item in history)
    if not history:
        raise RuntimeError("Tidak ada channel struktural dihapus.")
    audit["report"]["pruning_status"] = "PASS"
    return history


def measure_layer_inventory(core, image, audit, policy):
    """Per-layer Conv/BN estimates at the actual input size, on an unfused model."""
    shapes, handles = {}, []
    for name, module in core.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d)):
            def hook(layer, args, output, name=name):
                shapes[name] = list(output.shape)
            handles.append(module.register_forward_hook(hook))
    try:
        with torch.no_grad():
            core(image)
    finally:
        for handle in handles:
            handle.remove()
    structure = audit["structure"]
    membership, member_dimensions = {}, {}
    for entry in audit["eligible"].values():
        for module, dimension in entry["member_dimensions"].items():
            membership.setdefault(module, []).append(entry["group_id"])
            member_dimensions.setdefault(module, set()).add(dimension)
    rejected = {item["root"]: item["reason"] for item in audit["report"]["rejected_groups"]}
    backbone_count = len(core.yaml.get("backbone", []))
    rows = []
    for name, module in core.named_modules():
        if not isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d)):
            continue
        shape = shapes.get(name, [])
        if isinstance(module, torch.nn.Conv2d):
            inputs, outputs = module.in_channels, module.out_channels
            operations = (2 * math.prod(shape) * (inputs // module.groups) * math.prod(module.kernel_size)) if shape else None
        else:
            inputs = outputs = module.num_features
            operations = 2 * math.prod(shape) if shape else None
        stage = next((i for i, block in enumerate(core.model)
                      if any(child is module for child in block.modules())), None)
        reason = structure["protected"].get(module) or structure["split_outputs"].get(module) or rejected.get(name)
        rows.append({"layer_name": name, "module_type": type(module).__name__,
                     "region": "detection_head" if module in structure["protected"] and str(reason).startswith("detection")
                               else ("backbone" if stage is not None and stage < backbone_count else "neck"),
                     "in_channels": inputs, "out_channels": outputs,
                     "parameters": sum(parameter.numel() for parameter in module.parameters(recurse=False)),
                     "estimated_flops": operations, "flops_estimate_method": "Conv:2*elements*Cin/groups*Kh*Kw; BN:2*elements; functional attention excluded",
                     "policy": policy, "eligible": module in membership,
                     "protection_reason": reason or ("not_in_eligible_dependency_group" if module not in membership else ""),
                     "dependency_group": ";".join(membership.get(module, [])),
                     "eligible_dimensions": ";".join(sorted(member_dimensions.get(module, set()))), "output_shape": shape})
    return rows

In [ ]:
"""Standalone analysis helpers embedded in notebook 03 by its generator.

All figures describe measured structural changes, not activation feature maps.
Missing historical mappings, scores, and validation values are never fabricated.
"""
import csv as _eda_csv
import json as _eda_json
import math as _eda_math
from pathlib import Path as _EdaPath

import matplotlib.pyplot as _eda_plt
import numpy as _eda_np
from matplotlib.colors import ListedColormap as _EdaColors


def _eda_write_json(path, value):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(_eda_json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def _eda_write_csv(path, rows):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = _eda_csv.DictWriter(stream, fieldnames=keys)
        writer.writeheader()
        writer.writerows({key: _eda_json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else value
                         for key, value in row.items()} for row in rows)


def _eda_finite(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and _eda_math.isfinite(value)


def _eda_save_figure(fig, path):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path.with_suffix(".png"), dpi=int(globals().get("EDA_DPI", 300)), bbox_inches="tight")
    fig.savefig(path.with_suffix(".svg"), bbox_inches="tight")
    _eda_plt.close(fig)
    return {"png": path.with_suffix(".png").name, "svg": path.with_suffix(".svg").name}


def layer_eda_rows(inventory_before, inventory_after):
    """Join actual unfused inventory by module name, preserving baseline order."""
    previous = {row["layer_name"]: row for row in inventory_before}
    current = {row["layer_name"]: row for row in inventory_after}
    if len(previous) != len(inventory_before) or len(current) != len(inventory_after):
        raise ValueError("EDA inventory has duplicate module names.")
    if set(previous) != set(current):
        raise ValueError("EDA before/after module names differ; layer alignment is unsafe.")
    rows = []
    for name, before in previous.items():
        after = current[name]
        if before["module_type"] != after["module_type"]:
            raise ValueError(f"EDA module type changed at {name}.")
        if after["out_channels"] > before["out_channels"] or after["in_channels"] > before["in_channels"]:
            raise ValueError(f"Pruning unexpectedly increased channels at {name}.")
        parts = name.split(".")
        stage = before.get("stage", ".".join(parts[:2]))
        row = {"layer_name": name, "module_type": before["module_type"], "stage": stage,
               "region": before.get("region", "unknown"), "policy": before.get("policy"),
               "eligible": before.get("eligible", False), "protection_reason": before.get("protection_reason", ""),
               "dependency_group": before.get("dependency_group", "")}
        for key in ("in_channels", "out_channels", "parameters", "estimated_flops"):
            row[key + "_before"] = before.get(key)
            row[key + "_after"] = after.get(key)
            base, value = before.get(key), after.get(key)
            row[key + "_removed"] = base - value if _eda_finite(base) and _eda_finite(value) else None
            row[key + "_reduction_pct"] = 100 * (base - value) / base if _eda_finite(base) and _eda_finite(value) and base else None
        if (row["protection_reason"] and str(row["protection_reason"]).startswith(("detection", "attention"))
                and (row["out_channels_removed"] or row["in_channels_removed"])):
            raise ValueError(f"Protected head/attention channel dimension changed at {name}.")
        rows.append(row)
    return rows


def stage_eda_rows(rows):
    """Physical Conv-output sums; these are distinct from independent group channels."""
    stages = {}
    for row in rows:
        key = (str(row["stage"]), row["region"])
        stage = stages.setdefault(key, {"stage": row["stage"], "region": row["region"], "conv_layers": 0,
                                      "physical_conv_outputs_before": 0, "physical_conv_outputs_after": 0,
                                      "parameters_before": 0, "parameters_after": 0,
                                      "estimated_flops_before": 0, "estimated_flops_after": 0})
        if row["module_type"] == "Conv2d":
            stage["conv_layers"] += 1
            for suffix in ("before", "after"):
                stage["physical_conv_outputs_" + suffix] += row["out_channels_" + suffix]
        for metric in ("parameters", "estimated_flops"):
            for suffix in ("before", "after"):
                value = row[metric + "_" + suffix]
                if _eda_finite(value):
                    stage[metric + "_" + suffix] += value
    for row in stages.values():
        row["physical_conv_outputs_removed"] = row["physical_conv_outputs_before"] - row["physical_conv_outputs_after"]
        for key in ("physical_conv_outputs", "parameters", "estimated_flops"):
            base, after = row[key + "_before"], row[key + "_after"]
            row[key + "_reduction_pct"] = 100 * (base - after) / base if base else None
    return list(stages.values())


def make_channel_figure(rows, title="Channel comparison"):
    """Three panels share exact Conv order and Y limits, as in the reference image."""
    if not rows or any(row["module_type"] != "Conv2d" for row in rows):
        raise ValueError("Channel diagram requires a nonempty ordered Conv2d inventory.")
    count = len(rows)
    width = max(16, min(32, count * 0.16))
    fig, axes = _eda_plt.subplots(3, 1, figsize=(width, 12), sharex=True, sharey=True)
    indices = _eda_np.arange(count)
    before = [row["out_channels_before"] for row in rows]
    after = [row["out_channels_after"] for row in rows]
    upper = max(max(before), max(after), 1) * 1.12
    axes[0].bar(indices, before, color="blue", width=0.82, label="Before Pruning")
    axes[1].bar(indices, after, color="red", width=0.82, label="After Pruning")
    axes[2].bar(indices, before, color="blue", width=0.82, label="Before Pruning")
    axes[2].bar(indices, after, color="red", width=0.82, label="After Pruning")
    for ax, heading in zip(axes, ("Before Pruning", "After Pruning", "Channel Comparison")):
        ax.set_title(heading)
        ax.set_ylabel("Output channels")
        ax.set_ylim(0, upper)
        ax.set_xlim(-0.8, count - 0.2)
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
        ax.legend(loc="upper right")
        ax.tick_params(axis="x", labelbottom=True)
        ax.set_xticks(indices, [row["layer_name"] for row in rows], rotation=90, fontsize=6 if count > 32 else 8)
        ax.set_xlabel("Conv module (baseline order)")
    fig.suptitle(title, y=1.005, fontsize=14)
    fig.tight_layout()
    return fig


def _eda_channel_plots(conv_rows, folder, title):
    _eda_write_csv(folder / "channel_plot_data.csv", conv_rows)
    files = [_eda_save_figure(make_channel_figure(conv_rows, title), folder / "channels_overview")]
    page_size = min(32, max(1, int(globals().get("EDA_MAX_LAYERS_PER_PAGE", 32))))
    for offset in range(0, len(conv_rows), page_size):
        page = offset // page_size + 1
        files.append(_eda_save_figure(make_channel_figure(conv_rows[offset:offset + page_size],
                     f"{title} — layers {offset + 1}–{min(offset + page_size, len(conv_rows))}"),
                     folder / f"channels_detail_{page:02d}"))
    fig, ax = _eda_plt.subplots(figsize=(max(14, min(32, len(conv_rows) * 0.16)), 5))
    ax.bar(range(len(conv_rows)), [row["out_channels_reduction_pct"] for row in conv_rows], color="red")
    ax.set_xticks(range(len(conv_rows)), [row["layer_name"] for row in conv_rows], rotation=90, fontsize=6)
    ax.set_ylabel("Output channels removed (%)")
    ax.set_title(title + " — physical Conv-output reduction")
    ax.set_ylim(0, 100)
    fig.tight_layout()
    files.append(_eda_save_figure(fig, folder / "channel_reduction_percent"))
    return {"status": "MEASURED", "conv_layers": len(conv_rows), "detail_page_size": page_size, "files": files}


def _eda_mapping_plots(audit, conv_rows, folder):
    mapping = audit.get("channel_mapping")
    if not mapping:
        return {"status": "UNAVAILABLE", "reason": "Original channel indices were not recorded; no reconstruction attempted."}
    _eda_write_json(folder / "channel_mapping.json", mapping)
    _eda_write_csv(folder / "channel_mapping.csv", mapping)
    masks = audit.get("channel_masks")
    if masks:
        _eda_write_json(folder / "channel_masks.json", masks)
    by_layer = {row["layer"]: row for row in mapping if row.get("module_type") == "Conv2d"
                and row.get("dimension") in {"out", "out_channels", "output", "output_channels"}}
    selected = []
    for row in conv_rows:
        item = by_layer.get(row["layer_name"])
        if item is None:
            return {"status": "UNAVAILABLE", "reason": "Mapping does not cover every Conv output.", "layer": row["layer_name"]}
        before = int(item["channels_before"])
        kept, removed = item.get("kept_original_indices"), item.get("removed_original_indices")
        if not isinstance(kept, list) or not isinstance(removed, list):
            raise TypeError(f"Missing original-index ledger for {row['layer_name']}.")
        if (len(set(kept)) != len(kept) or len(set(removed)) != len(removed)
                or set(kept) & set(removed) or set(kept) | set(removed) != set(range(before))
                or len(kept) != item["channels_after"]
                or before != row["out_channels_before"] or len(kept) != row["out_channels_after"]):
            raise ValueError(f"Original-index ledger disagrees with measured channels at {row['layer_name']}.")
        selected.append((row["layer_name"], before, set(removed)))
    page_size = min(32, max(1, int(globals().get("EDA_MAX_LAYERS_PER_PAGE", 32))))
    files = []
    for offset in range(0, len(selected), page_size):
        page = selected[offset:offset + page_size]
        width = max(item[1] for item in page)
        data = _eda_np.full((len(page), width), _eda_np.nan)
        for index, (_name, count, removed) in enumerate(page):
            data[index, :count] = 0
            if removed:
                data[index, sorted(removed)] = 1
        fig, ax = _eda_plt.subplots(figsize=(16, max(4, len(page) * 0.25)))
        cmap = _EdaColors(["#4477cc", "#ee3333"]).with_extremes(bad="#eeeeee")
        plot = ax.imshow(data, interpolation="nearest", aspect="auto", cmap=cmap, vmin=0, vmax=1)
        ax.set_yticks(range(len(page)), [item[0] for item in page], fontsize=7)
        ax.set_xlabel("Original Conv-output channel index (before any pruning)")
        ax.set_title("Original-index mask: blue kept, red removed, grey outside layer width")
        fig.colorbar(plot, ax=ax, ticks=[0, 1]).ax.set_yticklabels(["Kept", "Removed"])
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / f"channel_mask_{offset // page_size + 1:02d}"))
    return {"status": "MEASURED", "conv_layers": len(selected), "files": files,
            "meaning": "Physical Conv-output masks, including coupled dimensions; never summed as independent channels."}


def _eda_lamp_plots(audit, folder):
    scores = audit.get("lamp_scores")
    if not scores:
        return {"status": "UNAVAILABLE", "reason": "Finite pre-cap LAMP scores were not recorded."}
    if any(not _eda_finite(row.get("score")) for row in scores):
        raise ValueError("LAMP EDA requires finite original scores; capacity sentinels must not be plotted.")
    _eda_write_csv(folder / "lamp_scores.csv", scores)
    _eda_write_json(folder / "lamp_scores.json", scores)
    fig, ax = _eda_plt.subplots(figsize=(10, 5))
    for removed, color, label in ((False, "blue", "Kept"), (True, "red", "Removed")):
        values = [row["score"] for row in scores if bool(row.get("removed")) == removed]
        if values:
            ax.hist(values, bins=min(50, max(5, int(len(values) ** 0.5))), alpha=0.55, color=color, label=label)
    ax.set_xlabel("Finite LAMP score before capacity mask")
    ax.set_ylabel("Independent root channels")
    ax.set_title("LAMP selection — descriptive score distribution")
    ax.legend()
    fig.tight_layout()
    files = [_eda_save_figure(fig, folder / "lamp_score_distribution")]
    capacities = {}
    for row in scores:
        group = capacities.setdefault(row["group_id"], {"channels": 0, "capacity_eligible": 0, "removed": 0})
        group["channels"] += 1
        group["capacity_eligible"] += bool(row.get("capacity_eligible"))
        group["removed"] += bool(row.get("removed"))
    capacity_rows = [{"group_id": key, **value} for key, value in capacities.items()]
    _eda_write_csv(folder / "lamp_group_capacity.csv", capacity_rows)
    if any(row["removed"] > row["capacity_eligible"] for row in capacity_rows):
        raise ValueError("LAMP selection removed channels outside the audited capacity mask.")
    fig, ax = _eda_plt.subplots(figsize=(max(12, min(24, len(capacity_rows) * .45)), 6))
    positions = _eda_np.arange(len(capacity_rows))
    blocked = [row["channels"] - row["capacity_eligible"] for row in capacity_rows]
    retained = [row["capacity_eligible"] - row["removed"] for row in capacity_rows]
    removed = [row["removed"] for row in capacity_rows]
    ax.bar(positions, blocked, color="grey", label="Protected by capacity cap")
    ax.bar(positions, retained, bottom=blocked, color="blue", label="Capacity eligible, kept")
    ax.bar(positions, removed, bottom=_eda_np.array(blocked) + retained, color="red", label="Removed")
    ax.set_xticks(positions, [row["group_id"] for row in capacity_rows], rotation=90, fontsize=7)
    ax.set_ylabel("Independent root channels")
    ax.set_title("Group capacity and actual LAMP selection")
    ax.legend()
    fig.tight_layout()
    files.append(_eda_save_figure(fig, folder / "lamp_group_capacity"))
    return {"status": "MEASURED", "finite_scores": len(scores),
            "capacity_eligible": sum(bool(row.get("capacity_eligible")) for row in scores),
            "removed": sum(bool(row.get("removed")) for row in scores), "files": files}


def _eda_efficiency_plots(stages, result, folder):
    files = []
    for metric, label in (("physical_conv_outputs", "Physical Conv-output channels"),
                          ("parameters", "Parameters (Conv + BN)"),
                          ("estimated_flops", "Estimated layer FLOPs (Conv + BN)")):
        fig, ax = _eda_plt.subplots(figsize=(14, 5))
        positions = _eda_np.arange(len(stages))
        ax.bar(positions - .2, [row[metric + "_before"] for row in stages], width=.4, color="blue", label="Before")
        ax.bar(positions + .2, [row[metric + "_after"] for row in stages], width=.4, color="red", label="After")
        ax.set_xticks(positions, [str(row["stage"]) for row in stages], rotation=90)
        ax.set_ylabel(label)
        ax.set_title("Measured stage inventory — " + label)
        ax.legend()
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / ("stage_" + metric)))
    baseline, pruned = result.get("baseline", {}), result.get("pruned", {})
    summary = {"independent_eligible_channels_before": result.get("eligible_channels_before"),
               "independent_eligible_channels_after": result.get("eligible_channels_after"),
               "actual_independent_channel_ratio": result.get("actual_eligible_channel_ratio"),
               "physical_conv_outputs_before": sum(row["physical_conv_outputs_before"] for row in stages),
               "physical_conv_outputs_after": sum(row["physical_conv_outputs_after"] for row in stages),
               "baseline": {key: baseline.get(key) for key in ("parameters", "gflops", "checkpoint_bytes", "latency")},
               "pruned": {key: pruned.get(key) for key in ("parameters", "gflops", "checkpoint_bytes", "latency")},
               "comparison": result.get("comparison", {}),
               "latency_scope": "raw model forward, not end-to-end",
               "layer_flops_scope": "Conv/BN estimate; model GFLOPs also counts functional attention"}
    _eda_write_json(folder / "efficiency_summary.json", summary)
    return {"status": "MEASURED", "files": files, **summary}


def _eda_evaluation_plots(result, folder):
    statuses, files = {}, []
    for identity in ("baseline", "pruned"):
        validation = (result.get(identity) or {}).get("validation")
        if not validation:
            statuses[identity] = {"status": "NOT_RUN", "reason": "Dataset validation was not run."}
            continue
        target = folder / identity
        target.mkdir(parents=True, exist_ok=True)
        per_class = validation.get("per_class") or []
        if per_class:
            _eda_write_csv(target / "per_class_metrics.csv", per_class)
            labels = [row.get("name", str(row.get("class_id"))) for row in per_class]
            fig, axes = _eda_plt.subplots(1, 3, figsize=(21, 5))
            for key, label in (("ap50", "AP50"), ("ap50_95", "AP50–95")):
                indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get(key))]
                if indices:
                    axes[0].plot(indices, [per_class[index][key] for index in indices], "o-", label=label)
            for key, label in (("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")):
                indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get(key))]
                if indices:
                    axes[1].plot(indices, [per_class[index][key] for index in indices], "o-", label=label)
            support_indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get("support"))]
            if support_indices:
                axes[2].bar(support_indices, [per_class[index]["support"] for index in support_indices], color="grey")
            else:
                axes[2].text(.5, .5, "Class support unavailable", ha="center", va="center", transform=axes[2].transAxes)
            for ax in axes:
                ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
                if ax.lines:
                    ax.legend()
            for ax in axes[:2]:
                ax.set_ylim(0, 1)
                ax.set_ylabel("Measured metric")
            axes[2].set_ylabel("Ground-truth instance support")
            fig.suptitle(identity + " — measured class metrics; missing values are omitted")
            fig.tight_layout()
            files.append(_eda_save_figure(fig, target / "class_metrics"))
        curves = validation.get("curves")
        if curves:
            _eda_write_json(target / "curves.json", curves)
        curve_status = _eda_curve_plots(curves, target) if curves else {"status": "UNAVAILABLE"}
        confusion = validation.get("confusion_matrix") or {}
        confusion_status = {"status": "UNAVAILABLE", "reason": "No measured confusion matrix was collected."}
        if (confusion.get("measured") is True or confusion.get("status") == "MEASURED") and confusion.get("matrix") is not None:
            matrix = _eda_np.asarray(confusion["matrix"], dtype=float)
            labels = confusion.get("labels", [])
            if matrix.ndim != 2 or matrix.shape != (len(labels), len(labels)) or not _eda_np.isfinite(matrix).all():
                raise ValueError("Confusion matrix dimensions/labels/values are invalid.")
            _eda_write_json(target / "confusion_matrix.json", confusion)
            _eda_write_csv(target / "confusion_matrix.csv", [
                {"predicted_class": labels[index], **{label: float(value) for label, value in zip(labels, row)}}
                for index, row in enumerate(matrix)])
            fig, ax = _eda_plt.subplots(figsize=(9, 8))
            plotted = ax.imshow(matrix, cmap="Blues")
            ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
            ax.set_yticks(range(len(labels)), labels)
            ax.set_xlabel(confusion.get("x_label", "True class"))
            ax.set_ylabel(confusion.get("y_label", "Predicted class"))
            conf = confusion.get("confidence_threshold", confusion.get("conf"))
            matching_iou = confusion.get("matching_iou_threshold", confusion.get("iou"))
            ax.set_title(f"{identity}: measured confusion matrix\nconf={conf}, matching IoU={matching_iou}")
            fig.colorbar(plotted, ax=ax)
            fig.tight_layout()
            files.append(_eda_save_figure(fig, target / "confusion_matrix"))
            confusion_status = {"status": "MEASURED", "confidence_threshold": conf, "matching_iou_threshold": matching_iou,
                                "validation_iou": confusion.get("validation_iou")}
        statuses[identity] = {"status": "MEASURED", "per_class": "MEASURED" if per_class else "UNAVAILABLE",
                              "curves": curve_status, "confusion_matrix": confusion_status}
    return {"status": "MEASURED" if any(row["status"] == "MEASURED" for row in statuses.values()) else "NOT_RUN",
            "measurements": statuses, "files": files,
            "interpretation": "Descriptive association; layer changes do not establish a cause of mAP changes."}


def _eda_curve_plots(curves, folder):
    entries = curves if isinstance(curves, list) else curves.get("curves", curves.get("items", []))
    count, files = 0, []
    for entry in entries:
        if not isinstance(entry, dict) or entry.get("status") in {"UNAVAILABLE", "NOT_RUN"}:
            continue
        x, y = entry.get("x"), entry.get("y")
        if x is None or y is None:
            continue
        xs, ys = _eda_np.asarray(x, dtype=float), _eda_np.asarray(y, dtype=float)
        if xs.ndim != 1 or not xs.size or not _eda_np.isfinite(xs).all() or not _eda_np.isfinite(ys).all():
            continue
        if ys.ndim == 1:
            ys = ys[None, :]
        if ys.ndim != 2 or ys.shape[-1] != len(xs):
            continue
        fig, ax = _eda_plt.subplots(figsize=(10, 6))
        labels = entry.get("labels") or entry.get("class_names") or []
        for index, values in enumerate(ys):
            ax.plot(xs, values, label=str(labels[index]) if index < len(labels) else f"series {index}", alpha=.75)
        ax.set_xlabel(entry.get("x_label", entry.get("xlabel", "x")))
        ax.set_ylabel(entry.get("y_label", entry.get("ylabel", "y")))
        ax.set_title(entry.get("name", "Measured validation curve"))
        if len(ys) <= 20:
            ax.legend()
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / f"evaluation_curve_{count:02d}"))
        count += 1
    return {"status": "MEASURED" if count else "UNAVAILABLE", "curve_count": count, "files": files}


def _build_run_eda(result, run_dir, inventory_before, inventory_after):
    """Export ordered EDA from actual run records; failures propagate to run reports."""
    folder = _EdaPath(run_dir) / "eda"
    folder.mkdir(parents=True, exist_ok=True)
    rows = layer_eda_rows(inventory_before, inventory_after)
    conv_rows = [row for row in rows if row["module_type"] == "Conv2d"]
    stages = stage_eda_rows(rows)
    _eda_write_csv(folder / "layer_eda.csv", rows)
    _eda_write_csv(folder / "stage_eda.csv", stages)
    audit = result.get("audit") or {}
    title = f"{result.get('model_name', 'YOLO26' + result.get('model_variant', 'n'))} {result.get('policy', '')} pr{result.get('ratio', 0):g}"
    report = {"status": "MEASURED", "model_variant": result.get("model_variant", "n"),
              "model_name": result.get("model_name", "YOLO26n"), "policy": result.get("policy"), "ratio": result.get("ratio"),
              "section_order": ["identity_coverage", "channels", "masks_lamp", "efficiency", "validation_tradeoffs"],
              "identity_coverage": {"provenance": result.get("provenance"), "conv_layers": len(conv_rows),
                                    "bn_layers": sum(row["module_type"] == "BatchNorm2d" for row in rows),
                                    "eligible_layers": sum(bool(row["eligible"]) for row in rows),
                                    "protected_layers": sum(bool(row["protection_reason"]) for row in rows),
                                    "group_count": result.get("pruned_group_count"),
                                    "protection_reasons": sorted({str(row["protection_reason"]) for row in rows if row["protection_reason"]})},
              "channels": _eda_channel_plots(conv_rows, folder, title),
              "channel_mapping": _eda_mapping_plots(audit, conv_rows, folder),
              "lamp_scores": _eda_lamp_plots(audit, folder),
              "efficiency": _eda_efficiency_plots(stages, result, folder),
              "validation_tradeoffs": _eda_evaluation_plots(result, folder),
              "notes": ["Channel figures include all Conv outputs in original module order.",
                        "Independent root-channel counts differ from physical coupled Conv-output changes.",
                        "No activation feature maps, simulated accuracy, or reconstructed historical channel indices."]}
    _eda_write_json(folder / "eda_status.json", report)
    return report


def build_run_eda(result, run_dir, inventory_before, inventory_after):
    """Persist an explicit failed EDA report before propagating analysis errors."""
    try:
        return _build_run_eda(result, run_dir, inventory_before, inventory_after)
    except Exception as error:
        _eda_write_json(_EdaPath(run_dir) / "eda/eda_status.json",
                        {"status": "FAIL", "model_variant": result.get("model_variant", "n"),
                         "policy": result.get("policy"), "ratio": result.get("ratio"),
                         "error": {"type": type(error).__name__, "message": str(error)}})
        raise


def plot_model_comparison(suites, output_dir):
    """Compare N/S only where each point has measured accuracy and efficiency."""
    folder = _EdaPath(output_dir)
    folder.mkdir(parents=True, exist_ok=True)
    suites = list(suites.values()) if isinstance(suites, dict) else list(suites)
    points = []
    for suite in suites:
        variant = suite.get("model_variant", "n")
        name = suite.get("model_name", "YOLO26" + variant)
        baseline = suite.get("baseline") or {}
        for ratio, policy, metrics in [(0., "baseline", baseline)] + [
                (run["ratio"], run.get("policy", "extended"), run.get("pruned") or {})
                for run in suite.get("results", [])]:
            validation = metrics.get("validation") or {}
            latency = metrics.get("latency") or {}
            points.append({"model_variant": variant, "model_name": name, "ratio": ratio, "policy": policy,
                           "map50_95": validation.get("map50_95"), "parameters": metrics.get("parameters"),
                           "gflops": metrics.get("gflops"),
                           "latency_ms": latency.get("median_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                           "latency_p05_ms": latency.get("p05_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                           "latency_p95_ms": latency.get("p95_ms") if latency.get("status") in {"PASS", "MEASURED"} else None})
    _eda_write_csv(folder / "model_comparison.csv", points)
    statuses = []
    for metric, label in (("parameters", "Parameters"), ("gflops", "GFLOPs"), ("latency_ms", "Median raw model forward latency (ms)")):
        valid = [row for row in points if _eda_finite(row.get(metric)) and _eda_finite(row.get("map50_95"))]
        if not valid:
            statuses.append({"plot": "map_vs_" + metric, "status": "NOT_RUN", "reason": "Paired measured accuracy/efficiency is unavailable."})
            continue
        fig, ax = _eda_plt.subplots(figsize=(10, 7))
        for variant in dict.fromkeys(row["model_variant"] for row in valid):
            rows = [row for row in valid if row["model_variant"] == variant]
            ax.scatter([row[metric] for row in rows], [row["map50_95"] for row in rows], label="YOLO26" + variant)
            for row in rows:
                if (metric == "latency_ms" and _eda_finite(row.get("latency_p05_ms"))
                        and _eda_finite(row.get("latency_p95_ms"))):
                    ax.hlines(row["map50_95"], row["latency_p05_ms"], row["latency_p95_ms"], alpha=.5)
                ax.annotate(f"{row['model_name']} {row['policy']} pr{row['ratio']:g}", (row[metric], row["map50_95"]),
                            xytext=(5, 5), textcoords="offset points", fontsize=8)
        ax.set_xlabel(label)
        ax.set_ylabel("Measured mAP50–95")
        ax.set_title("N/S compression–accuracy comparison")
        ax.legend()
        fig.tight_layout()
        files = _eda_save_figure(fig, folder / ("map_vs_" + metric))
        statuses.append({"plot": "map_vs_" + metric, "status": "MEASURED", "points": len(valid), "files": files})
    report = {"plots": statuses, "point_count": len(points), "scope": "Each model uses its own measured baseline; no missing accuracy is replaced with zero."}
    _eda_write_json(folder / "comparison_status.json", report)
    return report

## 07 — Implementasi LAMP dan Pencarian Target GFLOPs

Ranking global LAMP memakai skor berbasis bobot dan DepGraph menentukan channel yang dipangkas
bersama. Proteksi head, attention, split CSP, downsampling, dan antarmuka fitur mengikuti audit 06.
Feasibility diukur pada tiga tier batas penghapusan 25%/10%, 35%/20%, dan 50%/50% untuk group
hidden Bottleneck dan group eligible lain. Pencarian mencari anggaran channel terkecil yang
mencapai target pada tier pertama yang layak. Jika target melebihi kapasitas, evaluasi checkpoint
maksimum dan catat `TARGET_NOT_REACHED` beserta speed-up aktual.

Profil pencarian memakai bobot baseline masing-masing seed. Tiap perlakuan memuat ulang baseline
dan mengulang plan terpilih untuk memeriksa ledger, struktur, serta hash tensor dan buffer.

In [ ]:
"""GFLOPs-target planner and tensor oracle embedded after the V2 policy."""
# ruff: noqa: F821 -- The standalone notebook loads V2 declarations before this cell.

import math
from contextlib import nullcontext
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch_pruning as tp

GFLOPS_CAP_TIERS = ((25, 10), (35, 20), (50, 50))
GFLOPS_INTERFACE_STAGES = (0, 1, 3, 4, 5, 6, 7, 13, 16, 17, 19, 20, 22)
GFLOPS_P5_ROOT = "model.7.conv"


def _gflops_rank_space(audit, caps, protect_interfaces, exclude_p5):
    """Keep original LAMP scores and bound capacity before ranking candidates."""
    if len(caps) != 2 or any(type(cap) is not int or not 0 <= cap <= 50 for cap in caps):
        raise ValueError("Cap harus berisi dua persentase integer antara 0 dan 50.")
    structure = audit["structure"]
    interfaces = set()
    if protect_interfaces:
        for stage in GFLOPS_INTERFACE_STAGES:
            block = structure["modules"][f"model.{stage}"]
            interface = getattr(block, "cv2", block).conv
            if not isinstance(interface, torch.nn.Conv2d):
                raise TypeError(f"Output antarmuka YOLO26 tidak didukung: model.{stage}.")
            interfaces.add(interface)
    if exclude_p5:
        interfaces.add(structure["modules"][GFLOPS_P5_ROOT])

    ranked, capacities, eligible_channels = [], {}, 0
    for group in audit["eligible"].values():
        if interfaces.intersection(group["output_dimensions"]):
            continue
        width = group["initial_channels"]
        percent = caps[0] if group["root"] in structure["conservative"] else caps[1]
        capacity = min(width - 1, *(size * percent // 100 for size in group["output_dimensions"].values()))
        root = group["name"]
        scores = group["initial_lamp_scores"]
        if len(scores) != width or not all(math.isfinite(score) for score in scores):
            raise ValueError(f"Skor LAMP tidak valid: {root}.")
        eligible_channels += width
        capacities[root] = capacity
        ranked.extend(sorted((float(score), root, index) for index, score in enumerate(scores))[:capacity])
    return sorted(ranked), capacities, eligible_channels


def gflops_ranked_candidates(audit, caps, protect_interfaces=True, exclude_p5=False):
    return _gflops_rank_space(audit, caps, protect_interfaces, exclude_p5)[0]


def _gflops_prefix_plan(ranked, count):
    if type(count) is not int or not 0 <= count <= len(ranked):
        raise ValueError(f"Target {count} channel melampaui kapasitas {len(ranked)}.")
    plan = {}
    for _, root, index in ranked[:count]:
        plan.setdefault(root, []).append(index)
    return {root: sorted(indices) for root, indices in sorted(plan.items())}


def gflops_plan_key(plan):
    """Canonical identity for caching and deduplicating equal physical plans."""
    return tuple((root, tuple(sorted(indices))) for root, indices in sorted(plan.items()) if indices)


def p5_control_plans(audit, extended_10_history):
    """Use the historical 10% mask itself as the source of the P5-only dose."""
    groups = [entry for entry in audit["eligible"].values() if GFLOPS_P5_ROOT in entry["signature"]]
    if len(groups) != 1 or groups[0]["name"] != GFLOPS_P5_ROOT:
        raise ValueError("Root canonical P5 harus unik dan cocok dengan model.7.conv.")
    history = {entry["root"]: entry["removed_original_indices"] for entry in extended_10_history}
    if len(history) != len(extended_10_history) or not history.get(GFLOPS_P5_ROOT):
        raise ValueError("Kontrol extended 10% harus mencatat satu penghapusan group P5.")
    count = sum(len(indices) for indices in history.values())
    candidates = gflops_ranked_candidates(audit, (50, 50), protect_interfaces=False, exclude_p5=True)
    return {"p5-only": {GFLOPS_P5_ROOT: sorted(history[GFLOPS_P5_ROOT])},
            "p5-protected": _gflops_prefix_plan(candidates, count)}


def apply_exact_gflops_plan(pruner, audit, plan, caps=None):
    """Run the existing dependency verifier and ledger with exact finite selections."""
    groups = {entry["name"]: entry for entry in audit["eligible"].values()}
    expected_caps = {name: _v2_capacity(entry) for name, entry in groups.items()}
    if caps is not None:
        if any(name not in groups or type(cap) is not int or not 0 <= cap <= expected_caps[name]
               for name, cap in caps.items()):
            raise ValueError("Cap plan tidak sesuai dengan dependency group sumber.")
        expected_caps = {name: caps.get(name, 0) for name in groups}
        eligible_channels = sum(groups[name]["initial_channels"] for name in caps)
        audit["report"]["target_policy_eligible_channels"] = eligible_channels

    graph = pruner.DG
    selections = gflops_plan_key(plan)
    if not selections:
        raise ValueError("Plan pruning harus memiliki setidaknya satu channel.")
    for name, indices in selections:
        entry = groups.get(name)
        if (entry is None or len(indices) > expected_caps[name] or len(indices) != len(set(indices))
                or any(type(index) is not int or not 0 <= index < entry["initial_channels"] for index in indices)):
            raise ValueError(f"Plan channel invalid atau melebihi cap: {name}.")
        if entry["root"].out_channels != entry["initial_channels"]:
            raise ValueError(f"Plan harus dimulai dari indeks baseline: {name}.")

    audit["report"]["ranking"] = "Finite LAMP scores sorted by (score, root, original index), exact integer budget"
    audit["report"]["planned_original_indices"] = {name: list(indices) for name, indices in selections}
    audit["report"]["planned_group_capacities"] = expected_caps
    if caps is not None:
        permitted = {}
        for name, entry in groups.items():
            scores = entry["initial_lamp_scores"]
            permitted[name] = set(sorted(range(len(scores)), key=lambda index: (scores[index], index))[:expected_caps[name]])
        for row in audit["report"]["lamp_scores"]:
            row["capacity_eligible"] = row["channel_idx"] in permitted[row["root"]]
        if any(not set(indices).issubset(permitted[name]) for name, indices in selections):
            raise ValueError("Plan memilih channel di luar peringkat LAMP dalam batas cap.")

    def step(interactive=True):
        if not interactive:
            raise ValueError("Pruning memerlukan pemeriksaan dependency interaktif.")
        for name, indices in selections:
            entry = groups[name]
            group = graph.get_pruning_group(entry["root"], tp.prune_conv_out_channels, list(indices))
            # All coupled Conv outputs must lose exactly the same number of channels.
            for dependency, mapped in group:
                dimension = _v2_dependency_dimension(dependency, graph)
                if dimension is not None and dimension[1] == "out" and len(mapped) != len(indices):
                    raise ValueError(f"Jumlah survivor output residual tidak konsisten: {name}.")
            yield group

    history = apply_policy_pruning(SimpleNamespace(DG=graph, step=step), audit)
    applied = {entry["root"]: entry["removed_original_indices"] for entry in history}
    if gflops_plan_key(applied) != selections:
        raise ValueError("Mask aktual berbeda dari plan indeks asli.")
    return history


def search_gflops_targets(audit, baseline_gflops, measure_plan,
                         targets=(1.02, 1.05, 1.10, 1.20, 1.70), *, profile_cache=None, progress_callback=None):
    """Original finite LAMP prefix search, with resumable profiling and isolated failures."""
    if not math.isfinite(baseline_gflops) or baseline_gflops <= 0:
        raise ValueError("GFLOPs baseline harus finite dan positif.")
    targets = tuple(targets)
    if not targets or len(targets) != len(set(targets)) or any(not math.isfinite(t) or t <= 1 for t in targets):
        raise ValueError("Target GFLOPs harus unik, finite, dan lebih dari satu.")
    cached = {} if profile_cache is None else profile_cache
    cached[repr(())] = float(baseline_gflops)
    measurements, tiers, spaces, errors = {}, [], {}, []

    def profile(ranked, count, tier):
        plan = _gflops_prefix_plan(ranked, count)
        key = repr(gflops_plan_key(plan))
        if key not in cached:
            measured = float(measure_plan(plan))
            if not math.isfinite(measured) or measured <= 0 or measured > baseline_gflops + 1e-12:
                raise ValueError(f"GFLOPs tidak valid untuk tier {tier}, {count} channel.")
            cached[key] = measured
        value = cached[key]
        if not math.isfinite(value) or not 0 < value <= baseline_gflops + 1e-12:
            raise ValueError("Cache profil GFLOPs tidak valid.")
        measurements[tier, count] = {"tier": tier, "channels": count, "gflops": value,
                                     "speed_up": baseline_gflops / value}
        if progress_callback:
            progress_callback(cached, list(measurements.values()))
        return value, plan

    for tier, cap in enumerate(GFLOPS_CAP_TIERS, 1):
        ranked, capacities, eligible = _gflops_rank_space(audit, cap, True, False)
        entry = {"tier": tier, "hidden_cap_pct": cap[0], "other_cap_pct": cap[1],
                 "eligible_channels": eligible, "capacity_channels": len(ranked), "status": "NOT_RUN"}
        try:
            measured, plan = profile(ranked, len(ranked), tier)
            entry.update(status="PASS", max_gflops_speed_up=baseline_gflops / measured, gflops_at_capacity=measured)
            spaces[tier] = ranked, capacities, measured, plan
        except Exception as error:
            entry.update(status="FAIL", error=str(error))
            errors.append({"tier": tier, "stage": "capacity", "error": str(error)})
        tiers.append(entry)

    rows = []
    for target in targets:
        try:
            feasible = next((entry for entry in tiers if entry["status"] == "PASS"
                             and entry["max_gflops_speed_up"] >= target), None)
            if feasible is None:
                valid = [entry for entry in tiers if entry["status"] == "PASS"]
                if len(valid) != len(tiers):
                    raise ValueError("Kapasitas belum dapat dibuktikan karena ada profil tier yang gagal.")
                best = max(valid, key=lambda entry: (entry["max_gflops_speed_up"], -entry["tier"]))
                ranked, capacities, measured, plan = spaces[best["tier"]]
                rows.append({"target": target, "status": "TARGET_NOT_REACHED", "tier": best["tier"],
                             "channels": len(ranked), "gflops": measured, "speed_up": baseline_gflops / measured,
                             "plan": plan, "caps": capacities})
                continue
            tier = feasible["tier"]
            ranked, capacities, _, _ = spaces[tier]
            checkpoints = sorted({min(len(ranked), (feasible["eligible_channels"] * pct + 99) // 100)
                                  for pct in range(1, 51)} | {len(ranked)})
            previous, previous_flops = 0, baseline_gflops
            for checkpoint in checkpoints:
                measured, plan = profile(ranked, checkpoint, tier)
                if measured > previous_flops + 1e-12:
                    raise ValueError("GFLOPs bertambah pada prefix dalam tier.")
                if baseline_gflops / measured >= target:
                    break
                previous, previous_flops = checkpoint, measured
            for count in range(previous + 1, checkpoint + 1):
                measured, plan = profile(ranked, count, tier)
                if measured > previous_flops + 1e-12:
                    raise ValueError("GFLOPs bertambah pada refinement integer.")
                if baseline_gflops / measured >= target:
                    break
                previous_flops = measured
            rows.append({"target": target, "status": "REACHED", "tier": tier, "channels": count,
                         "gflops": measured, "speed_up": baseline_gflops / measured,
                         "plan": plan, "caps": capacities})
        except Exception as error:
            rows.append({"target": target, "status": "FAIL", "error": str(error), "plan": None})
    return {"targets": rows, "tiers": tiers, "errors": errors,
            "measurements": [measurements[key] for key in sorted(measurements)]}


def _gflops_checked_masks(audit):
    mappings = {}
    for row in audit["report"]["channel_mapping"]:
        key = row["layer"], row["dimension"]
        kept = row["kept_original_indices"]
        removed = row["removed_original_indices"]
        if (key in mappings or len(kept) != row["channels_after"] or not kept
                or kept != sorted(kept) or removed != sorted(removed)
                or sorted([*kept, *removed]) != list(range(row["channels_before"]))):
            raise ValueError(f"Ledger indeks asli tidak valid: {key}.")
        mappings[key] = row
    return mappings


def _gflops_index(tensor, axis, indices):
    return tensor.index_select(axis, torch.tensor(indices, device=tensor.device, dtype=torch.long))


def verify_gflops_oracle(baseline, pruned, image, audit):
    """Verify exact parameter slicing and functional parity against a dense oracle."""
    source, target = dict(baseline.named_modules()), dict(pruned.named_modules())
    if source.keys() != target.keys() or baseline.training or pruned.training:
        raise ValueError("Oracle memerlukan module tree yang sama dalam mode eval.")
    mappings = _gflops_checked_masks(audit)
    candidate_device = next(pruned.parameters()).device
    masked = deepcopy(baseline).to(candidate_device).eval()
    masked_modules = dict(masked.named_modules())
    verified, changed_modules, masked_inputs = 0, 0, 0
    with torch.no_grad():
        for name, module in source.items():
            current = target[name]
            if type(module) is not type(current):
                raise ValueError(f"Jenis modul berubah: {name}.")
            if isinstance(module, torch.nn.Conv2d):
                inputs, outputs = (mappings.pop((name, dimension)) for dimension in ("in", "out"))
                attributes = ("groups", "kernel_size", "stride", "padding", "dilation", "padding_mode")
                if any(getattr(module, key) != getattr(current, key) for key in attributes):
                    raise ValueError(f"Konfigurasi Conv berubah: {name}.")
                if ((inputs["channels_before"], outputs["channels_before"]) != (module.in_channels, module.out_channels)
                        or (inputs["channels_after"], outputs["channels_after"]) != (current.in_channels, current.out_channels)):
                    raise ValueError(f"Lebar Conv berbeda dari ledger: {name}.")
                dropped = inputs["removed_original_indices"]
                sliced = bool(dropped or outputs["removed_original_indices"])
                if sliced and (module.groups != 1 or current.groups != 1):
                    raise ValueError(f"Oracle tidak mendukung perubahan grouped/depthwise Conv: {name}.")
                weight, bias = module.weight, module.bias
                if sliced:
                    weight = _gflops_index(weight, 0, outputs["kept_original_indices"])
                    weight = _gflops_index(weight, 1, inputs["kept_original_indices"])
                    if bias is not None:
                        bias = _gflops_index(bias, 0, outputs["kept_original_indices"])
                    changed_modules += 1
                if dropped:
                    consumer = masked_modules[name]
                    consumer.weight.index_fill_(1, torch.tensor(dropped, device=candidate_device, dtype=torch.long), 0)
                    masked_inputs += 1
                tensors = [("weight", weight), ("bias", bias)]
            elif isinstance(module, torch.nn.BatchNorm2d):
                channels = mappings.pop((name, "channel"))
                if (channels["channels_before"] != module.num_features
                        or channels["channels_after"] != current.num_features or module.training or current.training):
                    raise ValueError(f"Lebar atau mode BN berbeda dari ledger: {name}.")
                if any(getattr(module, key) != getattr(current, key) for key in ("eps", "affine", "track_running_stats")):
                    raise ValueError(f"Konfigurasi BN berubah: {name}.")
                sliced = bool(channels["removed_original_indices"])
                changed_modules += int(sliced)
                tensors = []
                for key in ("weight", "bias", "running_mean", "running_var", "num_batches_tracked"):
                    value = getattr(module, key)
                    if sliced and value is not None and key != "num_batches_tracked":
                        value = _gflops_index(value, 0, channels["kept_original_indices"])
                    tensors.append((key, value))
            else:
                continue
            for key, value in tensors:
                actual = getattr(current, key)
                if ((value is None) != (actual is None)
                        or (value is not None and (value.dtype != actual.dtype or not torch.equal(value.to(actual.device), actual)))):
                    raise ValueError(f"Irisan tensor berbeda dari baseline: {name}.{key}.")
                verified += int(value is not None)
        if mappings:
            raise ValueError("Ledger memuat dimensi tanpa modul Conv/BN yang sesuai.")

        example = image.to(candidate_device)
        cudnn = torch.backends.cudnn
        # Dense and compact kernels can choose different accumulation paths. Use
        # FP32 for the comparison on CUDA and restore the caller's backend flags.
        precision = (cudnn.flags(enabled=cudnn.enabled, benchmark=cudnn.benchmark,
                                 deterministic=cudnn.deterministic, allow_tf32=False)
                     if candidate_device.type == "cuda" else nullcontext())
        with precision:
            oracle_outputs = _v2_trace_outputs(masked(example))
            compact_outputs = _v2_trace_outputs(pruned(example))
        metrics = {}
        names = ("one2many.boxes", "one2many.scores", "one2one.boxes", "one2one.scores")
        for name, reference, actual in zip(names, oracle_outputs, compact_outputs):
            if (reference.shape != actual.shape or not torch.isfinite(reference).all()
                    or not torch.isfinite(actual).all()
                    or not torch.allclose(reference, actual, rtol=1e-4, atol=1e-5)):
                raise ValueError(f"Forward raw berbeda dari oracle masked: {name}.")
            metrics[name] = {"shape": list(actual.shape), "max_abs_diff": float((reference - actual).abs().max())}
    return {"status": "PASS", "conv_bn_tensors_verified": verified, "changed_conv_bn_modules": changed_modules,
            "masked_conv_consumers": masked_inputs, "raw_outputs": metrics, "rtol": 1e-4, "atol": 1e-5,
            "scope": "Exact baseline slices and dense consumer-column mask parity on the supplied FP32 input."}

## 08 — Eksperimen Independen dan Kontrol P5

Antrean serial memakai baseline sesuai seed. Ketiga target seed 42 memulai pruning dari
`best.pt` yang sama dan menghasilkan paket raw terpisah.

| Urutan | Target GFLOPs | Seed | Baseline | Peran |
| ---: | --- | ---: | --- | --- |
| 1 | 1,10× | 42 | `v62` | Pembanding |
| 2 | 1,20× | 42 | `v62` | Utama |
| 3 | 1,70× | 42 | `v62` | Pembanding kapasitas |
| 4 | 1,20× | 0 | `v71` | Utama |
| 5 | 1,20× | 2024 | `v41` | Utama |
| 6 | 1,20× | 7 | `v49` | Utama |
| 7 | 1,20× | 3407 | `v43` | Utama |

Kontrol extended, P5-only, dan P5-protected berstatus `NOT_RUN` pada suite ini. Bagian 13
menyediakan konteks historis dan auditnya.

## 09 — Simpan Checkpoint dan Fresh Reload

Simpan checkpoint setiap perlakuan sebagai objek `DetectionModel` FP32 dengan struktur hasil pruning.
Oracle membandingkan output pruned dengan baseline yang diberi mask dari ledger. Fresh reload
di proses Python baru memeriksa struktur, state, dan kesetaraan output kedua cabang Detect dengan
model pruned sebelum disimpan. Manifest raw menyimpan hash checkpoint serta referensi baseline
dan bukti audit.

In [ ]:
RELOAD_SCRIPT = r'''
import json, sys
from pathlib import Path
import numpy as np
import torch
from ultralytics import YOLO

checkpoint, reference, result, device = sys.argv[1:]
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
payload = torch.load(reference, map_location="cpu", weights_only=True)
wrapper = YOLO(checkpoint, task="detect")
core = wrapper.model.to(device).float().eval()
with torch.no_grad():
    output = core(payload["image"].to(device))
snapshot = {"decoded": output[0].detach().cpu()}
for branch in ("one2many", "one2one"):
    for key in ("boxes", "scores"):
        snapshot[branch + "/" + key] = output[1][branch][key].detach().cpu()
for key, expected in payload["expected"].items():
    if not torch.isfinite(snapshot[key]).all():
        raise RuntimeError("Non-finite output after reload: " + key)
    torch.testing.assert_close(snapshot[key], expected, rtol=1e-4, atol=1e-5)
if sum(p.numel() for p in core.parameters()) != payload["parameters"]:
    raise RuntimeError("Parameter count changed after reload")
if not all(p.dtype == torch.float32 for p in core.parameters()):
    raise RuntimeError("Reload is not FP32")
torch.save(snapshot, result)
# Fresh wrapper protects the raw unfused model from inference backend fusion.
predictor = YOLO(checkpoint, task="detect")
size = payload["image"].shape[-1]
predictions = predictor.predict(source=np.zeros((size, size, 3), dtype=np.uint8),
    imgsz=size, device=device, conf=0.001, iou=0.7, max_det=300,
    quantize=None, nms=None, verbose=False, save=False)
if len(predictions) != 1 or predictions[0].boxes is None:
    raise RuntimeError("No valid detection Results after reload")
boxes = predictions[0].boxes.data.detach().cpu()
if boxes.ndim != 2 or boxes.shape[1] != 6 or not torch.isfinite(boxes).all():
    raise RuntimeError("Invalid detection tensor")
nc = len(core.names)
if boxes.numel() and (((boxes[:, 5] < 0) | (boxes[:, 5] >= nc) | (boxes[:, 5] != boxes[:, 5].floor())).any()
                     or ((boxes[:, 4] < 0) | (boxes[:, 4] > 1)).any()):
    raise RuntimeError("Invalid detection class/confidence")
Path(result).with_suffix(".json").write_text(json.dumps({"prediction_count": len(boxes),
    "parameters": payload["parameters"], "direct_yolo_load": True,
    "raw_parity": True, "prediction_shape": list(boxes.shape)}), encoding="utf-8")
'''


def save_fp32_checkpoint(core, path, provenance):
    """Simpan arsitektur hasil pruning memakai format model-object Ultralytics."""
    model = deepcopy(core).cpu().float().eval().requires_grad_(False)
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()
    model.args = {"task": "detect", "imgsz": IMGSZ, "data": str(DATA_YAML), "nms": None, "quantize": None}
    torch.save({"model": model, "ema": None, "optimizer": None, "epoch": -1,
                "train_args": dict(model.args), "version": ultralytics.__version__,
                "date": datetime.now(timezone.utc).isoformat(), "lamp_format_version": 1,
                "lamp_provenance": provenance, "recovery_enabled": False}, path)
    return {"path": str(Path(path).resolve()), "sha256": sha256_file(path), "file_bytes": Path(path).stat().st_size}


def verify_fresh_reload(checkpoint, core, image, output_dir, *, timeout=300):
    """Restore di interpreter baru; exit code/error tidak boleh disembunyikan."""
    output_dir = Path(output_dir).resolve()
    script = output_dir / "verify_reload.py"
    reference = output_dir / "reload_reference.pt"
    result = output_dir / "reload_output.pt"
    script.write_text(RELOAD_SCRIPT, encoding="utf-8")
    expected = forward_snapshot(core, image)
    torch.save({"image": image.detach().cpu(), "expected": expected,
                "parameters": sum(p.numel() for p in core.parameters())}, reference)
    child_env = dict(os.environ)
    child_env.pop("PYTHONPATH", None)
    child_env.pop("WANDB_API_KEY", None)
    child_env["WANDB_MODE"] = "disabled"
    child_env["YOLO_AUTOINSTALL"] = "false"
    completed = subprocess.run([sys.executable, "-B", str(script), str(Path(checkpoint).resolve()),
                                str(reference), str(result), str(next(core.parameters()).device)],
                               cwd=output_dir, env=child_env, capture_output=True, text=True,
                               encoding="utf-8", errors="replace", timeout=timeout)
    (output_dir / "reload_stdout.txt").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "reload_stderr.txt").write_text(completed.stderr, encoding="utf-8")
    if completed.returncode:
        raise RuntimeError(f"Fresh-process reload gagal (exit {completed.returncode}): {completed.stderr[-6000:]}")
    actual = torch.load(result, map_location="cpu", weights_only=True)
    for key in expected:
        torch.testing.assert_close(actual[key], expected[key], rtol=1e-4, atol=1e-5)
    report = json.loads(result.with_suffix(".json").read_text(encoding="utf-8"))
    report["max_absolute_difference"] = {key: float((actual[key] - expected[key]).abs().max()) for key in expected}
    report["process"] = "fresh Python, no repository PYTHONPATH"
    return report

## 10 — Validasi Baseline dan Pruned

Evaluasi lima baseline dan tujuh checkpoint raw pada split, daftar citra, dan konfigurasi validator
yang sama. `full_val` memakai seluruh validation Poltrada V3 dengan FP32 dan one-to-many/NMS.
Konfigurasi validator memakai confidence 0,001, IoU NMS 0,70, dan maksimum 300 deteksi per citra.
Status audit struktur, reload, validasi, latency CUDA, EDA, dan publikasi tercatat terpisah.
Hasil baseline seed 42 dipakai sebagai pembanding untuk ketiga target seed 42.

In [ ]:
def cuda_latency(core, image, warmup=50, repeats=200):
    """Latency forward model batch satu; tidak mencakup preprocessing/NMS/disk IO."""
    if image.device.type != "cuda":
        return {"status": "NOT_RUN", "reason": "CUDA tidak tersedia; tidak mengklaim benchmark GPU."}
    core = deepcopy(core).eval().requires_grad_(False)
    samples = []
    with torch.inference_mode():
        for _ in range(warmup):
            core(image)
        torch.cuda.synchronize(image.device)
        for _ in range(repeats):
            torch.cuda.synchronize(image.device)
            started = time.perf_counter()
            core(image)
            torch.cuda.synchronize(image.device)
            samples.append((time.perf_counter() - started) * 1000)
    return {"status": "PASS", "median_ms": float(np.median(samples)), "mean_ms": float(np.mean(samples)),
            "p05_ms": float(np.percentile(samples, 5)), "p95_ms": float(np.percentile(samples, 95)),
            "warmup": warmup, "repeats": repeats, "batch": 1, "imgsz": list(image.shape[-2:]),
            "precision": "FP32", "device": str(image.device), "fusion": False, "scope": "raw model forward"}

In [ ]:
"""Validation declarations embedded verbatim in standalone LAMP V2 notebooks.

The source never imports repository modules at notebook runtime. Numerical exports
come from the actual Ultralytics validator; missing measurements remain null.
"""
# ruff: noqa: F821 -- Notebook configuration/helpers provide these runtime globals.


def validation_arguments(data_yaml, output_dir, label):
    """Identical FP32 protocol for baseline/pruned; plots also collect confusion counts."""
    return {"data": str(data_yaml), "split": "val", "imgsz": IMGSZ, "batch": BATCH,
            "workers": WORKERS, "device": str(TARGET_DEVICE), "conf": 0.001,
            "iou": 0.7, "max_det": 300, "quantize": None, "nms": None,
            "augment": False, "plots": True, "verbose": False, "save_json": False,
            "project": str(output_dir), "name": label, "exist_ok": False}


def _validation_number(value):
    """JSON scalar: unsupported/nonfinite auxiliary metrics are unavailable, not zero."""
    if value is None:
        return None
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    return number if math.isfinite(number) else None


def _validation_array(value):
    import numpy as np

    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=float)


def _validation_json_array(value):
    import numpy as np

    array = _validation_array(value)
    return np.where(np.isfinite(array), array, None).tolist()


def _validation_per_class(metric, class_names):
    """Map SDK result rows via ap_class_index, never by names or array position."""
    import numpy as np

    box = metric.box
    support = getattr(metric, "nt_per_class", None)
    if support is not None:
        support = _validation_array(support)
        if (support.shape != (len(class_names),) or not np.isfinite(support).all()
                or (support < 0).any() or not np.equal(support, np.floor(support)).all()):
            raise ValueError("Support kelas SDK tidak valid atau tidak cocok dengan jumlah kelas.")
    rows = [{"class_id": i, "name": name, "ap50": None, "ap50_95": None,
             "precision": None, "recall": None, "f1": None,
             "support": int(support[i]) if support is not None else None}
            for i, name in enumerate(class_names)]
    indices = list(getattr(box, "ap_class_index", []))
    validated_indices = []
    for value in indices:
        scalar = _validation_number(value)
        if scalar is None or scalar != int(scalar) or not 0 <= int(scalar) < len(rows):
            raise ValueError("Indeks kelas SDK tidak valid.")
        validated_indices.append(int(scalar))
    if len(set(validated_indices)) != len(validated_indices):
        raise ValueError("Indeks kelas SDK duplikat.")
    for offset, class_id in enumerate(validated_indices):
        for key, attribute in (("ap50", "ap50"), ("ap50_95", "ap"),
                               ("precision", "p"), ("recall", "r"), ("f1", "f1")):
            values = getattr(box, attribute, None)
            if values is not None and len(values) > offset:
                rows[class_id][key] = _validation_number(values[offset])
    return rows, validated_indices


def _validation_curves(metric, class_names, class_ids):
    """Keep SDK axes and class mapping, replacing nonfinite samples with JSON null."""
    import numpy as np

    try:
        curve_values = metric.curves_results
    except (AttributeError, TypeError, ValueError) as error:
        return {"status": "UNAVAILABLE", "reason": str(error), "curves": []}
    names = getattr(metric, "curves", [])
    records = []
    for index, values in enumerate(curve_values):
        record = {"name": names[index] if index < len(names) else f"SDK curve {index}",
                  "class_ids": list(class_ids), "class_names": [class_names[i] for i in class_ids],
                  "status": "UNAVAILABLE", "x": [], "y": []}
        try:
            x_value, y_value, x_label, y_label = values
            x, y = _validation_array(x_value), _validation_array(y_value)
            record.update({"x_label": str(x_label), "y_label": str(y_label)})
            if y.ndim == 1 and len(class_ids) == 1:
                y = y[None, :]
            if (not class_ids or x.ndim != 1 or not x.size or y.ndim != 2
                    or y.shape != (len(class_ids), x.size)):
                record["reason"] = "Dimensi kurva tidak memiliki pemetaan kelas SDK yang valid."
            else:
                record.update({"x": _validation_json_array(x), "y": _validation_json_array(y),
                               "status": "MEASURED" if np.isfinite(x).all() and np.isfinite(y).all()
                               else "PARTIAL"})
                if record["status"] == "PARTIAL":
                    record["reason"] = "Sampel SDK NaN/Inf disimpan sebagai null."
        except (TypeError, ValueError, OverflowError) as error:
            record["reason"] = str(error)
        records.append(record)
    states = {item["status"] for item in records}
    status = "MEASURED" if states == {"MEASURED"} else (
        "PARTIAL" if states & {"MEASURED", "PARTIAL"} else "UNAVAILABLE")
    return {"status": status, "curves": records}


def _validation_confusion_matrix(validator, class_names, arguments):
    """Export actual counts and the distinct matching/NMS IoU thresholds."""
    import inspect

    import numpy as np

    matrix_object = getattr(validator, "confusion_matrix", None)
    result = {"status": "UNAVAILABLE", "labels": [*class_names, "background"],
              "orientation": "rows=predicted, columns=true", "matrix": None,
              "normalized_by_true": None, "confidence_threshold": None,
              "matching_iou_threshold": None, "validation_iou": arguments["iou"],
              "normalization": "Divide each true-class column by its sum; empty columns stay zero."}
    if matrix_object is None or getattr(matrix_object, "matrix", None) is None:
        result["reason"] = "Validator tidak menyediakan confusion matrix."
        return result
    matrix = _validation_array(matrix_object.matrix)
    expected = len(class_names) + 1
    if (matrix.shape != (expected, expected) or not np.isfinite(matrix).all() or (matrix < 0).any()):
        raise ValueError("Confusion matrix SDK tidak valid atau class order berbeda.")
    threshold = _validation_number(getattr(validator, "confusion_matrix_conf", None))
    matching_iou = None
    try:
        parameter = inspect.signature(matrix_object.process_batch).parameters.get("iou_thres")
        if parameter is not None and parameter.default is not inspect.Parameter.empty:
            matching_iou = _validation_number(parameter.default)
    except (AttributeError, TypeError, ValueError):
        pass
    totals = matrix.sum(axis=0, keepdims=True)
    normalized = np.divide(matrix, totals, out=np.zeros_like(matrix), where=totals != 0)
    result.update({"status": "MEASURED", "matrix": matrix.tolist(),
                   "normalized_by_true": normalized.tolist(),
                   "zero_support_columns": [int(i) for i in np.flatnonzero(totals[0] == 0)],
                   "confidence_threshold": threshold, "matching_iou_threshold": matching_iou,
                   "matching_iou_source": "Signature default of actual ConfusionMatrix.process_batch; validator calls without iou_thres.",
                   "confidence_source": "Actual DetectionValidator.confusion_matrix_conf"})
    return result


def _validation_metric_semantics(metric):
    """Record the SDK best-F1 reporting point without confusing it with CM confidence."""
    import numpy as np
    from ultralytics.utils.metrics import smooth

    result = {"precision_recall_f1": "SDK values at the global confidence maximizing smoothed mean F1 (IoU=0.50).",
              "reporting_confidence": None,
              "reporting_confidence_source": "SDK px[smooth(f1_curve.mean(0), 0.1).argmax()]",
              "map_iou_thresholds": [round(0.5 + 0.05 * i, 2) for i in range(10)],
              "absent_class": "Support 0 denotes no ground truth; class metrics without an SDK row remain null.",
              "confusion_matrix": "A separate operating point: see confidence_threshold and matching_iou_threshold."}
    box = metric.box
    try:
        px, f1 = _validation_array(box.px), _validation_array(box.f1_curve)
        if (px.ndim == 1 and f1.ndim == 2 and f1.shape[0] > 0 and f1.shape[1] == len(px)
                and np.isfinite(px).all() and np.isfinite(f1).all()):
            result["reporting_confidence"] = float(px[int(smooth(f1.mean(0), 0.1).argmax())])
    except (AttributeError, TypeError, ValueError):
        pass
    return result


def evaluate_checkpoint(path, dataset, output_dir, label):
    """Evaluate a fresh wrapper; preserve SDK metrics, curves and populated confusion counts."""
    wrapper = YOLO(str(path), task="detect")
    assert_class_match(wrapper.names, dataset["info"]["names"])
    class_names = list(dataset["class_names"])
    assert_class_match(wrapper.names, class_names)
    arguments = gflops_validation_arguments(dataset, output_dir, label)
    captured = {}
    capture_validator = None
    if hasattr(wrapper, "add_callback"):
        # Model.val() keeps its validator local in Ultralytics 8.4.155.
        # Read the actual instance at callback time rather than assuming an attribute.
        def capture_validator(validator):
            captured["validator"] = validator

        wrapper.add_callback("on_val_end", capture_validator)
    try:
        with gflops_require_complete_nms(), gflops_disable_label_cache():
            metric = wrapper.val(**arguments)
    finally:
        # Avoid a callback -> capture -> validator cycle retaining GPU allocations
        # between baseline/ratio runs. Removing it also affects the shared callbacks.
        callbacks = getattr(wrapper, "callbacks", {}).get("on_val_end", [])
        if capture_validator in callbacks:
            callbacks.remove(capture_validator)
    box = metric.box
    per_class, class_ids = _validation_per_class(metric, class_names)
    result = {"map50_95": _validation_number(box.map), "map50": _validation_number(box.map50),
              "precision": _validation_number(box.mp), "recall": _validation_number(box.mr),
              "per_class": per_class, "split": "val", "precision_mode": "FP32", "nms": None}
    if any(result[key] is None for key in ("map50_95", "map50", "precision", "recall")):
        raise ValueError("Metrik evaluasi mengandung NaN/Inf atau tidak tersedia.")
    result["curves"] = _validation_curves(metric, class_names, class_ids)
    validator = captured.get("validator", getattr(wrapper, "validator", None))
    result.update(gflops_validation_metadata(validator, arguments, dataset))
    result["confusion_matrix"] = _validation_confusion_matrix(validator, class_names, arguments)
    result["metric_semantics"] = _validation_metric_semantics(metric)
    export_dir = Path(output_dir) / label
    export_dir.mkdir(parents=True, exist_ok=True)
    write_csv(export_dir / "per_class_metrics.csv", per_class)
    write_json(export_dir / "curves.json", result["curves"])
    write_json(export_dir / "confusion_matrix.json", result["confusion_matrix"])
    write_json(export_dir / "metric_semantics.json", result["metric_semantics"])
    result["exports"] = {name: str(export_dir / name) for name in (
        "per_class_metrics.csv", "curves.json", "confusion_matrix.json", "metric_semantics.json")}
    return result

In [ ]:
"""Dataset and student-selection declarations embedded in notebook 05."""
# ruff: noqa: F821 -- Notebook runtime supplies the V2 evaluator and helpers.

import hashlib
import json
import logging
import math
from contextlib import contextmanager
from pathlib import Path
from unittest.mock import patch

POLTRADA_V3_ID = "ilhamrafiqin/poltrada-v3-yolo26-clean"
POLTRADA_V3_MOUNT = Path("/kaggle/input/datasets") / POLTRADA_V3_ID


def _gflops_split_images(root, entries):
    """Resolve directories and image lists using dataset-relative identities."""
    from ultralytics.data.utils import IMG_FORMATS

    if isinstance(entries, str):
        entries = [entries]
    if not isinstance(entries, list) or not entries:
        raise ValueError("Split Poltrada harus berupa direktori atau daftar gambar lokal.")
    images = []
    for entry in entries:
        if not isinstance(entry, str) or not entry.strip():
            raise ValueError("Path split kosong atau tidak valid.")
        path = (root / entry).resolve()
        if (not path.exists() or not path.is_relative_to(root)) and entry.startswith("../"):
            # Roboflow exports commonly use ../valid/images from data.yaml.
            path = (root / entry.removeprefix("../")).resolve()
        if not path.is_relative_to(root):
            raise ValueError(f"Path split berada di luar DATASET_ROOT: {path}")
        if path.is_dir():
            files = (item for item in path.rglob("*") if item.is_file())
        elif path.is_file() and path.suffix.lower() == ".txt":
            files = (path.parent / line.strip() for line in path.read_text(encoding="utf-8-sig").splitlines()
                     if line.strip())
        else:
            raise FileNotFoundError(f"Direktori atau daftar gambar split tidak tersedia: {path}")
        for image in files:
            if image.suffix.lower().removeprefix(".") not in IMG_FORMATS:
                continue
            image = image.resolve()
            if not image.is_file() or not image.is_relative_to(root):
                raise ValueError(f"Gambar tidak tersedia atau berada di luar DATASET_ROOT: {image}")
            images.append(image)
    if not images:
        raise ValueError("Split Poltrada tidak berisi gambar yang dapat dievaluasi.")
    if len(set(images)) != len(images):
        raise ValueError("Split Poltrada memuat path gambar duplikat.")
    return sorted(images, key=lambda image: image.relative_to(root).as_posix())


def _gflops_fingerprint(root, images, split, names):
    """Hash every selected image and label; missing and empty labels stay distinct."""
    from ultralytics.data.utils import img2label_paths

    records = []
    for image, label in zip(images, img2label_paths([str(image) for image in images]), strict=True):
        label = Path(label).resolve()
        if not label.is_relative_to(root):
            raise ValueError(f"Label berada di luar DATASET_ROOT: {label}")
        records.append({"image": image.relative_to(root).as_posix(), "image_sha256": sha256_file(image),
                        "label": label.relative_to(root).as_posix(),
                        "label_sha256": sha256_file(label) if label.is_file() else None})
    # Encoding and sorted path order make the digest independent of the mount point.
    payload = json.dumps({"split": split, "class_names": names, "records": records},
                         ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest(), records


def gflops_prepare_dataset(mode, data_yaml, dataset_root, suite_dir, *, seed,
                           expected_fingerprint=None, dataset_id=None):
    """Freeze the input set before evaluating any model in the suite."""
    import yaml

    if mode not in {"smoke", "full_val", "test"}:
        raise ValueError("Mode evaluasi harus smoke, full_val, atau test.")
    if expected_fingerprint is not None:
        if (not isinstance(expected_fingerprint, str) or len(expected_fingerprint) != 64
                or any(char not in "0123456789abcdefABCDEF" for char in expected_fingerprint)):
            raise ValueError("Fingerprint full val harus berupa SHA-256 heksadesimal.")
        expected_fingerprint = expected_fingerprint.lower()
    source = Path(data_yaml).expanduser().resolve()
    if not source.is_file():
        return None
    root = Path(dataset_root).expanduser().resolve() if dataset_root is not None else source.parent
    if not root.is_dir():
        raise FileNotFoundError(f"DATASET_ROOT tidak tersedia: {root}")
    raw = yaml.safe_load(source.read_text(encoding="utf-8-sig"))
    if not isinstance(raw, dict) or not isinstance(raw.get("names"), (dict, list)) or not raw["names"]:
        raise ValueError("DATA_YAML harus memuat names Poltrada yang berurutan.")
    class_names = ordered_names(raw["names"])
    if raw.get("nc", len(class_names)) != len(class_names):
        raise ValueError("nc di DATA_YAML berbeda dari jumlah names.")

    split = "test" if mode == "test" else "val"
    val_images = _gflops_split_images(root, raw.get("val"))
    full_val_fingerprint = None
    if mode != "smoke" or expected_fingerprint is not None:
        full_val_fingerprint, val_records = _gflops_fingerprint(root, val_images, "val", class_names)
        if expected_fingerprint is not None and full_val_fingerprint != expected_fingerprint:
            raise ValueError("Fingerprint full val tidak cocok dengan konfigurasi yang dikunci.")

    official_mount = root == POLTRADA_V3_MOUNT.resolve() and source.is_relative_to(root)
    verified_copy = dataset_id == POLTRADA_V3_ID and expected_fingerprint is not None
    provenance_status = "VERIFIED" if official_mount or verified_copy else "UNVERIFIED"
    provenance_basis = ("KAGGLE_MOUNT" if official_mount else
                        "EXPECTED_FULL_VAL_FINGERPRINT" if verified_copy else "USER_PATH_ONLY")
    if mode == "test":
        if not raw.get("test") or not raw.get("train"):
            return None
        images = _gflops_split_images(root, raw["test"])
        fingerprint, records = _gflops_fingerprint(root, images, split, class_names)
        test_hashes = {row["image_sha256"] for row in records}
        if any(row["image_sha256"] in test_hashes for row in val_records):
            raise ValueError("Split test berbagi konten gambar dengan full val.")
        train_images = _gflops_split_images(root, raw["train"])
        if any(sha256_file(image) in test_hashes for image in train_images):
            raise ValueError("Split test berbagi konten gambar dengan train.")
    elif mode == "smoke":
        ranked = sorted(val_images, key=lambda image: (
            hashlib.sha256(f"{seed}:{image.relative_to(root).as_posix()}".encode()).hexdigest(),
            image.relative_to(root).as_posix()))
        images = sorted(ranked[:128], key=lambda image: image.relative_to(root).as_posix())
        fingerprint, records = _gflops_fingerprint(root, images, split, class_names)
    else:
        images = val_images
        fingerprint, records = full_val_fingerprint, val_records
    if any(image.with_suffix(".npy").is_file() for image in images):
        raise ValueError("Cache gambar .npy berada di samping citra evaluasi. Gunakan salinan dataset tanpa cache.")

    folder = Path(suite_dir).resolve()
    folder.mkdir(parents=True, exist_ok=True)
    paths = folder / f"evaluation_{mode}_images.txt"
    paths.write_text("".join(f"{image.as_posix()}\n" for image in images), encoding="utf-8")
    # This descriptor permits no download or implicit split fallback. An unused train
    # key is required by Ultralytics even though this notebook only calls val().
    descriptor = {"path": str(root), "train": str(paths), "val": str(paths),
                  "names": class_names, "nc": len(class_names)}
    if split == "test":
        descriptor["test"] = str(paths)
    evaluation_yaml = folder / f"evaluation_{mode}_data.yaml"
    evaluation_yaml.write_text(yaml.safe_dump(descriptor, sort_keys=False, allow_unicode=True), encoding="utf-8")
    manifest = {"mode": mode, "split": split, "seed": seed, "image_count": len(images),
                "full_val_image_count": len(val_images), "fingerprint": fingerprint,
                "full_val_fingerprint": full_val_fingerprint, "provenance_status": provenance_status,
                "provenance_basis": provenance_basis,
                "dataset_id": POLTRADA_V3_ID if official_mount else dataset_id,
                "source_yaml": str(source), "source_yaml_sha256": sha256_file(source),
                "evaluation_yaml_sha256": sha256_file(evaluation_yaml), "class_names": class_names,
                "test_content_disjoint_from_train_val": True if mode == "test" else None}
    manifest_path = folder / f"evaluation_{mode}_manifest.json"
    write_json(manifest_path, {**manifest, "files": records})
    return {**manifest, "manifest_path": str(manifest_path), "yaml": str(evaluation_yaml),
            "info": descriptor, "val_paths": [str(paths)], "image_files": [str(image) for image in images]}


def gflops_validation_arguments(dataset, output_dir, label):
    """Preserve notebook 03 settings and explicitly select the prepared split."""
    arguments = validation_arguments(dataset["yaml"], output_dir, label)
    arguments.update({"split": dataset["split"], "fraction": 1.0})
    return arguments


@contextmanager
def gflops_disable_label_cache():
    """Force label parsing from the fingerprinted files without changing the input dataset."""
    from ultralytics.data import dataset as sdk_dataset

    def set_in_memory_version(_prefix, _path, records, version):
        # get_labels() expects the writer's in-memory version field.
        records["version"] = version

    with (patch.object(sdk_dataset, "load_dataset_cache_file", side_effect=FileNotFoundError),
          patch.object(sdk_dataset, "save_dataset_cache_file", side_effect=set_in_memory_version)):
        yield


@contextmanager
def gflops_require_complete_nms():
    """Fail validation if the SDK stops processing a batch because of its NMS time limit."""
    from ultralytics.utils import LOGGER

    timeouts = []

    class TimeoutHandler(logging.Handler):
        def emit(self, record):
            message = record.getMessage()
            if "NMS time limit" in message and "exceeded" in message:
                timeouts.append(message)

    handler = TimeoutHandler()
    LOGGER.addHandler(handler)
    try:
        yield
    finally:
        LOGGER.removeHandler(handler)
    if timeouts:
        raise RuntimeError(f"Validasi parsial karena NMS timeout: {timeouts[0]}")


def gflops_validation_metadata(validator, arguments, dataset):
    """Verify the actual SDK backend and loaded images before exporting mAP."""
    if validator is None or getattr(validator, "args", None) is None:
        raise ValueError("Validator SDK aktual tidak terekam.")
    fields = ("split", "imgsz", "batch", "workers", "device", "conf", "iou", "max_det",
              "quantize", "nms", "augment", "plots", "fraction", "rect", "data")
    effective = {field: getattr(validator.args, field, None) for field in fields}
    effective["device"] = str(validator.device)
    effective["data"] = str(effective["data"])
    effective["end2end"] = getattr(validator, "end2end", None)
    if (effective["split"] != dataset["split"] or effective["end2end"] is not False
            or effective["nms"] is not None or effective["quantize"] is not None):
        raise ValueError("Validator SDK tidak memakai split, FP32, atau mode one-to-many dengan NMS yang ditetapkan.")
    loaded = getattr(getattr(validator.dataloader, "dataset", None), "im_files", ())
    if sorted(str(Path(path).resolve()) for path in loaded) != sorted(dataset["image_files"]):
        raise ValueError("Daftar gambar validator berbeda dari manifest evaluasi.")
    effective["dataloader_batch"] = validator.dataloader.batch_size
    if effective["batch"] != arguments["batch"] or effective["dataloader_batch"] != min(arguments["batch"], len(loaded)):
        raise ValueError("Batch validator aktual berbeda dari protokol (atau batas ukuran subset).")
    return {"split": dataset["split"], "evaluation_mode": dataset["mode"],
            "dataset_fingerprint": dataset["fingerprint"],
            "dataset_provenance_status": dataset["provenance_status"],
            "validation_image_count": len(loaded), "validation_arguments_requested": arguments,
            "validation_arguments_effective": effective}


def _gflops_complete_student_row(row):
    if (row.get("evaluation_mode") != "full_val" or row.get("dataset_provenance_status") != "VERIFIED"
            or any(row.get(key) != "PASS" for key in ("structural_status", "reload_status", "validation_status"))
            or row.get("latency_status") not in {"PASS", "MEASURED"}):
        return False
    fields = ("target_speedup", "gflops_speedup", "map50_95", "baseline_map50_95", "gflops",
              "parameters", "latency_median_ms")
    if any(not isinstance(row.get(key), (float, int)) or isinstance(row[key], bool)
           or not math.isfinite(row[key]) for key in fields):
        return False
    return (row.get("model_variant") in {"n", "s"} and row["target_speedup"] in {1.02, 1.05, 1.10, 1.20, 1.70}
            and row.get("target_status", "REACHED") == "REACHED"
            and row.get("baseline_validation_parity_status", "PASS") == "PASS"
            and row["gflops_speedup"] >= row["target_speedup"]
            and 0 <= row["map50_95"] <= 1 and 0 <= row["baseline_map50_95"] <= 1
            and row["baseline_map50_95"] - row["map50_95"] <= 0.02 + 1e-12
            and all(row[key] > 0 for key in ("gflops", "parameters", "latency_median_ms")))

In [ ]:
"""Cross-configuration analysis for notebook 05, following notebook 03's EDA.

Consumes completed run records and their original inventories. It never prunes,
validates, selects a student, or supplies a value for a missing measurement.
"""
# ruff: noqa: F821 -- Shared EDA and evaluation declarations precede this cell.

import pandas as pd


def _gflops_eda_csv(path):
    if not path.is_file() or not path.stat().st_size or not path.read_text(encoding="utf-8").strip():
        return []
    frame = pd.read_csv(path)
    return frame.astype(object).where(pd.notna(frame), None).to_dict("records")


def _gflops_eda_delta(after, before, scale=1):
    return scale * (after - before) if _eda_finite(after) and _eda_finite(before) else None


def gflops_eda_tables(bundle, suite_dir):
    """Join classes by ID and keep model baselines, denominators and statuses explicit."""
    tables = {key: [] for key in (
        "metrics", "sources", "architecture", "coverage", "layers", "stages", "groups", "ledger",
        "per_class_metrics", "validation_protocol", "verification", "target_tiers", "target_search",
        "search_profiles", "p5_contrasts", "p5_per_class", "student_eligibility",
    )}
    for model in bundle.get("models", []):
        identity = {"model": model["model_name"], "model_variant": model["model_variant"]}
        folder = _EdaPath(suite_dir) / model["model_name"].lower()
        baseline_report = model.get("baseline_report", {})
        baseline = baseline_report.get("reserialized", model.get("baseline", {}))
        baseline_val = baseline.get("validation", {}) if baseline.get("validation_status") == "PASS" else {}
        baseline_classes = {row["class_id"]: row for row in baseline_val.get("per_class", [])}
        source = model.get("source", {})
        tables["sources"].append({**identity, **source,
                                   "baseline_validation_parity_status": baseline_report.get("validation_parity_status", "NOT_RUN")})
        for row in baseline_report.get("architecture", {}).get("stages", []):
            tables["architecture"].append({**identity, **row})
        records = {row["configuration_id"]: row for row in model.get("rows", [])}
        processed = set()
        stages = [(f"baseline-{kind}", "baseline", baseline_report.get(kind, {}), None)
                  for kind in ("source", "reserialized")]
        stages.extend((run["configuration_id"], "pruned", run.get("pruned", {}), run)
                      for run in model.get("results", []))
        for config_id, stage, metric, run in stages:
            if not metric and run is None:
                continue
            processed.add(config_id)
            audit = (run or {}).get("audit", {})
            info = {**identity, "configuration_id": config_id, "stage": stage,
                    "policy": run["policy"] if run else "baseline", "ratio": run.get("ratio") if run else 0}
            validation_status = run.get("validation_status", "NOT_RUN") if run else metric.get("validation_status", "NOT_RUN")
            val = metric.get("validation", {}) if validation_status == "PASS" else {}
            latency = metric.get("latency", {})
            row = {**records.get(config_id, {}), **info,
                   **{key: metric.get(key) for key in ("parameters", "gflops", "checkpoint_bytes")},
                   **{key: val.get(key) for key in ("map50_95", "map50", "precision", "recall")},
                   "baseline_map50_95": baseline_val.get("map50_95"), "validation_status": validation_status,
                   "evaluation_mode": val.get("evaluation_mode", bundle.get("mode")),
                   "dataset_fingerprint": val.get("dataset_fingerprint"),
                   "dataset_provenance_status": val.get("dataset_provenance_status"),
                   "validation_image_count": val.get("validation_image_count"),
                   "latency_status": latency.get("status", "NOT_RUN"),
                   "latency_median_ms": latency.get("median_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                   "latency_p95_ms": latency.get("p95_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                   "gflops_speedup": baseline.get("gflops") / metric["gflops"]
                   if _eda_finite(metric.get("gflops")) and metric["gflops"] > 0 and baseline.get("gflops") else None,
                   "latency_speedup": baseline["latency"]["median_ms"] / latency["median_ms"]
                   if latency.get("status") in {"PASS", "MEASURED"} and latency.get("median_ms", 0) > 0
                   and baseline.get("latency", {}).get("status") in {"PASS", "MEASURED"}
                   and baseline["latency"].get("median_ms") else None,
                   "delta_map50_95_pp": _gflops_eda_delta(val.get("map50_95"), baseline_val.get("map50_95"), 100),
                   "pruned_group_count": len((run or {}).get("groups", [])) if run else 0}
            if run is None:
                row.update(structural_status="BASELINE", actual_eligible_ratio=0, eda_status="NOT_RUN")
                reload = baseline_report.get("reload", {})
                row["reload_status"] = ("PASS" if config_id == "baseline-reserialized"
                                        and reload.get("direct_yolo_load") is True and reload.get("raw_parity") is True
                                        else row.get("reload_status", "NOT_RUN"))
            for key, label in (("parameters", "parameter"), ("gflops", "gflops"), ("checkpoint_bytes", "checkpoint")):
                delta = _gflops_eda_delta(metric.get(key), baseline.get(key), -100)
                row[label + "_reduction_pct"] = delta / baseline[key] if delta is not None and baseline.get(key) else None
            delta = _gflops_eda_delta(val.get("map50_95"), baseline_val.get("map50_95"), 100)
            row["relative_map50_95_pct"] = delta / baseline_val["map50_95"] if delta is not None and baseline_val.get("map50_95") else None
            tables["metrics"].append(row)
            protocol = {key: val.get(key) for key in (
                "split", "evaluation_mode", "dataset_fingerprint", "dataset_provenance_status", "validation_image_count",
            )}
            tables["validation_protocol"].append({**info, "validation_status": validation_status, **protocol,
                                                   **val.get("validation_arguments_effective", {})})
            # The reserialized baseline is the one common reference for all pruned students.
            if config_id != "baseline-source":
                for entry in val.get("per_class", []):
                    prior = baseline_classes.get(entry["class_id"], {})
                    if prior and prior.get("name") != entry.get("name"):
                        raise ValueError(f"Nama/ID kelas tidak konsisten pada {model['model_name']} / {config_id}.")
                    tables["per_class_metrics"].append({**info, **entry,
                        "validation_status": validation_status, "evaluation_mode": row["evaluation_mode"],
                        "baseline_ap50_95": prior.get("ap50_95"), "baseline_ap50": prior.get("ap50"),
                        **{f"delta_{key}_pp": _gflops_eda_delta(entry.get(key), prior.get(key), 100)
                           for key in ("ap50_95", "ap50", "precision", "recall", "f1")}})
            if run is None:
                continue
            eda_dir = folder / run.get("artifact_configuration_id", config_id) / "eda"
            layers = _gflops_eda_csv(eda_dir / "layer_eda.csv")
            convs = [layer for layer in layers if layer["module_type"] == "Conv2d"]
            denominator = audit.get("target_policy_eligible_channels", audit.get("independent_channels_before"))
            removed = audit.get("independent_channels_removed")
            coverage = {**info, "graph_eligible_groups": len(audit.get("eligible_groups", [])),
                        "graph_rejected_roots": len(audit.get("rejected_groups", [])),
                        "selected_groups": len(run.get("groups", [])),
                        "graph_independent_channels_before": audit.get("independent_channels_before"),
                        "policy_eligible_channels": denominator, "independent_channels_removed": removed,
                        "policy_eligible_channels_after": denominator - removed
                        if _eda_finite(denominator) and _eda_finite(removed) else None,
                        "actual_eligible_ratio": run.get("actual_eligible_channel_ratio"),
                        "physical_conv_outputs_removed": audit.get("physical_conv_output_channels_removed"),
                        "inventory_status": "MEASURED" if layers else "UNAVAILABLE",
                        "conv_layers": len(convs) if layers else None,
                        "changed_conv_outputs": sum(layer["out_channels_removed"] > 0 for layer in convs) if layers else None}
            tables["coverage"].append(coverage)
            tables["layers"].extend({**info, **layer} for layer in layers)
            tables["stages"].extend({**info, **item} for item in _gflops_eda_csv(eda_dir / "stage_eda.csv"))
            tables["ledger"].extend({**info, **item} for item in audit.get("channel_mapping", []))
            scores = {}
            for item in audit.get("lamp_scores", []):
                scores.setdefault(item["root"], []).append(item)
            for entry in audit.get("eligible_groups", []):
                values = scores.get(entry["name"], [])
                finite = sorted(item["score"] for item in values if _eda_finite(item.get("score")))
                tables["groups"].append({**info, "root": entry["name"], "group_id": entry["group_id"],
                    "decision": "GRAPH_ELIGIBLE", "initial_channels": entry["initial_channels"],
                    "capacity_eligible": sum(bool(item.get("capacity_eligible")) for item in values),
                    "removed": sum(bool(item.get("removed")) for item in values),
                    "score_min": finite[0] if finite else None,
                    "score_median": float(_eda_np.median(finite)) if finite else None,
                    "score_max": finite[-1] if finite else None,
                    "coupled_output_layers": entry.get("signature"), "reason": ""})
            tables["groups"].extend({**info, **entry, "decision": "GRAPH_REJECTED"}
                                    for entry in audit.get("rejected_groups", []))
            tables["verification"].append({**info,
                "structural_status": run.get("structural_smoke_status", "NOT_RUN"),
                "oracle_status": run.get("oracle", {}).get("status", "NOT_RUN"),
                "reproducibility_status": run.get("reproducibility_status", "NOT_RUN"),
                "reload_status": run.get("reload_status", "NOT_RUN"), "validation_status": validation_status,
                "baseline_validation_parity_status": baseline_report.get("validation_parity_status", "NOT_RUN"),
                "latency_status": row["latency_status"], "eda_status": run.get("eda_status", "NOT_RUN"),
                "checkpoint_sha256": run.get("saved_pruned", {}).get("sha256")})
            tables["student_eligibility"].append({**info,
                "target_speedup": row.get("target_speedup"), "target_status": row.get("target_status"),
                "gflops_speedup": row.get("gflops_speedup"), "delta_map50_95_pp": row["delta_map50_95_pp"],
                "evaluation_mode": row["evaluation_mode"], "dataset_provenance_status": row["dataset_provenance_status"],
                "latency_status": row["latency_status"], "qualifies": _gflops_complete_student_row(row)})
        tables["metrics"].extend({**row, "stage": "pruned"} for config_id, row in records.items() if config_id not in processed)
        search = model.get("target_search", {})
        for key, source_key in (("target_tiers", "tiers"), ("search_profiles", "measurements"), ("target_search", "targets")):
            tables[key].extend({**identity, **{k: v for k, v in item.items() if k not in {"plan", "caps"}}}
                               for item in search.get(source_key, []))
        for key in ("p5_contrasts", "p5_per_class"):
            tables[key].extend({**identity, **entry} for entry in _gflops_eda_csv(folder / f"{key}.csv"))
    return tables


def _gflops_eda_finish(fig, path, count, reason="Pengukuran berpasangan belum tersedia."):
    files = _eda_save_figure(fig, path)
    return {"plot": path.name, "status": "MEASURED" if count else "NOT_RUN",
            "measured_values": count, "reason": None if count else reason, **files}


def _gflops_eda_panel(ax, rows, x, y, xlabel, ylabel):
    valid = [row for row in rows if _eda_finite(row.get(x)) and _eda_finite(row.get(y))]
    if valid:
        ax.plot([row[x] for row in valid], [row[y] for row in valid], "o-")
    else:
        ax.text(.5, .5, "NOT_RUN", ha="center", transform=ax.transAxes)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=.2)
    return len(valid)


def gflops_eda_model_plots(tables, model_name, folder):
    """Notebook 03 sweep/AP plots, with computational targets shown separately."""
    folder = _EdaPath(folder)
    rows = [row for row in tables["metrics"] if row["model"] == model_name and row["configuration_id"] != "baseline-source"]
    protocol = next((row for row in tables["validation_protocol"] if row["model"] == model_name
                     and row.get("validation_status") == "PASS"), {})
    scope = " · " + str(protocol.get("evaluation_mode", "NOT_RUN"))
    if protocol.get("validation_image_count") is not None:
        scope += f" · {protocol['validation_image_count']} citra"
    if protocol.get("imgsz") is not None:
        scope += f" · {protocol['imgsz']} px"
    statuses = []
    controls = sorted((row for row in rows if row["policy"] in {"baseline", "extended"}), key=lambda row: row["ratio"])
    fig, axes = _eda_plt.subplots(1, 3, figsize=(17, 5))
    count = sum(_gflops_eda_panel(ax, controls, "ratio", metric, "Extended eligible-channel ratio", label)
                for ax, metric, label in zip(axes, ("map50_95", "parameter_reduction_pct", "gflops_reduction_pct"),
                                            ("mAP50–95", "Parameter reduction (%)", "GFLOPs reduction (%)")))
    fig.suptitle(model_name + scope + "\nKontrol extended 10–50%; target GFLOPs dianalisis terpisah")
    fig.tight_layout()
    statuses.append(_gflops_eda_finish(fig, folder / "extended_sweep", count))

    fig, axes = _eda_plt.subplots(2, 2, figsize=(17, 10))
    count = 0
    for ax, metric in zip(axes.flat, ("map50_95", "map50", "precision", "recall")):
        measured = [(index, row[metric]) for index, row in enumerate(rows) if _eda_finite(row.get(metric))]
        if measured:
            ax.scatter(*zip(*measured))
        else:
            ax.text(.5, .5, "NOT_RUN", ha="center", transform=ax.transAxes)
        ax.set_xticks(range(len(rows)), [row["configuration_id"] for row in rows], rotation=65, ha="right", fontsize=8)
        ax.set(ylabel=metric, ylim=(0, 1))
        ax.grid(axis="y", alpha=.2)
        if metric == "map50_95" and rows and _eda_finite(rows[0].get("baseline_map50_95")):
            ax.axhline(rows[0]["baseline_map50_95"] - .02, color="red", linestyle="--", label="baseline − 2 pp")
            ax.legend()
        count += len(measured)
    fig.suptitle(model_name + scope + "\nMetrik setiap konfigurasi")
    fig.tight_layout()
    statuses.append(_gflops_eda_finish(fig, folder / "configuration_metrics", count))

    classes = [row for row in tables["per_class_metrics"] if row["model"] == model_name]
    class_names = dict(sorted({row["class_id"]: row["name"] for row in classes}.items()))
    configs = [row["configuration_id"] for row in rows]
    fig, axes = _eda_plt.subplots(1, 2, figsize=(max(14, len(class_names) * 2.5), max(5, len(configs) * .45)))
    count = 0
    for ax, metric, label in zip(axes, ("ap50_95", "delta_ap50_95_pp"), ("AP50–95", "Delta AP50–95 vs baseline (pp)")):
        values = {(row["configuration_id"], row["class_id"]): row.get(metric) for row in classes}
        data = _eda_np.array([[values.get((config, class_id)) if _eda_finite(values.get((config, class_id))) else _eda_np.nan
                              for class_id in class_names] for config in configs], dtype=float)
        measured = int(_eda_np.isfinite(data).sum())
        if measured:
            options = {"vmin": 0, "vmax": 1, "cmap": "Blues"} if metric == "ap50_95" else {
                "vmin": -max(1, float(_eda_np.nanmax(_eda_np.abs(data)))),
                "vmax": max(1, float(_eda_np.nanmax(_eda_np.abs(data)))), "cmap": "RdBu"}
            plot = ax.imshow(_eda_np.ma.masked_invalid(data), aspect="auto", **options)
            fig.colorbar(plot, ax=ax)
            for i, j in zip(*_eda_np.where(_eda_np.isfinite(data))):
                ax.text(j, i, f"{data[i, j]:.3f}" if metric == "ap50_95" else f"{data[i, j]:+.2f}",
                        ha="center", va="center", fontsize=7)
        else:
            ax.text(.5, .5, "NOT_RUN", ha="center", transform=ax.transAxes)
        ax.set_xticks(range(len(class_names)), list(class_names.values()), rotation=45, ha="right")
        ax.set_yticks(range(len(configs)), configs)
        ax.set_title(label + " · sel kosong = tidak tersedia")
        count += measured
    fig.suptitle(model_name + scope + "\nAP per kelas untuk seluruh konfigurasi")
    fig.tight_layout()
    statuses.append(_gflops_eda_finish(fig, folder / "per_class_ap_all", count))

    for metric, label in (("parameters", "Parameters"), ("gflops", "GFLOPs"), ("latency_median_ms", "Median latency raw forward (ms)")):
        fig, ax = _eda_plt.subplots(figsize=(11, 6))
        valid = [row for row in rows if _eda_finite(row.get(metric)) and _eda_finite(row.get("map50_95"))]
        for row in valid:
            ax.scatter(row[metric], row["map50_95"])
            ax.annotate(row["configuration_id"], (row[metric], row["map50_95"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
        if not valid:
            ax.text(.5, .5, "NOT_RUN", ha="center", transform=ax.transAxes)
        ax.set(xlabel=label, ylabel="mAP50–95", title=model_name + scope + "\nAkurasi dan efisiensi")
        ax.grid(alpha=.2)
        fig.tight_layout()
        statuses.append(_gflops_eda_finish(fig, folder / ("tradeoff_" + metric), len(valid)))

    profiles = [row for row in tables["search_profiles"] if row["model"] == model_name]
    tiers = [row for row in tables["target_tiers"] if row["model"] == model_name]
    fig, axes = _eda_plt.subplots(1, 2, figsize=(15, 5))
    for tier in sorted({row["tier"] for row in profiles}):
        values = sorted((row for row in profiles if row["tier"] == tier), key=lambda row: row["channels"])
        axes[0].plot([row["channels"] for row in values], [row["speed_up"] for row in values], ".-", label=f"tier {tier}")
    if profiles:
        axes[0].legend()
    else:
        axes[0].text(.5, .5, "NOT_RUN", ha="center", transform=axes[0].transAxes)
    tiers = [row for row in tiers if _eda_finite(row.get("max_gflops_speed_up"))]
    if tiers:
        axes[1].bar([str(row["tier"]) for row in tiers], [row["max_gflops_speed_up"] for row in tiers])
        for target in sorted({row["target"] for row in tables["target_search"] if row["model"] == model_name}):
            axes[1].axhline(target, linestyle="--", label=f"target {target:g}×")
        axes[1].legend()
    else:
        axes[1].text(.5, .5, "NOT_RUN", ha="center", transform=axes[1].transAxes)
    axes[0].set(xlabel="Channel independen dihapus", ylabel="GFLOPs speed-up", title="Kandidat pencarian, tanpa evaluasi mAP")
    axes[1].set(xlabel="Tier cap", ylabel="Maksimum GFLOPs speed-up", title="Kapasitas dengan proteksi aktif")
    fig.suptitle(model_name + scope + "\nPencarian target komputasi")
    fig.tight_layout()
    statuses.append(_gflops_eda_finish(fig, folder / "target_search_capacity", len(profiles) + len(tiers)))
    return statuses


def _gflops_build_analysis(bundle, suite_dir):
    """Export analysis for every completed configuration independently of detail display."""
    folder = _EdaPath(suite_dir) / "comparison/analysis"
    folder.mkdir(parents=True, exist_ok=True)
    tables = gflops_eda_tables(bundle, suite_dir)
    for key, rows in tables.items():
        _eda_write_csv(folder / f"{key}.csv", rows)
        _eda_write_json(folder / f"{key}.json", rows)
    plots = []
    for model in bundle.get("models", []):
        plots.extend({"model": model["model_name"], **row}
                     for row in gflops_eda_model_plots(tables, model["model_name"], folder / model["model_name"].lower()))
    report = {"status": "PASS" if bundle.get("models") else "NOT_RUN", "mode": bundle.get("mode"),
              "table_rows": {key: len(rows) for key, rows in tables.items()}, "plots": plots,
              "interpretation": "Smoke checks the pipeline. Full validation selects a student. Layer distributions are descriptive.",
              "selection": bundle.get("selection"),
              "display_configurations": list(globals().get("EDA_CONFIGURATIONS", ())),
              "detail_configurations": list(globals().get("EDA_DETAIL_CONFIGURATIONS", ())), "tables": tables}
    _eda_write_json(folder / "analysis_status.json", {key: value for key, value in report.items() if key != "tables"})
    return report


def gflops_build_analysis(bundle, suite_dir):
    path = _EdaPath(suite_dir) / "comparison/analysis/analysis_status.json"
    _eda_write_json(path, {"status": "RUNNING"})
    try:
        return _gflops_build_analysis(bundle, suite_dir)
    except Exception as error:
        _eda_write_json(path, {"status": "FAIL", "error": str(error), "type": type(error).__name__})
        raise

In [ ]:
def gflops_latency(core, image):
    if not RUN_LATENCY_BENCHMARK:
        return {"status": "NOT_RUN", "reason": "Benchmark dimatikan."}
    try:
        return cuda_latency(core, image, LATENCY_WARMUP, LATENCY_REPEATS)
    except Exception as error:
        return {"status": "FAIL", "error": safe_error(error)}


def gflops_validation(row, checkpoint, dataset, folder):
    if dataset is None:
        return
    try:
        row["validation"] = evaluate_checkpoint(checkpoint, dataset, folder, "validation")
        row["validation_status"] = "PASS"
    except Exception as error:
        row["validation_status"] = "FAIL"
        row["errors"].append({"stage": "validation", "error": safe_error(error)})


def gflops_baseline(path, provenance, variant, image, folder, dataset):
    folder.mkdir(parents=True, exist_ok=True)
    previous_path = folder / "baseline_report.json"
    previous = json.loads(previous_path.read_text(encoding="utf-8")) if previous_path.is_file() else {}
    core = gflops_load_source(path, provenance, variant)
    forward_snapshot(core, image)
    measurements = model_measurements(core, IMGSZ)
    latency = previous.get("reserialized", {}).get("latency", {})
    measurements["latency"] = latency if latency.get("status") in {"PASS", "MEASURED"} else gflops_latency(core, image)
    source = previous.get("source") or {**measurements, "checkpoint_bytes": Path(path).stat().st_size,
                                       "validation_status": "NOT_RUN", "errors": []}
    if source["validation_status"] != "PASS":
        gflops_validation(source, path, dataset, folder / "source")
    saved = workflow_save_checkpoint(core, folder / "baseline_fp32.pt", provenance)
    reload_report = previous.get("reload")
    if not reload_report:
        reload_report = verify_fresh_reload(saved["path"], core, image, folder, timeout=RELOAD_TIMEOUT_SECONDS)
    clone = gflops_reload(saved["path"], core)
    if model_state_sha256(core) != model_state_sha256(clone):
        raise ValueError("Baseline hasil reload mengubah state sumber.")
    baseline = previous.get("reserialized") or {**measurements, "checkpoint_bytes": saved["file_bytes"],
                                                "validation_status": "NOT_RUN", "errors": []}
    source["latency"] = baseline["latency"] = measurements["latency"]
    if baseline["validation_status"] != "PASS":
        gflops_validation(baseline, saved["path"], dataset, folder / "reserialized")
    parity = "NOT_RUN"
    if dataset is not None:
        fields = ("map50_95", "map50", "precision", "recall")
        parity = ("PASS" if source["validation_status"] == baseline["validation_status"] == "PASS"
                  and all(abs(source["validation"][key] - baseline["validation"][key]) <= 1e-6 for key in fields)
                  else "FAIL")
    result = {"source": source, "reserialized": baseline, "validation_parity_status": parity,
              "state_sha256": model_state_sha256(core), "checkpoint": saved, "reload": reload_report,
              "architecture": architecture_audit(core, variant)["report"]}
    write_json(previous_path, result)
    return result


def gflops_reload(path, reference):
    model = YOLO(str(path), task="detect").model.to(TARGET_DEVICE).float().eval()
    model.end2end = reference.end2end
    return model


def gflops_make_graph(core, image, variant, ratio):
    return build_policy_pruner(core, image, "extended", ratio, variant)


def gflops_plan_from_history(history):
    return {item["root"]: sorted(item["removed_original_indices"]) for item in history}

In [ ]:
"""Standalone orchestration for the YOLO26s LAMP multi-seed notebook."""
# ruff: noqa: F821, BLE001 -- Notebook stages share declarations and record independent failures.
import gc
import shutil
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

from filelock import FileLock

PRIMARY_TARGET = 1.20
PRUNING_SEEDS = (42, 0, 2024, 7, 3407)
PRUNING_QUEUE = ((1.10, 42), (PRIMARY_TARGET, 42), (1.70, 42)) + tuple(
    (PRIMARY_TARGET, seed) for seed in PRUNING_SEEDS[1:]
)
EVAL_SEED = 42


def ms_read(path, default=None):
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else default


def ms_id(target, seed):
    return f"target-{target:.2f}x-seed{seed}"


def ms_run_name(target, seed):
    return f"yolo26s-lamp-su{target:.2f}x-seed{seed}"


def ms_model_name(seed):
    return f"YOLO26s-seed{seed}"


def ms_run_group(suite_id):
    return f"yolo26s-lamp-multiseed-{suite_id}"





def ms_dataset(folder):
    portable = ms_read(Path(DATASET_MANIFEST_PATH).expanduser()) if DATASET_MANIFEST_PATH else None
    if DATASET_MANIFEST_PATH and (not portable or portable.get("dataset_id") != DATASET_ID
                                  or portable.get("provenance_status") != "VERIFIED"):
        raise ValueError("Gunakan dataset_manifest.json Poltrada V3 dari sumber terverifikasi.")
    check_dir = folder / "dataset_checks" / uuid.uuid4().hex[:8]
    data = gflops_prepare_dataset(MODE, DATA_YAML, DATASET_ROOT, check_dir, seed=EVAL_SEED,
                                  expected_fingerprint=portable["val_fingerprint"] if portable else None,
                                  dataset_id=DATASET_ID)
    if data is None:
        raise FileNotFoundError("Atur DATASET_ROOT dan DATA_YAML ke Poltrada V3 YOLO26 Clean.")
    if MODE == "full_val" and data["provenance_status"] != "VERIFIED":
        raise ValueError("Full validation memerlukan mount Kaggle resmi atau manifest Poltrada V3 terverifikasi.")
    record, _, _ = workflow_dataset_manifest(data, DATA_YAML, DATASET_ROOT, check_dir)
    if portable and (record["train_fingerprint"] != portable["train_fingerprint"]
                     or record["val_fingerprint"] != portable["val_fingerprint"]):
        raise ValueError("Fingerprint train/val berbeda dari manifest sumber.")
    data["train_fingerprint"] = record["train_fingerprint"]
    return data, record


def ms_bind(folder, dataset):
    path = folder / "resume_contract.json"
    previous = ms_read(path, {}).get("contract", {})
    if RESUME_SUITE_DIR and "suite_id" not in previous:
        raise ValueError("RESUME_SUITE_DIR harus menunjuk ke suite per-seed yang sudah terikat.")
    return workflow_bind(folder, "resume_contract.json", {
        "suite_id": previous.get("suite_id", folder.name),
        "implementation": IMPLEMENTATION_SHA256, "variant": "s", "mode": MODE,
        "queue": PRUNING_QUEUE, "baselines": BASELINE_SOURCES,
        "dataset_id": dataset["dataset_id"], "val_fingerprint": dataset["fingerprint"],
        "train_fingerprint": dataset["train_fingerprint"], "provenance": dataset["provenance_status"],
        "imgsz": IMGSZ, "batch": BATCH, "workers": WORKERS, "eval_seed": EVAL_SEED,
        "run_latency": RUN_LATENCY_BENCHMARK, "latency_warmup": LATENCY_WARMUP, "latency_repeats": LATENCY_REPEATS,
        "platform": PLATFORM, "device": str(TARGET_DEVICE), "environment": ENVIRONMENT,
        "wandb_entity": WANDB_ENTITY, "wandb_project": WANDB_PROJECT,
        "wandb_base_url": os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai").rstrip("/"),
    })


def ms_check_source(path, record, seed):
    pinned = BASELINE_SOURCES[seed]
    if (record["baseline_seed"] != seed or record["requested_artifact"] != pinned["ref"]
            or record["sha256"] != pinned["sha256"] or sha256_file(path) != pinned["sha256"]):
        raise ValueError(f"Identitas atau SHA-256 best.pt baseline seed {seed} berbeda dari pin.")





def gflops_load_source(path, provenance, variant):
    ms_check_source(path, provenance, provenance["baseline_seed"])
    core = YOLO(str(path), task="detect").model.to(TARGET_DEVICE).float().eval()
    core.end2end = True
    architecture_audit(core, variant)
    return core


def ms_probe(seed):
    return torch.rand(1, 3, IMGSZ, IMGSZ, generator=torch.Generator().manual_seed(seed)).to(TARGET_DEVICE)


def ms_baseline(folder, path, source, dataset):
    previous = ms_read(folder / "baseline_report.json")
    if previous:
        checkpoint = folder / "baseline_fp32.pt"
        if sha256_file(checkpoint) != previous["checkpoint"]["sha256"]:
            raise ValueError("SHA-256 baseline FP32 tersimpan berubah.")
        if (previous["validation_parity_status"] == "PASS"
                and (MODE == "smoke" or previous["reserialized"]["latency"]["status"] == "PASS")):
            previous["checkpoint"]["path"] = str(checkpoint.resolve())
            return previous
    seed_environment(EVAL_SEED)
    return gflops_baseline(path, source, "s", ms_probe(EVAL_SEED), folder, dataset)


def ms_search(folder, path, source, baseline, seed, targets):
    if source["baseline_seed"] != seed:
        raise ValueError("Seed pencarian berbeda dari seed baseline.")
    seed_environment(seed)
    image = ms_probe(seed)
    graph = gflops_load_source(path, source, "s")
    _, audit = gflops_make_graph(graph, image, "s", 0.10)

    def measure(plan):
        probe = gflops_load_source(path, source, "s")
        pruner, candidate_audit = gflops_make_graph(probe, image, "s", 0.10)
        apply_exact_gflops_plan(pruner, candidate_audit, plan)
        return model_measurements(probe, IMGSZ)["gflops"]

    cache_path = folder / "profile_cache.json"

    def progress(values, points):
        write_json(cache_path, values)
        write_json(folder / "profile_progress.json", points)

    search = search_gflops_targets(audit, baseline["reserialized"]["gflops"], measure, targets,
                                   profile_cache=ms_read(cache_path, {}), progress_callback=progress)
    write_json(folder / "target_search.json", search)
    return search


def ms_signature(core):
    return {
        "parameters": {name: list(value.shape) for name, value in core.named_parameters()},
        "conv_bn": {name: [module.in_channels, module.out_channels, module.groups]
                    if isinstance(module, torch.nn.Conv2d) else [module.num_features]
                    for name, module in core.named_modules()
                    if isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d))},
    }


def ms_audit(path, source, baseline, seed, target_row):
    if source["baseline_seed"] != seed:
        raise ValueError("Seed pruning berbeda dari seed baseline.")
    seed_environment(seed)
    image = ms_probe(seed)
    core = gflops_load_source(path, source, "s")
    if model_state_sha256(core) != baseline["state_sha256"]:
        raise ValueError("State awal kandidat berbeda dari baseline seed yang sama.")
    original = deepcopy(core)
    protected = protected_policy_snapshot(core, "s")
    pruner, audit = gflops_make_graph(core, image, "s", 0.10)
    before = measure_layer_inventory(core, image, audit, "extended")
    plan, caps = target_row["plan"], target_row["caps"]
    history = apply_exact_gflops_plan(pruner, audit, plan, caps=caps)
    assert_layer_consistency(core)
    assert_policy_protected_unchanged(core, protected)
    forward_snapshot(core, image)
    oracle = verify_gflops_oracle(original, core, image, audit)
    del original

    seed_environment(seed)
    repeat = gflops_load_source(path, source, "s")
    repeat_pruner, repeat_audit = gflops_make_graph(repeat, image, "s", 0.10)
    replay = apply_exact_gflops_plan(repeat_pruner, repeat_audit, plan, caps=caps)
    report = audit["report"]
    ledger = report["channel_mapping"]
    state = model_state_sha256(core)
    signature = ms_signature(core)
    if (gflops_plan_key(gflops_plan_from_history(history)) != gflops_plan_key(plan)
            or history != replay or ledger != repeat_audit["report"]["channel_mapping"]
            or model_state_sha256(repeat) != state or ms_signature(repeat) != signature):
        raise ValueError("Replay mengubah plan, ledger, struktur, atau tensor.")
    del repeat, repeat_pruner, repeat_audit

    after = measure_layer_inventory(core, image, audit, "extended")
    measured = model_measurements(core, IMGSZ)
    if (measured["parameters"] >= baseline["reserialized"]["parameters"]
            or not math.isclose(measured["gflops"], target_row["gflops"], rel_tol=1e-10)):
        raise ValueError("Parameter atau GFLOPs kandidat berbeda dari profil pencarian.")
    identity = {"plan_sha256": workflow_digest(gflops_plan_key(plan)), "ledger_sha256": workflow_digest(ledger),
                "structure_sha256": workflow_digest(signature), "state_sha256": state}
    return core, {**identity, "identity_sha256": workflow_digest({**identity, "source": source["sha256"]}),
                  "structure_signature": signature, "oracle": oracle, "audit": report, "groups": history,
                  "selected_plan": plan, "pruned": measured}, before, after


def ms_verify_files(folder, candidate):
    saved = candidate["saved_pruned"]
    if sha256_file(folder / "pruned_fp32.pt") != saved["sha256"]:
        raise ValueError("SHA-256 raw checkpoint tersimpan berubah.")
    for name, digest in candidate["audit_files"].items():
        if sha256_file(folder / name) != digest:
            raise ValueError(f"SHA-256 audit berubah: {name}")


def ms_commit_raw(folder, core, proof, source, before, after):
    key = proof["identity_sha256"]
    journal = folder / "pruning_manifest.json"
    candidate = ms_read(journal)
    if candidate:
        if candidate["identity_sha256"] != key:
            raise ValueError("Direktori perlakuan sudah berisi kandidat berbeda.")
        ms_verify_files(folder, candidate)
    else:
        saved = workflow_save_checkpoint(core, folder / "pruned_fp32.pt", source)
        proof["pruned"]["checkpoint_bytes"] = saved["file_bytes"]
        write_json(folder / "dependency_audit.json", proof["audit"])
        write_json(folder / "inventories.json", {"before": before, "after": after})
        candidate = {**proof, "artifact_configuration_id": folder.name, "model_variant": "s",
                     "model_name": ms_model_name(source["baseline_seed"]),
                     "provenance": source,
                     "policy": "gflops-target", "ratio": sum(map(len, proof["selected_plan"].values())) /
                     proof["audit"]["target_policy_eligible_channels"],
                     "pruned_group_count": len(proof["groups"]), "saved_pruned": {"sha256": saved["sha256"],
                     "file_bytes": saved["file_bytes"]}, "audit_files": {
                         name: sha256_file(folder / name) for name in ("dependency_audit.json", "inventories.json")},
                     "structural_smoke_status": "PASS", "reproducibility_status": "PASS", "reload_status": "NOT_RUN",
                     "validation_status": "NOT_RUN", "eda_status": "NOT_RUN", "errors": []}
        candidate["actual_eligible_channel_ratio"] = candidate["ratio"]
        write_json(journal, candidate)
    if candidate["reload_status"] != "PASS":
        checkpoint = folder / "pruned_fp32.pt"
        image = ms_probe(EVAL_SEED)
        try:
            candidate["reload"] = verify_fresh_reload(checkpoint, core, image, folder, timeout=RELOAD_TIMEOUT_SECONDS)
            restored = gflops_reload(checkpoint, core)
            if model_state_sha256(restored) != proof["state_sha256"] or ms_signature(restored) != proof["structure_signature"]:
                raise ValueError("State atau signature arsitektur berubah setelah reload.")
        except Exception as error:
            candidate["reload_status"] = "FAIL"
            candidate["errors"].append({"stage": "reload", "error": safe_error(error)})
            write_json(journal, candidate)
            raise
        candidate["reload_status"] = "PASS"
        write_json(journal, candidate)
    return candidate


def ms_attempts(folder):
    return [ms_read(folder / "attempts" / f"{ms_id(target, seed)}.json", {
        "configuration_id": ms_id(target, seed), "target_speedup": target, "pruning_seed": seed,
        "recovery_seed": seed, "target_status": "NOT_RUN", "search_status": "NOT_RUN", "audit_status": "NOT_RUN",
    }) for target, seed in PRUNING_QUEUE]


def ms_find_targets(folder, path, source, baseline, seed, attempts):
    search = ms_read(folder / "target_search.json", {})
    ready = {item["target"] for item in search.get("targets", []) if item["status"] != "FAIL"}
    if any(row["audit_status"] != "PASS" and row["target_speedup"] not in ready for row in attempts):
        try:
            search = ms_search(folder, path, source, baseline, seed, tuple(row["target_speedup"] for row in attempts))
        except Exception as error:
            search = {"targets": [{"target": row["target_speedup"], "status": "FAIL", "error": safe_error(error)}
                                  for row in attempts]}
            write_json(folder / "target_search.json", search)
    return {item["target"]: item for item in search.get("targets", [])}


def ms_prune(folder, row, item, path, source, baseline):
    row.update(target_status=item["status"], search_status="FAIL" if item["status"] == "FAIL" else "PASS")
    if item["status"] == "FAIL":
        row.update(error=item["error"], audit_status="NOT_RUN")
        return
    row["actual_gflops_speedup"] = item["speed_up"]
    try:
        core, proof, before, after = ms_audit(path, source, baseline, row["pruning_seed"], item)
        ms_commit_raw(folder, core, proof, source, before, after)
        row.update({name: proof[name] for name in (
            "identity_sha256", "state_sha256", "structure_sha256", "ledger_sha256", "plan_sha256")})
        row.update(audit_status="PASS", oracle_status=proof["oracle"]["status"], reproducibility_status="PASS", error=None)
    except Exception as error:
        row.update(audit_status="FAIL", error=safe_error(error))


def ms_raw_manifest(candidate, attempt, baseline, source, dataset, suite_id):
    target = attempt["target_speedup"]
    return {
        "format_version": 2, "stage": "raw_pruned", "model_variant": "s",
        "configuration_id": f"target-{target:.2f}x", "target_speedup": target,
        "experiment_id": attempt["configuration_id"], "suite_id": suite_id,
        "target_status": attempt["target_status"], "actual_gflops_speedup": attempt["actual_gflops_speedup"],
        "raw_checkpoint": {"relative_path": "pruned_fp32.pt", "sha256": candidate["saved_pruned"]["sha256"]},
        "baseline_checkpoint": {"relative_path": "../baseline/baseline_fp32.pt", "artifact_path": "baseline_fp32.pt",
                                "sha256": baseline["checkpoint"]["sha256"]},
        "state_sha256": candidate["state_sha256"], "structure_signature": candidate["structure_signature"],
        "source_baseline": source, "structural_status": candidate["structural_smoke_status"],
        "oracle_status": candidate["oracle"]["status"], "reload_status": candidate["reload_status"],
        "evaluation_mode": MODE, "dataset": {key: dataset[key] for key in (
            "dataset_id", "fingerprint", "full_val_fingerprint", "train_fingerprint", "provenance_status",
            "class_names", "image_count", "split", "seed")},
        "baseline": baseline["reserialized"], "raw_pruned": candidate["pruned"], "audit_files": candidate["audit_files"],
        "validation_status": candidate["validation_status"],
        "baseline_validation_parity_status": baseline["validation_parity_status"],
        "seed": attempt["pruning_seed"], "baseline_seed": source["baseline_seed"], "imgsz": IMGSZ, "platform": PLATFORM,
        "implementation_sha256": IMPLEMENTATION_SHA256,
    }


def ms_metrics(metrics, prefix=""):
    validation = metrics.get("validation", {})
    latency = metrics.get("latency", {})
    return {prefix + name: value for name, value in {
        "gflops": metrics.get("gflops"), "parameters": metrics.get("parameters"),
        "map50_95": validation.get("map50_95"), "map50": validation.get("map50"),
        "latency_p50_ms": latency.get("median_ms"), "latency_p95_ms": latency.get("p95_ms"),
    }.items()}


def ms_published_ref(reference, name):
    if workflow_version_ref(reference).rsplit(":", 1)[0] != f"{WANDB_ENTITY}/{WANDB_PROJECT}/{name}":
        raise ValueError("Referensi W&B tidak menunjuk ke koleksi perlakuan ini.")
    return reference


def ms_publish(folder, manifest):
    name = ms_run_name(manifest["target_speedup"], manifest["seed"])
    collection = f"{name}-raw"
    group = ms_run_group(manifest["suite_id"])
    checkpoint = folder / "pruned_fp32.pt"
    files = {"pruned_fp32.pt": checkpoint, "baseline_fp32.pt": folder.parent / "baseline/baseline_fp32.pt",
             "raw_pruned_manifest.json": folder / "raw_pruned_manifest.json",
             **{key: folder / key for key in manifest["audit_files"]}}
    hashes = {key: sha256_file(path) for key, path in files.items()}
    if (hashes["pruned_fp32.pt"] != manifest["raw_checkpoint"]["sha256"]
            or hashes["baseline_fp32.pt"] != manifest["baseline_checkpoint"]["sha256"]
            or any(hashes[key] != digest for key, digest in manifest["audit_files"].items())):
        raise ValueError("Hash paket artefak berbeda dari manifest raw.")
    journal = folder / "artifact_publications.json"
    record = ms_read(journal, {}).get("raw-pruned")
    if record and (record["files_sha256"] != hashes or record["name"] != collection or record["group"] != group):
        raise ValueError("Paket publikasi tersimpan berubah. Gunakan suite baru.")
    if record and record["status"] == "CONFIRMED":
        ms_published_ref(record["reference"], collection)
        return record
    existed = record is not None
    record = record or {"name": collection, "run_id": uuid.uuid4().hex, "group": group, "files_sha256": hashes}
    record["status"] = "PENDING"
    write_json(journal, {"raw-pruned": record})
    try:
        ms_wandb_environment()
        api = wandb.Api(timeout=30)
        raw_metrics = ms_metrics(manifest["raw_pruned"])
        baseline_metrics = ms_metrics(manifest["baseline"], "baseline_")
        meta = {"stage": "raw_pruned", "model_variant": "s", "platform": PLATFORM, "seed": manifest["seed"],
                "suite_id": manifest["suite_id"], "experiment_id": manifest["experiment_id"],
                "baseline_seed": manifest["baseline_seed"],
                "target_speedup": manifest["target_speedup"], "target_status": manifest["target_status"],
                "actual_gflops_speedup": manifest["actual_gflops_speedup"], **raw_metrics, **baseline_metrics,
                "raw_sha256": hashes["pruned_fp32.pt"], "baseline_fp32_sha256": hashes["baseline_fp32.pt"],
                "baseline_source_sha256": manifest["source_baseline"]["sha256"],
                "state_sha256": manifest["state_sha256"], "dataset_fingerprint": manifest["dataset"]["fingerprint"],
                "dataset_id": manifest["dataset"]["dataset_id"], "train_fingerprint": manifest["dataset"]["train_fingerprint"],
                "validation_image_count": manifest["dataset"]["image_count"], "imgsz": manifest["imgsz"],
                "gpu": ENVIRONMENT["gpu"], "baseline_artifact_ref": manifest["source_baseline"]["requested_artifact"],
                "files_sha256": hashes}
        if not record.get("reference"):
            # A wait timeout can occur after the server has committed the model. Recover that version by its run and hashes.
            if record.get("upload_started"):
                logged = api.run(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{record['run_id']}").logged_artifacts()
                for artifact in logged:
                    if (artifact.type == "model" and artifact.name.rsplit(":", 1)[0] == record["name"]
                            and artifact.metadata.get("files_sha256") == hashes):
                        record["reference"] = ms_published_ref(artifact.qualified_name, collection)
                        break
            if not record.get("reference"):
                with wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=name, id=record["run_id"], dir=str(folder),
                                resume="allow" if existed else "never", job_type="pruning-no-recovery",
                                group=group,
                                tags=["yolo26s", "lamp", "depgraph", "poltrada-v3", PLATFORM], config=meta) as run:
                    run.use_artifact(manifest["source_baseline"]["requested_artifact"], type="model")
                    run.summary.update({**raw_metrics, **baseline_metrics, "target_status": manifest["target_status"],
                                        "actual_gflops_speedup": manifest["actual_gflops_speedup"]})
                    artifact = wandb.Artifact(record["name"], type="model", metadata=meta)
                    for entry, path in files.items():
                        artifact.add_file(str(path), name=entry)
                    record["upload_started"] = True
                    write_json(journal, {"raw-pruned": record})
                    uploaded = run.log_artifact(artifact).wait(timeout=300)
                    record["reference"] = ms_published_ref(uploaded.qualified_name, collection)
                    write_json(journal, {"raw-pruned": record})
            write_json(journal, {"raw-pruned": record})
        reference = ms_published_ref(record["reference"], collection)
        remote = api.artifact(reference, type="model")
        if remote.qualified_name != reference:
            raise ValueError("Versi W&B yang diunduh berbeda dari versi upload.")
        with TemporaryDirectory(prefix="wandb_check_", dir=folder) as verify_dir:
            for entry, digest in hashes.items():
                downloaded = remote.get_path(entry).download(root=verify_dir)
                if sha256_file(downloaded) != digest:
                    raise ValueError(f"SHA-256 unduhan berbeda dari paket raw lokal: {entry}.")
        record.update(status="CONFIRMED", artifact_url=remote.url, error=None)
    except Exception as error:
        record.update(status="PENDING", error=safe_error(error))
    write_json(journal, {"raw-pruned": record})
    return record


def ms_complete_raw(folder, candidate, attempt, baseline, source, dataset, suite_id):
    journal = folder / "pruning_manifest.json"
    stage = "integrity"
    try:
        if (candidate["identity_sha256"] != attempt["identity_sha256"] or
                candidate["provenance"] != source or source["baseline_seed"] != attempt["pruning_seed"]):
            raise ValueError("Jurnal raw dan baseline tidak cocok dengan perlakuan.")
        ms_verify_files(folder, candidate)
        candidate["integrity_status"] = "PASS"
        stage = "validation"
        if candidate["validation_status"] != "PASS":
            seed_environment(EVAL_SEED)
            gflops_validation(candidate, folder / "pruned_fp32.pt", dataset, folder)
            if "validation" in candidate:
                candidate["pruned"]["validation"] = candidate.pop("validation")
            write_json(journal, candidate)
        stage = "latency"
        latency_status = candidate["pruned"].get("latency", {}).get("status")
        if latency_status != "PASS" and (latency_status != "NOT_RUN" or TARGET_DEVICE.type == "cuda"):
            core = YOLO(str(folder / "pruned_fp32.pt"), task="detect").model.to(TARGET_DEVICE).float().eval()
            core.end2end = True
            candidate["pruned"]["latency"] = gflops_latency(core, ms_probe(EVAL_SEED))
            del core
            write_json(journal, candidate)
        candidate["baseline"] = baseline["reserialized"]
        candidate["comparison"] = metric_comparison(candidate["baseline"], candidate["pruned"])
        stage = "eda"
        eda_key = workflow_digest({"baseline": candidate["baseline"], "pruned": candidate["pruned"]})
        if candidate["eda_status"] != "PASS" or candidate.get("eda_input_sha256") != eda_key:
            inventory = ms_read(folder / "inventories.json")
            candidate["eda"] = build_run_eda(candidate, folder, inventory["before"], inventory["after"])
            candidate["eda_status"] = "PASS" if candidate["eda"]["status"] == "MEASURED" else candidate["eda"]["status"]
            candidate["eda_input_sha256"] = eda_key
            write_json(journal, candidate)
    except Exception as error:
        candidate[stage + "_status"] = "FAIL"
        candidate["errors"].append({"stage": stage, "error": safe_error(error)})

    if candidate.get("integrity_status") == "PASS":
        manifest_path = folder / "raw_pruned_manifest.json"
        existing = ms_read(folder / "artifact_publications.json", {}).get("raw-pruned")
        # Keep package bytes fixed after the first publication attempt, even on a relocated suite.
        manifest = (ms_read(manifest_path) if existing else
                    ms_raw_manifest(candidate, attempt, baseline, source, dataset, suite_id))
        if not existing:
            write_json(manifest_path, manifest)
        if (MODE == "full_val" and candidate["validation_status"] == "PASS"
                and candidate["pruned"].get("latency", {}).get("status") == "PASS"
                and baseline["validation_parity_status"] == "PASS" and baseline["reserialized"]["latency"]["status"] == "PASS"):
            try:
                candidate["publication"] = ms_publish(folder, manifest)
            except Exception as error:
                candidate["publication"] = {"status": "FAIL", "error": safe_error(error)}
    write_json(journal, candidate)


def ms_report(folder, attempts, dataset, seeds, suite_id):
    rows, models = [], {}
    for attempt in attempts:
        seed = attempt["pruning_seed"]
        identity = BASELINE_SOURCES[seed]
        state = seeds.get(seed, {})
        baseline = state.get("baseline", {})
        source = state.get("source", {})
        baseline_parity = baseline.get("validation_parity_status", "NOT_RUN")
        baseline_metrics = baseline.get("reserialized", {})
        model_name = ms_model_name(seed)
        model_dir = folder / model_name.lower()
        raw_path = model_dir / attempt["configuration_id"] / "pruned_fp32.pt"
        audit_passed = attempt["audit_status"] == "PASS"
        candidate = ms_read(raw_path.with_name("pruning_manifest.json"), {}) if audit_passed else {}
        metrics = candidate.get("pruned", {})
        validation = metrics.get("validation", {}) if candidate.get("validation_status") == "PASS" else {}
        latency = metrics.get("latency", {})
        publication = candidate.get("publication", {})
        confirmed = publication.get("status") == "CONFIRMED"
        usable = (candidate.get("integrity_status") == "PASS" and candidate.get("reload_status") == "PASS"
                  and source == candidate.get("provenance") and bool(baseline))
        row = {**attempt, "model": model_name, "model_variant": "s", "evaluation_mode": MODE, "suite_id": suite_id,
               "actual_gflops_speedup": attempt.get("actual_gflops_speedup"),
               "dataset_id": dataset["dataset_id"], "dataset_fingerprint": dataset["fingerprint"],
               "train_fingerprint": dataset["train_fingerprint"],
               "dataset_provenance_status": dataset["provenance_status"], "baseline_seed": seed,
               "source_baseline_artifact_ref": identity["ref"], "source_baseline_sha256": identity["sha256"],
               "baseline_fp32_sha256": baseline.get("checkpoint", {}).get("sha256"),
               "source_status": state.get("source_status", "NOT_RUN"),
               "baseline_validation_parity_status": baseline_parity, "baseline_error": state.get("error"),
               **ms_metrics(baseline_metrics, "baseline_"),
               "checkpoint_relative_path": raw_path.relative_to(folder).as_posix() if candidate else None,
               "raw_sha256": candidate.get("saved_pruned", {}).get("sha256"),
               "artifact_ref": publication.get("reference") if confirmed else None,
               "artifact_url": publication.get("artifact_url") if confirmed else None,
               "wandb_run_id": publication.get("run_id"), "wandb_group": ms_run_group(suite_id),
               "publication_status": publication.get("status", "NOT_RUN"), "publication_error": publication.get("error"),
               "structural_status": candidate.get("structural_smoke_status", attempt["audit_status"]),
               "integrity_status": candidate.get("integrity_status", "FAIL" if audit_passed and not candidate else "NOT_RUN"),
               "reload_status": candidate.get("reload_status", "NOT_RUN"), "validation_status": candidate.get("validation_status", "NOT_RUN"),
               "eda_status": candidate.get("eda_status", "NOT_RUN"), "latency_status": latency.get("status", "NOT_RUN"),
               **ms_metrics(metrics),
               "validation_image_count": validation.get("validation_image_count"),
               "delta_map50_95_pp": candidate.get("comparison", {}).get("delta_map50_95_pp"),
               "recovery_ready": bool(usable and confirmed and validation and baseline_parity == "PASS" and MODE == "full_val"),
        }
        rows.append(row)
        if seed not in models:
            models[seed] = {"model_name": model_name, "model_variant": "s", "baseline_seed": seed,
                            "baseline_report": baseline, "source": source, "baseline": baseline_metrics,
                            "results": [], "rows": [], "target_search": ms_read(model_dir / "search/target_search.json", {})}
        models[seed]["rows"].append(row)
        if usable:
            models[seed]["results"].append({**candidate, "configuration_id": row["configuration_id"],
                                            "target_speedup": row["target_speedup"], "target_status": row["target_status"],
                                            "baseline": baseline_metrics, "status": "PASS" if validation else "PARTIAL"})
    counts = {"pruned_checkpoint_count": sum(bool(row["raw_sha256"]) for row in rows),
              "confirmed_artifact_count": sum(row["publication_status"] == "CONFIRMED" for row in rows)}
    registry = {"format_version": 2, "workflow": "lamp-yolo26s-multiseed", "mode": MODE, "suite_id": suite_id,
                **counts, "rows": rows}
    write_json(folder / "checkpoint_registry.json", registry)
    write_csv(folder / "checkpoint_registry.csv", rows)
    status = workflow_status(("PASS" if row["validation_status"] == row["eda_status"] == "PASS"
                               and row["integrity_status"] == row["baseline_validation_parity_status"] == "PASS"
                               and (MODE == "smoke" or row["recovery_ready"])
                               else "PARTIAL" if row["audit_status"] == "PASS" else
                               "FAIL" if row["baseline_error"] or row.get("error") else "NOT_RUN"
                               for row in rows), useful=any(model["baseline"] for model in models.values()))
    bundle = {"mode": MODE, "status": status, "registry": registry, "models": list(models.values())}
    write_json(folder / "suite_manifest.json", {"mode": MODE, "status": status, "suite_id": suite_id,
                                               "registry": "checkpoint_registry.json", **counts})
    return bundle


def ms_execute(folder):
    with FileLock(str(folder / ".suite.lock"), timeout=0):
        persist = None
        try:
            if MODE not in {"full_val", "smoke"}:
                raise ValueError("Pilih full_val atau smoke.")
            if not GLOBAL_PRUNING or any((RECOVERY_ENABLED, FINETUNING_ENABLED, KD_ENABLED)):
                raise ValueError("Workflow mengunci global pruning dan menonaktifkan training, recovery, serta KD.")
            if MODE == "full_val" and not DATASET_MANIFEST_ONLY and (
                    TARGET_DEVICE.type != "cuda" or (IMGSZ, BATCH) != (640, 16) or not RUN_LATENCY_BENCHMARK):
                raise ValueError("Full validation penelitian memerlukan CUDA, latency, 640 px, dan batch 16.")
            dataset, data_record = ms_dataset(folder)
            if DATASET_MANIFEST_ONLY:
                if (folder / "resume_contract.json").exists():
                    raise ValueError("Gunakan suite baru untuk ekspor manifest dataset.")
                write_json(folder / "dataset_manifest.json", data_record)
                return {"mode": MODE, "status": "NOT_RUN", "reason": "Manifest dataset tersedia.", "models": []}
            suite_id = ms_bind(folder, dataset)["contract"]["suite_id"]
            attempts = ms_attempts(folder)
            seeds = {}

            def persist():
                return ms_report(folder, attempts, dataset, seeds, suite_id)

            write_json(folder / "dataset_manifest.json", data_record)
            write_json(folder / "environment.json", ENVIRONMENT)
            persist()
            for seed in dict.fromkeys(row["pruning_seed"] for row in attempts):
                jobs = [row for row in attempts if row["pruning_seed"] == seed]
                model_dir = folder / ms_model_name(seed).lower()
                state = seeds[seed] = {}
                try:
                    path, source = ms_source(model_dir, seed)
                    state.update(source=source, source_status="PASS")
                    baseline = ms_baseline(model_dir / "baseline", path, source, dataset)
                    state["baseline"] = baseline
                except Exception as error:
                    state.setdefault("source_status", "FAIL")
                    state["error"] = safe_error(error)
                    print(f"Baseline seed {seed}: {state['error']}", flush=True)
                    persist()
                    continue
                persist()
                targets = ms_find_targets(model_dir / "search", path, source, baseline, seed, jobs)
                for row in jobs:
                    config = row["configuration_id"]
                    raw_dir = model_dir / config
                    if row["audit_status"] != "PASS":
                        ms_prune(raw_dir, row, targets[row["target_speedup"]], path, source, baseline)
                        write_json(folder / "attempts" / f"{config}.json", row)
                        persist()
                    candidate = ms_read(raw_dir / "pruning_manifest.json") if row["audit_status"] == "PASS" else None
                    if candidate:
                        ms_complete_raw(raw_dir, candidate, row, baseline, source, dataset, suite_id)
                    persist()
                    print(config, row["target_status"], row["audit_status"],
                          (candidate or {}).get("validation_status", "NOT_RUN"),
                          (candidate or {}).get("publication", {}).get("status", "NOT_RUN"), flush=True)
                    gc.collect()
                    if TARGET_DEVICE.type == "cuda":
                        torch.cuda.empty_cache()
            (folder / "last_error.json").unlink(missing_ok=True)
            return persist()
        except (Exception, KeyboardInterrupt) as error:
            failure = {"status": "PARTIAL" if persist else "NOT_RUN", "error": safe_error(error) or "Interrupted"}
            write_json(folder / "last_error.json", failure)
            if persist:
                bundle = persist()
                return {**bundle, **failure}
            if (folder / "resume_contract.json").exists():
                raise
            return {**failure, "mode": MODE, "models": []}

In [ ]:
BUNDLE = ms_execute(SUITE_DIR)
print(BUNDLE["status"], BUNDLE.get("error") or BUNDLE.get("reason", ""))

## 11 — EDA Identitas, Cakupan, dan Channel

Ringkasan memuat ketujuh percobaan dengan pasangan baseline dan checkpoint raw masing-masing.
Detail dan grafik tersedia untuk setiap perlakuan. `EDA_CONFIGURATIONS` mengatur tampilan overview
dan `EDA_DETAIL_CONFIGURATIONS` mengatur detail. Isi `()` untuk melihat seluruh hasil.
Sel 11–14 bisa dijalankan ulang dari `BUNDLE` untuk menampilkan hasil tersimpan.

### 11.1 Identitas eksperimen dan cakupan arsitektur

Periksa checkpoint sumber dan hash, jumlah citra, fingerprint, serta mode evaluasi. Tabel cakupan
membedakan group eligible dari graph, kapasitas policy, channel independen, dan perubahan fisik
output Conv. Registry mengikat seed, versi baseline, dan status dari tiap perlakuan.

In [ ]:
from IPython.display import Image as NotebookImage, Markdown, display

REGISTRY = BUNDLE.get("registry", {})
REGISTRY_ROWS = REGISTRY.get("rows", [])
if REGISTRY_ROWS:
    try:
        with FileLock(str(SUITE_DIR / ".suite.lock"), timeout=0):
            ANALYSIS = gflops_build_analysis(BUNDLE, SUITE_DIR)
    except Exception as error:
        print("EDA agregat PARTIAL", safe_error(error))
        ANALYSIS = {"status": "PARTIAL", "plots": [], "tables": gflops_eda_tables(BUNDLE, SUITE_DIR)}
else:
    print("Status suite", BUNDLE["status"], BUNDLE.get("error") or BUNDLE.get("reason", ""))
    ANALYSIS = {"status": "NOT_RUN", "plots": [], "tables": gflops_eda_tables(BUNDLE, SUITE_DIR)}
TABLES = ANALYSIS["tables"]


def show_eda_table(rows, columns=None):
    frame = pd.DataFrame(rows)
    if frame.empty:
        print("UNAVAILABLE — lihat status pengukuran pada tabel ringkasan.")
        return
    if columns:
        frame = frame.reindex(columns=columns)
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 100):
        display(frame)


def show_eda_images(folder, *patterns):
    for pattern in patterns:
        for path in sorted(folder.glob(pattern)):
            display(NotebookImage(filename=str(path)))

show_eda_table(TABLES["sources"], ["model", "baseline_seed", "requested_artifact", "sha256", "source_verification",
                                  "baseline_validation_parity_status"])
show_eda_table(REGISTRY_ROWS, ["configuration_id", "baseline_seed", "source_baseline_artifact_ref", "source_status",
                             "target_speedup", "target_status", "actual_gflops_speedup", "structural_status",
                             "reload_status", "validation_status", "baseline_map50_95", "map50_95",
                             "delta_map50_95_pp", "eda_status", "evaluation_mode",
                             "validation_image_count", "dataset_provenance_status", "dataset_fingerprint"])
show_eda_table(TABLES["architecture"], ["model", "index", "type", "from", "region", "parameters"])
show_eda_table(TABLES["coverage"])

### 11.2 Perbandingan channel sebelum dan sesudah pruning

Diagram **Before Pruning** biru, **After Pruning** merah, dan **Channel Comparison** overlay memakai
urutan Conv dan batas sumbu yang sama, termasuk layer dengan channel tetap. Detail membandingkan
channel input dan output, parameter, serta estimasi operasi setiap layer sebelum dan sesudah
pruning. Satu penghapusan channel independen dapat memengaruhi beberapa output Conv yang terikat
oleh residual.

In [ ]:
EDA_RUNS = []
for model in BUNDLE["models"]:
    model_name = model["model_name"]
    available = {result["configuration_id"]: result for result in model["results"]
                 if result["eda_status"] == "PASS"}
    if not available:
        print(f"{model_name} — EDA belum tersedia. Lihat status tiap konfigurasi pada tabel.")
    for key, result in available.items():
        if EDA_CONFIGURATIONS and key not in EDA_CONFIGURATIONS:
            continue
        folder = SUITE_DIR / model_name.lower() / result.get("artifact_configuration_id", key) / "eda"
        EDA_RUNS.append((model_name, key, result, folder))
        display(Markdown(f"#### {model_name} · {key}"))
        show_eda_images(folder, "channels_overview.png", "channel_reduction_percent.png")
EDA_DETAILS = [entry for entry in EDA_RUNS if not EDA_DETAIL_CONFIGURATIONS or entry[1] in EDA_DETAIL_CONFIGURATIONS]
for model_name, key, result, folder in EDA_DETAILS:
    display(Markdown(f"#### Detail layer · {model_name} · {key}"))
    selected = [row for row in TABLES["layers"] if row["model"] == model_name and row["configuration_id"] == key]
    show_eda_table(selected, ["layer_name", "module_type", "region", "in_channels_before", "in_channels_after",
        "out_channels_before", "out_channels_after", "out_channels_removed", "out_channels_reduction_pct",
        "parameters_before", "parameters_after", "estimated_flops_before", "estimated_flops_after",
        "dependency_group", "protection_reason"])
    show_eda_images(folder, "channels_detail_*.png")

### 11.3 Keputusan LAMP, kapasitas group, dan ledger indeks asli

Lihat group yang diterima atau ditolak, rentang skor finite, kapasitas, dan jumlah channel
yang dipilih. Skor `+inf` dari pembatas kapasitas dikeluarkan dari distribusi importance.
Heatmap memakai indeks asli sebelum pruning. Ledger mencatat kolom input Conv konsumen dan
BatchNorm, termasuk offset concat dan seluruh konsumen fan-out. Ekspor per perlakuan
tersedia di `comparison/analysis/groups.csv` dan `comparison/analysis/ledger.csv`.

In [ ]:
for model_name, key, result, folder in EDA_DETAILS:
    display(Markdown(f"#### LAMP dan dependency · {model_name} · {key}"))
    selected = [row for row in TABLES["groups"] if row["model"] == model_name and row["configuration_id"] == key]
    show_eda_table(selected, ["root", "group_id", "decision", "initial_channels", "capacity_eligible",
        "removed", "score_min", "score_median", "score_max", "coupled_output_layers", "reason"])
    changed = [row for row in TABLES["ledger"] if row["model"] == model_name and row["configuration_id"] == key
               and row.get("removed_original_indices")]
    show_eda_table(changed, ["layer", "module_type", "dimension", "channels_before", "channels_after",
                            "kept_original_indices", "removed_original_indices", "dependency_group_ids"])
    show_eda_images(folder, "channel_mask_*.png", "lamp_score_distribution.png", "lamp_group_capacity.png")

## 12 — EDA Efisiensi dan Metrik Deteksi

### 12.1 Efisiensi per model, layer, dan stage

Bandingkan parameter, GFLOPs, dan ukuran checkpoint setiap kandidat dengan baseline seed yang
sama yang disimpan ulang sebagai FP32. Ukuran file sumber ditampilkan terpisah karena formatnya dapat berbeda.
Estimasi per layer dan stage mencakup Conv/BN, sedangkan total GFLOPs juga memasukkan attention.
Latency p50 dan p95 memakai raw dual-head forward FP32 batch 1 di luar preprocessing, NMS,
dan I/O. Speed-up latency membagi median baseline dengan median pruned. Jika CUDA belum diukur,
status latency tetap `NOT_RUN`.

In [ ]:
show_eda_table(TABLES["metrics"], [
    "model", "configuration_id", "stage", "parameters", "parameter_reduction_pct",
    "gflops", "gflops_reduction_pct", "gflops_speedup", "checkpoint_bytes", "checkpoint_reduction_pct",
    "latency_median_ms", "latency_p95_ms", "latency_speedup", "latency_status",
])
for model_name, key, result, folder in EDA_DETAILS:
    display(Markdown(f"#### Efisiensi stage · {model_name} · {key}"))
    selected = [row for row in TABLES["stages"] if row["model"] == model_name and row["configuration_id"] == key]
    show_eda_table(selected)
    show_eda_images(folder, "stage_physical_conv_outputs.png", "stage_parameters.png", "stage_estimated_flops.png")

### 12.2 Akurasi keseluruhan dan AP/P/R/F1 per kelas untuk seluruh konfigurasi

Delta `mAP50–95` dihitung terhadap baseline dan dilaporkan dalam poin persentase. Perubahan
relatif memakai mAP baseline sebagai denominator dan kosong jika baseline nol. Delta AP per kelas
memakai ID kelas yang sama. Support menghitung jumlah objek ground-truth, dan kelas tanpa metrik
tetap kosong. Tabel dan grafik memakai hasil setiap perlakuan. Ringkasan lintas seed pada 13.3
merangkum delta berpasangan untuk target utama. Angka `smoke` berasal dari subset untuk pemeriksaan alur.

In [ ]:
show_eda_table(TABLES["metrics"], [
    "model", "configuration_id", "stage", "map50_95", "map50", "precision", "recall",
    "delta_map50_95_pp", "relative_map50_95_pct", "validation_status", "evaluation_mode", "validation_image_count",
])
PER_CLASS = pd.DataFrame(TABLES["per_class_metrics"])
for model in BUNDLE["models"]:
    model_name = model["model_name"]
    display(Markdown(f"#### Semua konfigurasi · {model_name}"))
    selected = [row for row in TABLES["per_class_metrics"] if row["model"] == model_name]
    show_eda_table(selected, ["configuration_id", "stage", "class_id", "name", "support", "ap50_95", "ap50",
                            "precision", "recall", "f1", "delta_ap50_95_pp", "delta_ap50_pp"])
    show_eda_images(SUITE_DIR / "comparison/analysis" / model_name.lower(),
                    "configuration_metrics.png", "per_class_ap_all.png")

### 12.3 Kurva PR/F1/confidence, confusion matrix, dan protokol efektif

Kurva dan confusion matrix berasal dari validator aktual. Tampilkan panel baseline sekali per
seed, lalu panel tiap checkpoint raw. Precision, recall, dan F1 per kelas memakai
titik confidence terbaik menurut smoothed mean F1 SDK pada IoU 0,50. Confusion matrix memakai
**baris predicted dan kolom true**, termasuk background. Confidence dan IoU pencocokan confusion
matrix dilaporkan pada metadata panel. Tabel protokol mencatat split, jumlah citra, fingerprint,
presisi, serta pengaturan one-to-many/NMS yang digunakan.

In [ ]:
show_eda_table(TABLES["validation_protocol"], [
    "model", "configuration_id", "validation_status", "evaluation_mode", "split", "validation_image_count",
    "dataset_fingerprint", "dataset_provenance_status", "imgsz", "batch", "quantize", "conf", "iou", "max_det",
    "nms", "end2end", "augment", "rect", "device",
])
baseline_curves_shown = set()
for model_name, key, result, folder in EDA_DETAILS:
    for stage in ("baseline", "pruned"):
        if stage == "baseline" and model_name in baseline_curves_shown:
            continue
        label = "Baseline" if stage == "baseline" else key
        display(Markdown(f"#### Kurva dan confusion matrix · {model_name} · {label}"))
        metrics = result[stage].get("validation") or {}
        validation_status = result["validation_status"] if stage == "pruned" else result[stage].get("validation_status", "NOT_RUN")
        if stage == "baseline":
            baseline_curves_shown.add(model_name)
        if validation_status != "PASS" or not metrics:
            print(validation_status, "— metrik deteksi belum tersedia.")
            continue
        display(metrics.get("metric_semantics", {}))
        matrix = metrics.get("confusion_matrix", {})
        display({field: matrix.get(field) for field in (
            "status", "labels", "confidence_threshold", "matching_iou_threshold", "validation_iou")})
        show_eda_images(folder / stage, "class_metrics.png", "evaluation_curve_*.png", "confusion_matrix.png")

## 13 — Analisis Trade-off Akurasi dan Efisiensi

### 13.1 Bukti awal extended dari tabel historis pengguna

Enam baris YOLO26s dari tabel historis pengguna disajikan dengan provenance `UNVERIFIED`.
Pada extended 10%, mAP50–95 yang dilaporkan turun dari 0,5612 ke 0,2546. Pada extended 20%,
nilainya di bawah 0,003. Status di tabel berasal dari catatan awal. Artefak run, fingerprint,
protokol evaluator, dan counter pada catatan tersebut masih perlu diverifikasi. Data historis
disimpan terpisah dari grafik hasil suite saat ini.

In [ ]:
from io import StringIO
HISTORICAL_EXTENDED_CSV = 'index,model,model_variant,policy,ratio,parameters,gflops,checkpoint_bytes,map50_95,map50,precision,recall,parameter_reduction_pct,gflops_reduction_pct,delta_map50_95_pp,relative_map50_95_pct,actual_eligible_ratio,pruned_group_count,latency_median_ms,structural_status,validation_status,provenance\n0,YOLO26n,n,baseline,0.0,2506140,6.025830,10516163,5.261053e-01,0.742727,0.769929,0.690453,0.000000,0.000000,0.000000,0.000000,0.000000,0,13.690454,BASELINE,PASS,UNVERIFIED\n1,YOLO26n,n,extended,0.1,2236276,5.691846,9433063,1.692238e-01,0.286811,0.641445,0.269618,10.768114,5.542539,-35.688155,-67.834620,0.100195,7,13.545400,PASS,PASS,UNVERIFIED\n2,YOLO26n,n,extended,0.2,2013751,5.075542,8541159,1.987566e-03,0.006276,0.002626,0.066285,19.647306,15.770241,-52.411776,-99.622211,0.200389,10,13.722865,PASS,PASS,UNVERIFIED\n3,YOLO26n,n,extended,0.3,1800137,4.532680,7684519,8.236284e-04,0.003035,0.001787,0.051145,28.170932,24.779164,-52.528170,-99.843448,0.300097,16,14.463144,PASS,PASS,UNVERIFIED\n4,YOLO26n,n,extended,0.4,1625418,4.083136,6982951,3.451889e-07,0.000001,0.000131,0.000861,35.142570,32.239447,-52.610498,-99.999934,0.400292,27,13.931716,PASS,PASS,UNVERIFIED\n5,YOLO26n,n,extended,0.5,1536092,3.494195,6623847,0.000000e+00,0.000000,0.000000,0.000000,38.706856,42.013051,-52.610533,-100.000000,0.500000,35,13.696097,PASS,PASS,UNVERIFIED\n6,YOLO26s,s,baseline,0.0,9952508,23.006822,40378115,5.611909e-01,0.773451,0.775606,0.721401,0.000000,0.000000,0.000000,0.000000,0.000000,0,14.290742,BASELINE,PASS,UNVERIFIED\n7,YOLO26s,s,extended,0.1,8858156,21.711862,35995303,2.545885e-01,0.412033,0.674990,0.311926,10.995741,5.628591,-30.660239,-54.634238,0.100195,7,13.983301,PASS,PASS,UNVERIFIED\n8,YOLO26s,s,extended,0.2,7936472,19.764531,32304935,2.826255e-03,0.006314,0.164648,0.013613,20.256562,14.092738,-55.836468,-99.496383,0.200146,9,14.155988,PASS,PASS,UNVERIFIED\n9,YOLO26s,s,extended,0.3,7075142,17.591168,28855527,6.153192e-07,0.000002,0.000198,0.000842,28.910964,23.539341,-56.119032,-99.999890,0.300097,16,14.166117,PASS,PASS,UNVERIFIED\n10,YOLO26s,s,extended,0.4,6426986,15.373741,26256487,0.000000e+00,0.000000,0.000000,0.000000,35.423453,33.177470,-56.119094,-100.000000,0.400049,29,14.516314,PASS,PASS,UNVERIFIED\n11,YOLO26s,s,extended,0.5,6077564,13.034906,24857383,0.000000e+00,0.000000,0.000000,0.000000,38.934347,43.343303,-56.119094,-100.000000,0.500000,35,14.411619,PASS,PASS,UNVERIFIED\n'

historical = pd.read_csv(StringIO(HISTORICAL_EXTENDED_CSV))
historical_rows = historical.loc[historical["model_variant"].eq("s")].to_dict("records")
show_eda_table(historical_rows)
if ANALYSIS["status"] == "PASS":
    with FileLock(str(SUITE_DIR / ".suite.lock"), timeout=0):
        write_csv(SUITE_DIR / "comparison/analysis/historical_unverified.csv", historical_rows)

### 13.2 Kontrol extended dan pencarian target komputasi

Kontrol extended berstatus `NOT_RUN`. Tabel tier dan grafik kapasitas menampilkan profil pencarian
setiap seed, termasuk tiga target untuk seed 42. Profil tersimpan di `yolo26s-seed<seed>/search/`.
Pencarian mengukur GFLOPs. Nilai mAP berasal dari validasi kandidat sesudah audit dan reload.

In [ ]:
show_eda_table(TABLES["target_tiers"])
show_eda_table(TABLES["target_search"], ["model", "target", "status", "tier", "channels", "gflops", "speed_up"])
for model in BUNDLE["models"]:
    model_name = model["model_name"]
    display(Markdown(f"#### Sweep dan kapasitas · {model_name}"))
    show_eda_images(SUITE_DIR / "comparison/analysis" / model_name.lower(),
                    "target_search_capacity.png")

### 13.3 Trade-off seluruh konfigurasi model ini

Setiap grafik membandingkan kandidat dengan baseline seed yang sama. Bandingkan mAP50–95 terhadap
parameter, GFLOPs, dan median latency CUDA sesuai status pengukuran masing-masing. Ringkasan target
1,20× mencatat capaian tiap seed dan mean serta simpangan baku sampel dari delta berpasangan
yang tersedia. Hitung dan baca hasil parsial sesuai jumlah pasangan dan status pencapaian target.

In [ ]:
display(Markdown("#### Ringkasan lintas seed · target 1,20×"))
primary_rows = [row for row in REGISTRY_ROWS if row["target_speedup"] == PRIMARY_TARGET]
show_eda_table(primary_rows, ["baseline_seed", "target_status", "actual_gflops_speedup",
                              "baseline_map50_95", "map50_95", "delta_map50_95_pp",
                              "baseline_latency_p50_ms", "latency_p50_ms", "validation_status"])
deltas = pd.Series([row["delta_map50_95_pp"] for row in primary_rows if row["delta_map50_95_pp"] is not None
                    and row["validation_status"] == row["baseline_validation_parity_status"] == "PASS"], dtype=float)
if not deltas.empty:
    display({"mode": BUNDLE["mode"], "pasangan_terukur": int(deltas.count()), "pasangan_direncanakan": len(primary_rows),
             "target_tercapai": sum(row["target_status"] == "REACHED" for row in primary_rows),
             "rerata_delta_mAP50_95_pp": deltas.mean(), "simpangan_baku_sampel_pp": deltas.std() if len(deltas) > 1 else None})
show_eda_table([plot for plot in ANALYSIS["plots"] if plot["plot"] != "extended_sweep"],
               ["model", "plot", "status", "measured_values", "reason"])
for model in BUNDLE["models"]:
    display(Markdown(f"#### Trade-off · {model['model_name']}"))
    show_eda_images(SUITE_DIR / "comparison/analysis" / model["model_name"].lower(), "tradeoff_*.png")

### 13.4 Kontras sensitivitas P5 dan kelayakan student

Dalam audit 06, **P5-only vs baseline** memeriksa penghapusan channel P5 pilihan extended 10%.
**P5-protected vs extended 10%** memeriksa alokasi ulang pada jumlah channel independen yang sama.
Kontras itu membaca delta mAP dan AP per kelas bersama perubahan GFLOPs. Distribusi channel
sendiri memberi konteks tentang lokasi pruning. Keduanya berstatus `NOT_RUN` pada suite ini.

Garis baseline −2 poin persentase pada grafik metrik merujuk pada kriteria akurasi di notebook
06 final. Kolom `recovery_ready` pada suite ini menandai kelengkapan audit, full validation,
dan publikasi paket. Gunakan tabel akurasi untuk menilai hasil deteksi. Baris `TARGET_NOT_REACHED`
menunjukkan kandidat maksimum dan membutuhkan pilihan diagnostik eksplisit saat checkpoint terkonfirmasi.

In [ ]:
print("P5-only", "NOT_RUN", "| P5-protected", "NOT_RUN")
show_eda_table(REGISTRY_ROWS, ["target_speedup", "actual_gflops_speedup", "target_status", "recovery_seed",
                             "source_baseline_artifact_ref", "structural_status", "reload_status",
                             "validation_status", "publication_status", "recovery_ready"])

## 14 — Ringkasan, Artefak, dan Reproducibility

Setiap perlakuan punya run dan Model Artifact sendiri dalam satu grup W&B. Nama mengikuti pola
`yolo26s-lamp-su1.20x-seed0-raw`. Metadata menyimpan versi dan SHA-256 baseline, target, capaian
aktual, seed, GFLOPs, parameter, mAP, latency, SHA-256 raw, dan fingerprint dataset. Ringkasan
mencatat reproducibility replay tiap perlakuan. Perbandingan lintas seed memakai lima baseline
berbeda.

`CONFIRMED` berarti semua SHA-256 paket cocok setelah diunduh dari versi tetap `:vN`. Suite baru
selesai saat tujuh baris registry memiliki hasil evaluasi dan referensi W&B terverifikasi.
Pakai versi `:vN` dari registry suite yang dijalankan. Versi `:v0` seed 42 dari desain sebelumnya
tetap tercatat sebagai riwayat W&B.

Untuk melanjutkan, isi `RESUME_SUITE_DIR` dengan folder suite tersimpan. Gunakan `OUTPUT_DIR`
persisten di Filkom. Di Kaggle, salin suite dari `/kaggle/working` ke lokasi yang dapat ditulis
sebelum menjalankan ulang notebook di sesi baru. Kontrak resume mengunci kelima versi dan SHA-256
baseline, fingerprint dataset, implementasi, dan runtime. Pencarian, validasi, EDA, dan progres
publikasi tersimpan per tahap. Tabel CSV/JSON dan grafik PNG/SVG tersedia di `comparison/analysis/`
dan folder `eda/` tiap perlakuan.

Salin sel pemuat registry di bawah ke notebook recovery sesudah konfigurasi dan sebelum bootstrap.
Isi path JSON dan seed recovery. Target default `1.20` memilih eksperimen utama. Ambil versi raw
dan identitas baseline dari baris yang sama. Pasang lock baseline sesudah bootstrap recovery.
Untuk kandidat `TARGET_NOT_REACHED` yang sudah terkonfirmasi, aktifkan `allow_diagnostic=True`.

In [ ]:
from IPython.display import FileLink

print("Folder suite", SUITE_DIR)
print("Status suite", BUNDLE["status"], "| Status EDA", ANALYSIS["status"])
print("Suite", REGISTRY.get("suite_id", "NOT_RUN"), "| Checkpoint raw", REGISTRY.get("pruned_checkpoint_count", 0),
      "| Artefak terverifikasi", REGISTRY.get("confirmed_artifact_count", 0), "/", len(PRUNING_QUEUE))
for filename in ("suite_manifest.json", "resume_contract.json", "dataset_manifest.json", "environment.json",
                 "checkpoint_registry.csv", "checkpoint_registry.json", "comparison/analysis/analysis_status.json",
                 "comparison/analysis/metrics.csv", "comparison/analysis/per_class_metrics.csv"):
    path = SUITE_DIR / filename
    if path.is_file():
        display(FileLink(Path(os.path.relpath(path)).as_posix()))
show_eda_table(TABLES["verification"])
show_eda_table(REGISTRY_ROWS, ["configuration_id", "pruning_seed", "recovery_seed", "reproducibility_status",
                             "target_status", "checkpoint_relative_path", "raw_sha256", "integrity_status",
                             "source_baseline_artifact_ref", "source_baseline_sha256", "baseline_fp32_sha256",
                             "publication_status", "artifact_ref", "wandb_run_id", "publication_error"])

In [ ]:
def recovery_config_from_registry(path, seed, target=1.20, *, allow_diagnostic=False, include_baseline=False):
    import json
    from pathlib import Path
    import re

    registry = json.loads(Path(path).expanduser().read_text(encoding="utf-8"))
    version = registry.get("format_version")
    if (version not in {1, 2} or registry.get("workflow") != "lamp-yolo26s-multiseed"
            or registry.get("mode") != "full_val"):
        raise ValueError("Gunakan checkpoint_registry.json dari full validation LAMP YOLO26s.")
    matches = [row for row in registry["rows"] if row.get("target_speedup") == float(target)
               and row.get("recovery_seed") == seed]
    if len(matches) != 1:
        raise ValueError("Target dan seed recovery harus memilih tepat satu baris.")
    row = matches[0]
    if row.get("target_status") != "REACHED" and not (
            allow_diagnostic and row.get("target_status") == "TARGET_NOT_REACHED"):
        raise ValueError("Target belum tercapai. Pilih allow_diagnostic=True untuk kandidat maksimum yang terverifikasi.")
    reference, digest = row.get("artifact_ref"), row.get("raw_sha256")
    version_ref = r"[^/:]+/[^/:]+/[^/:]+:v[0-9]+"
    sha256 = r"[0-9a-f]{64}"
    if (row.get("model_variant") != "s" or row.get("evaluation_mode") != "full_val"
            or row.get("dataset_provenance_status") != "VERIFIED" or row.get("publication_status") != "CONFIRMED"
            or row.get("recovery_ready") is not True
            or not isinstance(reference, str) or not re.fullmatch(version_ref, reference)
            or not isinstance(digest, str) or not re.fullmatch(sha256, digest)):
        raise ValueError("Pilih baris terverifikasi dengan versi W&B tetap dan SHA-256 raw.")
    config = (reference, digest, row["recovery_seed"])
    if version == 2:
        baseline_ref = row.get("source_baseline_artifact_ref")
        baseline_sha = row.get("source_baseline_sha256")
        collection = f"yolo26s-lamp-su{float(target):.2f}x-seed{seed}-raw"
        if (row.get("baseline_seed") != seed or row.get("pruning_seed") != seed
                or not registry.get("suite_id") or row.get("suite_id") != registry["suite_id"]
                or reference.split("/")[-1].rsplit(":", 1)[0] != collection
                or not isinstance(baseline_ref, str) or not re.fullmatch(version_ref, baseline_ref)
                or not isinstance(baseline_sha, str) or not re.fullmatch(sha256, baseline_sha)):
            raise ValueError("Identitas suite, perlakuan, atau baseline dari baris registry tidak valid.")
        if include_baseline:
            return (*config, baseline_ref, baseline_sha)
    elif include_baseline:
        raise ValueError("Identitas baseline per seed memerlukan registry format 2.")
    return config


# Salin sel ini sesudah konfigurasi recovery 02–05, lalu aktifkan pengaturan berikut.
# RAW_ARTIFACT_REF, EXPECTED_RAW_SHA256, SEED, REGISTRY_BASELINE_REF, REGISTRY_BASELINE_SHA256 = (
#     recovery_config_from_registry("checkpoint_registry.json", seed=42, include_baseline=True)
# )
# RAW_MANIFEST_PATH = None
# Di sel terpisah sesudah bootstrap recovery dan sebelum run, pasang lock dari baris yang sama.
# BASELINE_REF, BASELINE_SHA256 = REGISTRY_BASELINE_REF, REGISTRY_BASELINE_SHA256